# 예금보험공사 RAG 최종 통합 파이프라인 V4.3 — HCX API 연동

이 노트북 하나에 다음 기능이 모두 포함되어 있습니다.

## 결정론적 데이터 파이프라인

- 42개 URL 정책 CSV
- 전체 크롤링과 페이지네이션
- 본문·FAQ·표·법령·Action 파싱
- 일반 이미지 다운로드
- 영상·웹툰 링크 전용 정책
- 법령 카드 그룹화
- `fn_layer` 팝업 규정 전문 추출
- 구조 기반 청킹
- 품질·회귀 검사

## HCX 후처리 파이프라인

- Colab Secrets를 통한 API 키 보호
- API 연결 및 사용 모델 확인
- 문서별 메타데이터 자동 생성
- 원문과 HCX 생성 데이터를 분리해 저장
- `bge-m3` 기반 청크 임베딩
- 코사인 유사도 기반 의미 검색
- 검색된 공식 근거만 사용하는 HCX 답변 생성
- HCX 처리 결과를 포함한 최종 ZIP 재생성

## 중요한 원칙

```text
HTML 수집·본문 변환·청킹
→ 기존 규칙 기반 코드가 담당

HCX
→ 요약·하위 분류·사용자 역할·키워드 생성
→ 이미 생성된 청크의 임베딩
→ 검색 근거 기반 답변 생성
```

HCX가 공식 원문을 수정하거나 청킹 기준을 임의로 바꾸지 않습니다.

HCX 생성 결과는 다음 별도 필드와 파일에 저장합니다.

```text
llm_metadata
documents_hcx_enriched.jsonl
hcx_metadata.jsonl
chunk_embeddings_hcx.jsonl
```

## 1. 라이브러리 설치

In [ ]:
%pip -q install "requests>=2.32,<3" "beautifulsoup4>=4.12,<5" "lxml>=5,<7" "pandas>=2.2,<4" "tqdm>=4.66,<5" "pypdf>=5,<7" "olefile>=0.47,<1" "openai>=1.68,<3"

## 2. 최종 크롤링 모듈 생성

In [ ]:
%%writefile kdic_final_pipeline.py
from __future__ import annotations

import hashlib
import json
import mimetypes
import re
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import parse_qs, urlencode, urljoin, urlparse, urlunparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

KST = timezone(timedelta(hours=9))


# ============================================================
# 설정
# ============================================================

@dataclass
class PipelineConfig:
    select_business_functions: list[str] | None = None
    run_only_url_ids: list[str] | None = None
    max_urls: int | None = None

    request_delay_seconds: float = 1.2
    request_timeout_seconds: int = 30
    max_retries: int = 3
    max_asset_bytes: int = 50 * 1024 * 1024

    respect_robots_txt: bool = True
    enable_generic_pagination: bool = True
    pagination_max_pages: int = 100
    pagination_page_size: int = 10

    download_direct_attachments: bool = True
    download_images: bool = False
    download_js_attachments_with_playwright: bool = False

    # 영상·웹툰은 원본 파일을 내려받지 않고 공식 게시 페이지 링크만 제공합니다.
    collect_supplementary_media_links: bool = True
    download_videos: bool = False
    download_webtoons: bool = False

    # 첨부파일 하이브리드 정책
    extract_attachment_text: bool = True
    create_attachment_documents: bool = True
    attachment_index_min_chars: int = 120
    attachment_max_text_chars: int = 1_000_000
    attachment_pdf_max_pages: int = 300
    attachment_keep_failed_metadata: bool = True

    enable_playwright_snapshot: bool = False
    playwright_wait_ms: int = 1500

    create_chunks: bool = True
    chunk_max_chars: int = 1600
    min_chunk_chars: int = 80

    user_agent: str = (
        "KDIC-RAG-Academic-Crawler/1.0 "
        "(low-rate public-document collection; no authentication automation)"
    )

    def __post_init__(self) -> None:
        self.select_business_functions = self.select_business_functions or []
        self.run_only_url_ids = self.run_only_url_ids or []


@dataclass
class OutputPaths:
    root: Path
    raw_html: Path
    raw_assets: Path
    response_meta: Path
    parsed_markdown: Path
    parsed_json: Path
    parsed_attachments: Path
    pagination: Path
    dynamic_diagnostics: Path
    screenshots: Path
    logs: Path
    processed: Path
    manifest: Path
    quality: Path


def create_output_paths(runtime_root: Path) -> OutputPaths:
    run_id = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
    root = runtime_root / f"kdic_crawl_output_{run_id}"
    paths = OutputPaths(
        root=root,
        raw_html=root / "raw" / "html",
        raw_assets=root / "raw" / "assets",
        response_meta=root / "raw" / "response_meta",
        parsed_markdown=root / "parsed" / "markdown",
        parsed_json=root / "parsed" / "json",
        parsed_attachments=root / "parsed" / "attachments",
        pagination=root / "diagnostics" / "pagination",
        dynamic_diagnostics=root / "diagnostics" / "dynamic",
        screenshots=root / "diagnostics" / "screenshots",
        logs=root / "logs",
        processed=root / "processed",
        manifest=root / "manifest",
        quality=root / "quality",
    )
    for value in asdict(paths).values():
        Path(value).mkdir(parents=True, exist_ok=True)
    return paths


# ============================================================
# 공통 유틸리티
# ============================================================

ATTACHMENT_EXTENSIONS = {
    ".pdf", ".hwp", ".hwpx", ".doc", ".docx", ".xls", ".xlsx",
    ".ppt", ".pptx", ".zip", ".csv", ".txt",
}
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".gif", ".webp", ".svg", ".bmp"}
VIDEO_EXTENSIONS = {".mp4", ".webm", ".mov", ".avi", ".mkv"}

NOISE_SELECTORS = [
    "script", "style", "noscript", "template", ".floatTop", ".floatBottom",
    ".paging", ".pagination", ".pageNavi", ".page_navi", ".sr-only",
    ".skip", ".skipnav", ".skip-navigation", ".loading", ".waitPage",
    "#waitPage", ".pdfViewerGuide", ".adobe-reader", ".tabMenu.clone",
]

NOISE_EXACT_TEXTS = {
    "퀵메뉴", "글자 크기", "글자확대", "글자축소", "KOR", "ENG",
    "상단으로 이동", "검색 초기화", "좌우로 움직여보세요", "표 더보기",
    "계산하기", "닫기", "Adobe Reader 바로가기",
    "영상내용입니다.", "영상 내용입니다.", "제도 소개 안내영상입니다.",
}

NOISE_CONTAINS_TEXTS = {
    "Adobe Reader", "영상내용입니다", "제도 소개 안내영상입니다",
}

NOISE_REGEXES = [
    re.compile(r"PDF\s*파일\s*내용이\s*보이지\s*않으시면.*?설치하시기\s*바랍니다[.。]?", re.I),
    re.compile(r"PDF파일\s*내용이\s*보이지\s*않으시면.*?설치하시기\s*바랍니다[.。]?", re.I),
]

ALLOWED_ACTION_DOMAIN_SUFFIXES = {
    "kdic.or.kr", "fins.kdic.or.kr", "ccrs.or.kr", "scourt.go.kr",
    "klac.or.kr", "law.go.kr", "easylaw.go.kr", "gov.kr",
}

ACTION_KEYWORDS = {
    "신청", "바로가기", "조회", "검색", "자가진단", "상담", "진행상황", "접수",
    "신고", "발급", "환급", "위치", "찾아오시는 길", "법령", "규정",
    "다운로드", "전화문의", "문의",
}

DOWNLOAD_CALL_RE = re.compile(
    r"gfn_downloadFile\(\s*['\"]([^'\"]+)['\"]\s*,\s*['\"]([^'\"]+)['\"]\s*\)"
)
URL_IN_JS_PATTERNS = [
    re.compile(r"gfn_openUrl\(\s*['\"]([^'\"]+)['\"]\s*\)"),
    re.compile(r"gfn_moveUrl\(\s*['\"]([^'\"]+)['\"]\s*\)"),
    re.compile(r"window\.open\(\s*['\"]([^'\"]+)['\"]"),
    re.compile(r"(?:window\.)?location(?:\.href)?\s*=\s*['\"]([^'\"]+)['\"]"),
    re.compile(r"location\.replace\(\s*['\"]([^'\"]+)['\"]\s*\)"),
]

BLOCK_TAGS = {
    "div", "section", "article", "aside", "header", "footer", "main",
    "p", "ul", "ol", "dl", "table", "figure", "h1", "h2", "h3",
    "h4", "h5", "h6",
}


def now_kst_iso() -> str:
    return datetime.now(KST).isoformat(timespec="seconds")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def norm(text: str | None) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def safe_name(value: str, max_length: int = 100) -> str:
    value = re.sub(r'[\\/:*?"<>|]+', "_", norm(value))
    value = re.sub(r"_+", "_", value).strip("._ ")
    return (value or "untitled")[:max_length]


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    ensure_parent(path)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def append_jsonl(path: Path, data: dict[str, Any]) -> None:
    ensure_parent(path)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(data, ensure_ascii=False) + "\n")


def absolute_url(base_url: str, candidate: str | None, allow_contact: bool = False) -> str | None:
    if not candidate:
        return None
    candidate = candidate.strip()
    if not candidate or candidate.startswith(("#", "javascript:")):
        return None
    if candidate.startswith(("tel:", "mailto:")):
        return candidate if allow_contact else None
    return urljoin(base_url, candidate)


def is_noise_text(text: str) -> bool:
    value = norm(text)
    if not value:
        return True
    if value in NOISE_EXACT_TEXTS:
        return True
    if any(pattern.search(value) for pattern in NOISE_REGEXES):
        return True
    return any(fragment in value for fragment in NOISE_CONTAINS_TEXTS)


def clean_visible_text(text: str) -> str:
    value = norm(text)
    for pattern in NOISE_REGEXES:
        value = norm(pattern.sub(" ", value))
    for phrase in ["검색 초기화", "Adobe Reader 바로가기", "계산하기", "닫기"]:
        value = norm(value.replace(phrase, " "))
    for fragment in NOISE_CONTAINS_TEXTS:
        if fragment in value:
            value = norm(value.replace(fragment, " "))
    return value


def row_to_dict(row: pd.Series) -> dict[str, str]:
    return {str(k): "" if pd.isna(v) else str(v) for k, v in row.to_dict().items()}


def unique_dicts(items: Iterable[dict[str, Any]], keys: tuple[str, ...]) -> list[dict[str, Any]]:
    result: list[dict[str, Any]] = []
    seen: set[tuple[Any, ...]] = set()
    for item in items:
        key = tuple(item.get(name) for name in keys)
        if key in seen:
            continue
        seen.add(key)
        result.append(item)
    return result


# ============================================================
# 42개 검토 CSV → 실행 Manifest
# ============================================================

REVIEW_COLUMNS = {
    "문서_ID", "업무_도메인", "목표_도메인", "문서명", "페이지_유형",
    "권장_최종결정", "RAG_본문_인덱싱", "인덱싱_범위", "Action_링크",
    "권장_Action_Type", "Action_인증", "다중페이지_수집정책", "검토_의견",
    "검토_근거", "출처_URL",
    "첨부파일_원본수집", "첨부파일_RAG_정책", "첨부파일_사용자제공정책",
    "영상_처리정책", "웹툰_처리정책", "일반이미지_처리정책",
    "보조콘텐츠_표시조건", "보조콘텐츠_링크라벨",
}

RUN_DECISIONS = {
    "include_full", "include_reference", "include_full_public", "include_partial",
    "include_partial_action", "include_full_filtered", "include_separate_domain",
    "link_only",
}


def _site_name(url: str) -> str:
    host = (urlparse(url).hostname or "").lower()
    return "금융안심포털" if host.startswith("fins.") else "예금보험공사"


def _page_type(row: pd.Series) -> str:
    doc_id = row["문서_ID"]
    title = row["문서명"]
    policy = row["다중페이지_수집정책"]
    decision = row["권장_최종결정"]
    raw = row.get("페이지_유형", "")
    if decision == "link_only":
        return "link_only"
    if "FAQ" in title.upper() or "FAQ" in policy.upper():
        return "faq"
    if doc_id == "BI-004":
        return "dynamic_lookup"
    if "상호작용" in policy or "인증 서비스" in policy or "신청 서비스" in policy:
        return "interactive_service"
    return raw or "static_page"


def _fetch_strategy(row: pd.Series) -> str:
    decision = row["권장_최종결정"]
    policy = row["다중페이지_수집정책"]
    action = row["Action_링크"]
    if decision == "link_only":
        return "link_only"
    if "필수" in policy or "자동탐지" in policy:
        return "paginated_public"
    if "다운로드" in action:
        return "requests_html_assets"
    return "requests_html"


def _index_mode(value: str) -> str:
    if value == "O":
        return "full"
    if "공개 설명만" in value:
        return "public_only"
    if "보조 인덱스" in value:
        return "reference"
    if "일부 제외" in value:
        return "filtered"
    if "별도 도메인" in value:
        return "separate_domain"
    return "none"


def normalize_review_manifest(review_df: pd.DataFrame) -> pd.DataFrame:
    review_df = review_df.fillna("").astype(str)
    missing = sorted(REVIEW_COLUMNS - set(review_df.columns))
    if missing:
        raise ValueError(f"42개 검토 CSV 필수 열 누락: {missing}")

    records: list[dict[str, str]] = []
    for _, row in review_df.iterrows():
        url = row["출처_URL"]
        decision = row["권장_최종결정"]
        if decision not in RUN_DECISIONS:
            raise ValueError(f"지원하지 않는 결정: {row['문서_ID']}={decision}")
        page_type = _page_type(row)
        action_auth = row["Action_인증"]
        records.append({
            "url_id": row["문서_ID"],
            "business_function": row["업무_도메인"],
            "target_business_function": row["목표_도메인"] or row["업무_도메인"],
            "page_title": row["문서명"],
            "source_url": url,
            "site_name": _site_name(url),
            "normalized_decision": decision,
            "decision_reason": row["검토_의견"],
            "review_basis": row["검토_근거"],
            "page_type": page_type,
            "fetch_strategy": _fetch_strategy(row),
            "parser_profile": "kdic_final_integrated_v4",
            "auth_required": action_auth,
            "asset_policy": row["Action_링크"],
            "content_scope": row["인덱싱_범위"],
            "crawl_wave": (
                "META" if decision == "link_only" else
                "W2_DYNAMIC" if ("필수" in row["다중페이지_수집정책"] or row["문서_ID"] == "BI-004") else
                "W1_ASSET" if "다운로드" in row["Action_링크"] else
                "W1_STATIC"
            ),
            "rag_index_mode": _index_mode(row["RAG_본문_인덱싱"]),
            "rag_index_label": row["RAG_본문_인덱싱"],
            "action_policy": row["Action_링크"],
            "expected_action_types": row["권장_Action_Type"],
            "action_auth": action_auth,
            "pagination_policy": row["다중페이지_수집정책"],
            "attachment_download_policy": row["첨부파일_원본수집"],
            "attachment_rag_policy": row["첨부파일_RAG_정책"],
            "attachment_user_delivery_policy": row["첨부파일_사용자제공정책"],
            "video_policy": row["영상_처리정책"],
            "webtoon_policy": row["웹툰_처리정책"],
            "image_policy": row["일반이미지_처리정책"],
            "supplementary_display_condition": row["보조콘텐츠_표시조건"],
            "supplementary_link_label": row["보조콘텐츠_링크라벨"],
        })

    manifest = pd.DataFrame(records)
    if manifest["url_id"].duplicated().any():
        raise ValueError("Manifest url_id 중복")
    if len(manifest) != 42:
        raise ValueError(f"검토표는 42개여야 합니다. 현재 {len(manifest)}개")
    return manifest



# ============================================================
# 공통 표시 제목 및 Action 라벨 정규화
# ============================================================

GENERIC_PAGE_HEADINGS = {
    "top 10", "top10", "개요", "안내", "유의사항", "구비서류안내",
    "구비서류 안내", "신고센터", "faq", "자주 묻는 질문",
}

GENERIC_BREADCRUMB_PARTS = {
    "고객센터", "faq", "소개와 방법안내", "소개와 신청방법 안내",
    "반환지원 신청하기", "지원자금관리", "부실책임조사", "채무조정 재기지원",
}

DOMAIN_DISPLAY_NAMES = {
    "예금자보호제도": "예금자보호제도",
    "예금보험금 안내": "예금보험금",
    "고객 미수령금 신청": "고객 미수령금",
    "착오송금 반환 신청": "착오송금 반환지원",
    "채무조정 안내": "채무조정",
    "은닉재산 신고": "은닉재산 신고",
}


def normalize_title_phrase(value: str) -> str:
    value = norm(value)
    value = re.sub(r"\([^)]*사이트 분류상 명칭[^)]*\)", "", value)
    replacements = {
        "착오송금반환지원": "착오송금 반환지원",
        "착오송금수취인": "착오송금 수취인",
        "은닉재산신고": "은닉재산 신고",
        "구비서류안내": "구비서류 안내",
        "채무정보조회": "채무정보조회",
        "미수령금통합신청": "미수령금 통합신청",
    }
    for source, target in replacements.items():
        value = value.replace(source, target)
    return norm(value)


def split_manifest_breadcrumb(title: str) -> list[str]:
    return [normalize_title_phrase(part) for part in re.split(r"\s*>\s*", title or "") if normalize_title_phrase(part)]


def is_generic_heading(value: str) -> bool:
    return normalize_title_phrase(value).lower() in GENERIC_PAGE_HEADINGS


def resolve_display_title(manifest_row: dict[str, str], page_heading: str = "") -> str:
    """TOP 10·개요·유의사항처럼 일반적인 제목을 업무 맥락이 있는 제목으로 바꿉니다."""
    manifest_title = manifest_row.get("page_title", "")
    parts = split_manifest_breadcrumb(manifest_title)
    heading = normalize_title_phrase(page_heading)
    business = DOMAIN_DISPLAY_NAMES.get(
        manifest_row.get("business_function", ""),
        normalize_title_phrase(manifest_row.get("business_function", "")),
    )
    page_type = manifest_row.get("page_type", "")

    if manifest_row.get("normalized_decision") == "link_only" and "self_check" in manifest_row.get("expected_action_types", ""):
        return norm(f"{business} 신청대상 자가진단")

    if page_type == "faq":
        leaf = parts[-1] if parts else ""
        leaf_lower = leaf.lower()
        if "사이트 분류상 명칭" in manifest_title:
            subject = business
            suffix = "FAQ"
        elif leaf_lower in {"top 10", "top10"}:
            subject = business
            suffix = "주요 FAQ"
        else:
            subject = leaf
            subject = re.sub(r"FAQ$", "", subject, flags=re.I).strip()
            if subject.endswith("신청") and "반환지원" in subject:
                subject = subject[:-2].strip()
            if not subject or is_generic_heading(subject):
                subject = business
            suffix = "FAQ"
        title = f"{subject} {suffix}"
        return norm(title)

    if heading and not is_generic_heading(heading):
        return heading[:100]

    meaningful = [part for part in parts if part.lower() not in GENERIC_BREADCRUMB_PARTS]
    generic = heading or (meaningful[-1] if meaningful else "")

    if generic in {"개요"} and len(meaningful) >= 2:
        return norm(" ".join(meaningful[-3:]))[:100]
    if generic in {"유의사항"}:
        subject = meaningful[-2] if len(meaningful) >= 2 else business
        return norm(f"{subject} 유의사항")[:100]
    if generic in {"구비서류 안내", "구비서류안내"}:
        # 제목 뒤에 대상(착오송금인/수취인)이 있는 구조까지 공통 처리
        subject_candidates = [p for p in meaningful if p not in {"구비서류 안내", "구비서류안내"}]
        subject = subject_candidates[-1] if subject_candidates else business
        return norm(f"{subject} 구비서류 안내")[:100]
    if generic == "신고센터":
        subject = meaningful[-2] if len(meaningful) >= 2 else business
        return norm(f"{subject} 신고센터")[:100]
    if heading and heading not in {"안내"}:
        return heading[:100]
    if meaningful:
        return norm(" ".join(meaningful[-3:]))[:100]
    return business or normalize_title_phrase(manifest_title)


def compact_action_label(raw_label: str, action_type: str, manifest_row: dict[str, str]) -> tuple[str, str]:
    """버튼 카드 전체 문구가 라벨로 붙는 경우 사용자용 라벨을 짧게 만듭니다."""
    raw = norm(raw_label)
    cleaned = clean_visible_text(raw.replace("예금보험공사 핵심가치", " "))
    business = DOMAIN_DISPLAY_NAMES.get(
        manifest_row.get("business_function", ""),
        normalize_title_phrase(manifest_row.get("business_function", "")),
    )
    patterns = [
        ("온라인 신청", f"{business} 온라인 신청"),
        ("방문 신청", f"{business} 방문 신청 위치"),
        ("자가진단", f"{resolve_display_title(manifest_row)} 자가진단"),
        ("진행현황", f"{resolve_display_title(manifest_row)} 진행현황 조회"),
    ]
    for token, replacement in patterns:
        if token in cleaned:
            return norm(replacement), raw
    if len(cleaned) > 80:
        first = re.split(r"\s+(?:접속방법|준비물|연락처|주소)\s*:", cleaned, maxsplit=1)[0]
        first = re.sub(r"\s*\([^)]*\)", "", first)
        cleaned = norm(first)
    if not cleaned:
        cleaned = f"{resolve_display_title(manifest_row)} 바로가기"
    return cleaned[:80], raw

# ============================================================
# HTTP 수집
# ============================================================

@dataclass
class FetchResult:
    requested_url: str
    final_url: str
    status_code: int
    ok: bool
    content_type: str
    encoding: str
    elapsed_seconds: float
    collected_at: str
    content_length: int
    raw_sha256: str
    redirect_history: list[dict[str, Any]]
    selected_headers: dict[str, str]
    body: bytes


def create_session(config: PipelineConfig) -> requests.Session:
    retry = Retry(
        total=config.max_retries,
        connect=config.max_retries,
        read=config.max_retries,
        status=config.max_retries,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "POST", "HEAD"}),
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=2, pool_maxsize=2)
    session = requests.Session()
    session.headers.update({
        "User-Agent": config.user_agent,
        "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.5",
    })
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def choose_encoding(response: requests.Response) -> str:
    encoding = response.encoding
    if not encoding or encoding.lower() in {"iso-8859-1", "ascii"}:
        encoding = response.apparent_encoding or "utf-8"
    return encoding


def robots_allows(url: str, config: PipelineConfig) -> tuple[bool, str]:
    if not config.respect_robots_txt:
        return True, "disabled"
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    try:
        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.read()
        allowed = parser.can_fetch(config.user_agent, url)
        return allowed, "allowed" if allowed else "disallowed"
    except Exception as error:
        return True, f"unavailable:{type(error).__name__}"


def response_to_fetch_result(response: requests.Response, requested_url: str, elapsed: float) -> FetchResult:
    encoding = choose_encoding(response)
    response.encoding = encoding
    selected = {
        key: value for key, value in response.headers.items()
        if key in {"Content-Type", "Content-Length", "Last-Modified", "ETag", "Cache-Control", "Content-Disposition"}
    }
    return FetchResult(
        requested_url=requested_url,
        final_url=response.url,
        status_code=response.status_code,
        ok=response.ok,
        content_type=response.headers.get("Content-Type", ""),
        encoding=encoding,
        elapsed_seconds=round(elapsed, 3),
        collected_at=now_kst_iso(),
        content_length=len(response.content),
        raw_sha256=sha256_bytes(response.content),
        redirect_history=[
            {"status_code": item.status_code, "url": item.url, "location": item.headers.get("Location")}
            for item in response.history
        ],
        selected_headers=selected,
        body=response.content,
    )


def fetch_url(session: requests.Session, url: str, config: PipelineConfig) -> FetchResult:
    allowed, status = robots_allows(url, config)
    if not allowed:
        raise PermissionError(f"robots.txt에서 허용하지 않음: {url}")
    started = time.perf_counter()
    response = session.get(url, timeout=config.request_timeout_seconds, allow_redirects=True)
    result = response_to_fetch_result(response, url, time.perf_counter() - started)
    result.selected_headers["Robots-Check"] = status
    return result


def save_fetch_result(paths: OutputPaths, row: dict[str, str], result: FetchResult) -> tuple[Path, Path]:
    folder = safe_name(row["business_function"])
    html_path = paths.raw_html / folder / f"{row['url_id']}.html"
    meta_path = paths.response_meta / folder / f"{row['url_id']}.json"
    ensure_parent(html_path)
    html_path.write_bytes(result.body)
    meta = asdict(result)
    meta.pop("body", None)
    meta.update({
        "url_id": row["url_id"],
        "business_function": row["business_function"],
        "page_title_manifest": row["page_title"],
        "fetch_strategy": row["fetch_strategy"],
        "parser_profile": row["parser_profile"],
        "raw_html_path": str(html_path.relative_to(paths.root)),
    })
    write_json(meta_path, meta)
    return html_path, meta_path


# ============================================================
# 결정론적 HTML 파싱
# ============================================================

def safe_text(node: Tag) -> str:
    fragment = BeautifulSoup(str(node), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""
    for child in root.select(
        ".sr-only,.hide,script,style,noscript,template,.floatTop,.floatBottom,.paging,.pagination"
    ):
        child.decompose()
    for image in root.find_all("img"):
        image.replace_with(norm(image.get("alt", "")))
    for br in root.find_all("br"):
        br.replace_with(" ")
    return clean_visible_text(root.get_text(" ", strip=True))


def infer_file_format(node: Tag) -> str:
    text = (" ".join(node.get("class", [])).lower() + " " + norm(node.get_text(" ", strip=True)).lower())
    for keyword, fmt in [
        ("icohwp", "HWP"), ("hwp", "HWP"), ("icopdf", "PDF"), ("pdf", "PDF"),
        ("xlsx", "XLSX"), ("excel", "XLSX"), ("docx", "DOCX"), ("doc", "DOC"),
    ]:
        if keyword in text:
            return fmt
    return "FILE"


def cell_text(cell: Tag) -> str:
    fragment = BeautifulSoup(str(cell), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""
    for child in root.select("script,style,noscript,template,.sr-only"):
        child.decompose()
    for button in root.find_all("button"):
        fmt = infer_file_format(button)
        label = re.sub(r"다운로드$", "", norm(button.get_text(" ", strip=True))).strip()
        button.replace_with(f"{label} {fmt} 다운로드".strip())
    for image in root.find_all("img"):
        image.replace_with(norm(image.get("alt", "")))
    for br in root.find_all("br"):
        br.replace_with(" ")
    return norm(root.get_text(" ", strip=True))


def expand_table(table: Tag) -> tuple[list[str], list[list[str]]]:
    grid: list[list[str]] = []
    active: dict[int, list[Any]] = {}
    header_flags: list[bool] = []
    max_columns = 0

    for tr in table.find_all("tr"):
        row: list[str] = []
        column = 0
        cells = tr.find_all(["th", "td"], recursive=False)

        def fill_active() -> None:
            nonlocal column
            while column in active:
                remaining, value = active[column]
                while len(row) <= column:
                    row.append("")
                row[column] = value
                remaining -= 1
                if remaining <= 0:
                    del active[column]
                else:
                    active[column] = [remaining, value]
                column += 1

        fill_active()
        header_flags.append(any(cell.name == "th" for cell in cells))
        for cell in cells:
            fill_active()
            text = cell_text(cell)
            rowspan = max(1, int(cell.get("rowspan", 1) or 1))
            colspan = max(1, int(cell.get("colspan", 1) or 1))
            for offset in range(colspan):
                target = column + offset
                while len(row) <= target:
                    row.append("")
                row[target] = text
                if rowspan > 1:
                    active[target] = [rowspan - 1, text]
            column += colspan
        if active:
            final_col = max(active)
            while column <= final_col:
                if column in active:
                    fill_active()
                else:
                    while len(row) <= column:
                        row.append("")
                    column += 1
        max_columns = max(max_columns, len(row))
        if row and any(row):
            grid.append(row)

    if not grid:
        return [], []
    for row in grid:
        row.extend([""] * (max_columns - len(row)))
    if table.thead:
        header_count = len(table.thead.find_all("tr", recursive=False))
    else:
        header_count = 1 if header_flags and header_flags[0] else 0
    header_count = max(1, header_count)
    header_rows = grid[:header_count]
    rows = grid[header_count:]
    headers: list[str] = []
    for col in range(max_columns):
        values: list[str] = []
        for hrow in header_rows:
            value = norm(hrow[col])
            if value and value not in values:
                values.append(value)
        headers.append(" / ".join(values) if values else f"열{col + 1}")
    counts: dict[str, int] = {}
    unique_headers: list[str] = []
    for header in headers:
        counts[header] = counts.get(header, 0) + 1
        unique_headers.append(header if counts[header] == 1 else f"{header} {counts[header]}")
    return unique_headers, rows




def classify_action_type(label: str, expected_types: str = "") -> str:
    text = norm(label)
    tests = [
        ("자가진단", "self_check"), ("진행상황", "progress"), ("신고", "report"),
        ("상담", "consult"), ("부채증명", "issue"), ("발급", "issue"),
        ("환급", "refund"), ("법령", "legal_reference"), ("규정", "legal_reference"),
        ("위치", "location"), ("찾아오시는", "location"), ("전화", "contact"),
        ("문의", "contact"), ("조회", "lookup"), ("검색", "lookup"),
        ("다운로드", "download"),
        ("신청", "apply"), ("접수", "apply"),
    ]
    for keyword, action_type in tests:
        if keyword in text:
            return action_type
    return "related_service"


def allowed_action_domain(url: str) -> bool:
    if url.startswith(("tel:", "mailto:")):
        return True
    host = (urlparse(url).hostname or "").lower()
    return any(host == suffix or host.endswith("." + suffix) for suffix in ALLOWED_ACTION_DOMAIN_SUFFIXES)


def _extract_js_url(onclick: str) -> str | None:
    for pattern in URL_IN_JS_PATTERNS:
        match = pattern.search(onclick or "")
        if match:
            return match.group(1)
    direct = re.search(r"https?://[^'\"\s)]+", onclick or "")
    return direct.group(0) if direct else None


def action_label_with_context(node: Tag, label: str) -> str:
    label = norm(label)
    if label not in {"바로가기", "신청", "조회", "다운로드"}:
        return label
    parent = node.find_parent(["div", "li", "td", "section", "article"])
    if parent:
        context_node = parent.find(["strong", "h2", "h3", "h4", "dt"], recursive=True)
        context = norm(context_node.get_text(" ", strip=True)) if context_node else ""
        if context and context != label:
            return f"{context} {label}"
    previous = node.find_previous(["strong", "h2", "h3", "h4", "dt"])
    context = norm(previous.get_text(" ", strip=True)) if previous else ""
    return f"{context} {label}".strip() if context else label


def extract_actions(
    root: Tag,
    base_url: str,
    manifest_row: dict[str, str],
    attachments: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    actions: list[dict[str, Any]] = []
    expected = manifest_row.get("expected_action_types", "")
    expected_set = {value.strip() for value in expected.split("|") if value.strip()}
    action_policy = manifest_row.get("action_policy", "")
    if action_policy.startswith("X"):
        return []
    seen: set[tuple[str, str]] = set()

    def add(label: str, url: str, source_tag: str, js_function: str = "") -> None:
        raw_label = norm(label)
        if not raw_label or is_noise_text(raw_label):
            return
        if any(noise in raw_label for noise in ["디지털역사관", "예솜이", "챗봇", "앱 설치", "공식 홈페이지"]):
            return
        action_like = any(keyword in raw_label for keyword in ACTION_KEYWORDS)
        if not action_like:
            return
        action_type = classify_action_type(raw_label, expected)
        label, raw_label = compact_action_label(raw_label, action_type, manifest_row)
        host = (urlparse(url).hostname or "").lower()
        if host == "law.go.kr" or host.endswith(".law.go.kr"):
            action_type = "legal_reference"
        if expected_set and action_type not in expected_set and action_type not in {"download", "legal_reference", "location", "contact"}:
            return
        key = (url, label)
        if key in seen:
            return
        seen.add(key)
        actions.append({
            "action_id": sha256_text(f"{manifest_row['url_id']}|{url}|{label}")[:16],
            "label": label,
            "raw_label": raw_label if raw_label != label else "",
            "url": url,
            "action_type": action_type,
            "source_tag": source_tag,
            "javascript_function": js_function,
            "auth_required": manifest_row.get("action_auth", manifest_row.get("auth_required", "")),
            "official_domain": allowed_action_domain(url),
            "requires_review": not allowed_action_domain(url),
        })

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "").strip()
        url = absolute_url(base_url, href, allow_contact=True)
        if not url:
            continue
        label = norm(anchor.get_text(" ", strip=True)) or norm(anchor.get("title", ""))
        label = action_label_with_context(anchor, label)
        if any(keyword in label for keyword in ACTION_KEYWORDS):
            add(label, url, "a")

    for node in root.find_all(["button", "a", "input"], onclick=True):
        onclick = node.get("onclick", "")
        candidate = _extract_js_url(onclick)
        if not candidate:
            continue
        url = absolute_url(base_url, candidate, allow_contact=True)
        if not url:
            continue
        label = norm(node.get_text(" ", strip=True)) or norm(node.get("value", "")) or norm(node.get("title", ""))
        label = action_label_with_context(node, label)
        function_name = onclick.split("(", 1)[0].strip()
        add(label, url, node.name, function_name)

    # 다운로드도 챗봇 Action으로 제공할 수 있도록 연결합니다.
    for attachment in attachments:
        url = attachment.get("url") or base_url
        label = attachment.get("display_name", "첨부파일 다운로드")
        key = (url, label)
        if key in seen:
            continue
        seen.add(key)
        actions.append({
            "action_id": attachment["attachment_id"],
            "label": label,
            "url": url,
            "action_type": "download",
            "source_tag": "attachment",
            "attachment_id": attachment["attachment_id"],
            "download_method": attachment["download_method"],
            "auth_required": "불필요",
            "official_domain": True,
            "requires_review": False,
        })

    return actions


GLOBAL_UI_LINK_TEXTS = {
    "창립 30주년 예금보험공사 디지털역사관 바로가기",
    "KDIC(예금보험공사) 공식 홈페이지",
    "상단으로 이동",
}


def is_global_ui_link(anchor: Tag, url: str, text: str) -> bool:
    """본문 아래 고정 배너·퀵메뉴·공통 이동 링크를 업무 링크에서 제외합니다."""
    if text in GLOBAL_UI_LINK_TEXTS:
        return True
    lowered = f"{url} {text}".lower()
    if any(token in lowered for token in ["kdic30th.kr", "디지털역사관", "예솜이", "챗봇"]):
        return True
    for parent in [anchor, *list(anchor.parents)[:6]]:
        if not isinstance(parent, Tag):
            continue
        tokens = " ".join(parent.get("class", [])).lower() + " " + norm(parent.get("id", "")).lower()
        if any(token in tokens for token in [
            "floattop", "floatbottom", "quickmenu", "quick-menu", "floating",
            "snslist", "share", "site-move", "sitemove", "thirty",
        ]):
            return True
    return False


def extract_links(root: Tag, base_url: str) -> list[dict[str, Any]]:
    links: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()
    for anchor in root.find_all("a", href=True):
        url = absolute_url(base_url, anchor.get("href"))
        text = norm(anchor.get_text(" ", strip=True))
        if not url or not text or is_noise_text(text) or is_global_ui_link(anchor, url, text):
            continue
        key = (url, text)
        if key in seen:
            continue
        seen.add(key)
        links.append({
            "url": url,
            "anchor_text": text,
            "link_role": "content",
            "source_tag": "a",
        })
    return links


def merge_links_with_actions(
    content_links: list[dict[str, Any]],
    actions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """JSON의 links에서도 버튼형 Action URL을 한 번에 확인할 수 있게 합칩니다."""
    merged = list(content_links)
    seen = {(item.get("url"), item.get("anchor_text")) for item in merged}
    for action in actions:
        url = action.get("url")
        label = action.get("label")
        if not url or not label:
            continue
        key = (url, label)
        if key in seen:
            continue
        seen.add(key)
        merged.append({
            "url": url,
            "anchor_text": label,
            "link_role": "action",
            "action_type": action.get("action_type", "related_service"),
            "source_tag": action.get("source_tag", ""),
            "action_id": action.get("action_id", ""),
        })
    return merged


def _looks_decorative_image(image: Tag, filename: str, alt: str) -> bool:
    lowered = " ".join([
        filename.lower(), alt.lower(),
        " ".join(image.get("class", [])).lower(),
        norm(image.get("id", "")).lower(),
    ])
    decorative_tokens = {
        "logo", "icon", "ico_", "bullet", "btn_", "button", "banner",
        "bg_", "background", "qr", "character", "mascot", "예솜이",
    }
    return any(token in lowered for token in decorative_tokens)


def extract_videos(root: Tag, base_url: str, manifest_row: dict[str, str]) -> list[dict[str, Any]]:
    """영상 URL은 관리용 메타데이터로만 저장하고 다운로드·인덱싱하지 않습니다."""
    videos: list[dict[str, Any]] = []
    seen: set[str] = set()

    def add(media_url: str | None, poster_url: str | None, source_tag: str, label: str) -> None:
        url = absolute_url(base_url, media_url)
        if not url or url in seen:
            return
        seen.add(url)
        videos.append({
            "video_id": sha256_text(f"{manifest_row['url_id']}|{url}")[:16],
            "label": label or f"{manifest_row['page_title']} 영상",
            "media_url": url,
            "poster_url": absolute_url(base_url, poster_url),
            "source_page_url": manifest_row["source_url"],
            "source_tag": source_tag,
            "indexable": False,
            "download": False,
            "content_role": "supplementary",
            "delivery_mode": "official_page",
        })

    for video in root.find_all("video"):
        label = norm(video.get("title", "")) or norm(video.get("aria-label", ""))
        add(video.get("src"), video.get("poster"), "video", label)
        for source_node in video.find_all("source", src=True):
            add(source_node.get("src"), video.get("poster"), "video/source", label)

    for iframe in root.find_all("iframe", src=True):
        src = iframe.get("src", "")
        lowered = src.lower()
        if any(token in lowered for token in ["youtube", "youtu.be", "vimeo", ".mp4", ".webm"]):
            label = norm(iframe.get("title", ""))
            add(src, None, "iframe", label)

    return videos


def _has_webtoon(root: Tag) -> bool:
    for image in root.find_all("img"):
        value = " ".join([
            image.get("src", ""), image.get("data-src", ""),
            image.get("alt", ""), image.get("title", ""),
        ]).lower()
        if "webtoon" in value or "웹툰" in value:
            return True
    return "웹툰" in norm(root.get_text(" ", strip=True))


def extract_supplementary_links(
    root: Tag,
    base_url: str,
    manifest_row: dict[str, str],
    videos: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """영상·웹툰은 Action이 아닌 참고 링크로 공식 게시 페이지만 제공합니다."""
    links: list[dict[str, Any]] = []
    label_override = norm(manifest_row.get("supplementary_link_label", ""))
    display_condition = norm(manifest_row.get("supplementary_display_condition", ""))

    if videos and manifest_row.get("video_policy") == "reference_only_official_page":
        links.append({
            "supplementary_id": sha256_text(f"{manifest_row['url_id']}|video|{manifest_row['source_url']}")[:16],
            "label": label_override or f"{manifest_row['page_title']} 영상 보기",
            "url": manifest_row["source_url"],
            "link_type": "video",
            "indexable": False,
            "content_status": "reference_only",
            "display_condition": display_condition or "user_requests_video",
            "media_urls": [item["media_url"] for item in videos],
            "notice": "영상은 이해를 돕는 참고 자료이며 최신 제도 답변의 직접 근거로 사용하지 않습니다.",
        })

    if _has_webtoon(root) and manifest_row.get("webtoon_policy") == "reference_only_official_page":
        links.append({
            "supplementary_id": sha256_text(f"{manifest_row['url_id']}|webtoon|{manifest_row['source_url']}")[:16],
            "label": label_override or f"{manifest_row['page_title']} 웹툰 보기",
            "url": manifest_row["source_url"],
            "link_type": "webtoon",
            "indexable": False,
            "content_status": "reference_only",
            "display_condition": display_condition or "user_requests_visual_guide",
            "notice": "웹툰은 참고용 시각 자료이며 금액·신청 조건은 현재 공식 본문을 우선합니다.",
        })

    return links


def extract_images(
    root: Tag,
    base_url: str,
    manifest_row: dict[str, str],
    video_poster_urls: set[str] | None = None,
) -> list[dict[str, Any]]:
    """웹툰·영상 포스터·장식 이미지는 제외하고 핵심 이미지 메타데이터만 저장합니다."""
    images: list[dict[str, Any]] = []
    seen: set[str] = set()
    poster_urls = video_poster_urls or set()
    for image in root.find_all("img"):
        source = image.get("src") or image.get("data-src") or image.get("data-original")
        url = absolute_url(base_url, source)
        if not url or url in seen or url in poster_urls:
            continue
        alt = norm(image.get("alt", ""))
        filename = Path(urlparse(url).path).name
        lowered = f"{filename} {alt}".lower()

        # 웹툰은 이미지 목록·다운로드·RAG에서 제외하고 공식 페이지 참고 링크만 제공합니다.
        if "webtoon" in lowered or "웹툰" in lowered:
            continue
        if _looks_decorative_image(image, filename, alt):
            continue

        role = "content_image"
        if any(keyword in lowered for keyword in ["proc", "process", "step", "flow", "절차", "과정"]):
            role = "process_diagram"
        elif not alt:
            # 파일명만으로 의미를 판단할 수 없는 이미지는 저장하지 않습니다.
            continue

        seen.add(url)
        images.append({
            "url": url,
            "alt": alt,
            "filename": filename,
            "image_role": role,
            "indexable": False,
            "download_policy": manifest_row.get("image_policy", "metadata_only_nondecorative"),
        })
    return images


class StructureParser:
    def __init__(self, page_type: str = "static_page"):
        self.blocks: list[dict[str, Any]] = []
        self.page_type = page_type

    def add(self, block: dict[str, Any] | None) -> None:
        if not block:
            return
        if self.blocks and block.get("type") in {"paragraph", "heading"}:
            if self.blocks[-1].get("type") == block.get("type") and self.blocks[-1].get("text") == block.get("text"):
                return
        self.blocks.append(block)

    def parse(self, root: Tag) -> list[dict[str, Any]]:
        faq_dls = root.select(".accoWrap .accodian > dl, .accodian.con.ico > dl, .accordion > dl")
        if self.page_type == "faq" or faq_dls:
            if not faq_dls:
                faq_dls = [dl for dl in root.find_all("dl") if dl.find("dt", recursive=False) and dl.find("dd", recursive=False)]
            faq_ids = {id(node) for node in faq_dls}
            for child in root.find_all(recursive=False):
                if isinstance(child, Tag) and any(id(dl) in faq_ids for dl in child.select("dl")):
                    continue
                self.process_node(child)
            for dl in faq_dls:
                self.parse_faq(dl)
        else:
            for child in root.find_all(recursive=False):
                self.process_node(child)
        return self.blocks

    def parse_faq(self, dl: Tag) -> None:
        dt = dl.find("dt", recursive=False)
        dd = dl.find("dd", recursive=False)
        if not dt or not dd:
            return
        question = re.sub(r"^\s*질문\s*", "", safe_text(dt))
        question = re.sub(r"\s*열기\s*$", "", question)
        answer_parser = StructureParser("static_page")
        answer_parser.process_node(dd)
        cleaned_answer_blocks: list[dict[str, Any]] = []
        for block in answer_parser.blocks:
            if block.get("type") == "paragraph":
                value = re.sub(r"^\s*답변\s*[:：]?\s*", "", block.get("text", ""))
                if not norm(value):
                    continue
                block = {**block, "text": norm(value)}
            cleaned_answer_blocks.append(block)
        answer_parser.blocks = cleaned_answer_blocks
        if not answer_parser.blocks:
            fallback_answer = re.sub(r"^\s*답변\s*[:：]?\s*", "", safe_text(dd))
            answer_parser.add({"type": "paragraph", "text": norm(fallback_answer)})
        answer_text = blocks_text(answer_parser.blocks)
        if question and answer_text:
            self.add({
                "type": "faq", "question": norm(question),
                "answer_blocks": answer_parser.blocks, "answer_text": answer_text,
            })

    def parse_definition_list(self, dl: Tag) -> None:
        children = [child for child in dl.find_all(recursive=False) if isinstance(child, Tag)]
        index = 0
        while index < len(children):
            if children[index].name != "dt":
                self.process_node(children[index])
                index += 1
                continue
            dt = children[index]
            dd = children[index + 1] if index + 1 < len(children) and children[index + 1].name == "dd" else None
            term = re.sub(r"\s*열기\s*$", "", safe_text(dt))
            term = re.sub(r"^\s*\d{1,2}\s+(?=\D)", "", term)
            if dd:
                parser = StructureParser("static_page")
                parser.process_node(dd)
                if not parser.blocks:
                    parser.add({"type": "paragraph", "text": safe_text(dd)})
                self.add({
                    "type": "definition", "term": norm(term),
                    "definition_blocks": parser.blocks,
                    "definition_text": blocks_text(parser.blocks),
                })
                index += 2
            else:
                self.add({"type": "heading", "level": 3, "text": norm(term)})
                index += 1

    def parse_resource_card(self, node: Tag) -> bool:
        """제목·설명/목록·바로가기 버튼으로 구성된 카드 한 개를 의미 단위로 묶습니다."""
        title_node = node.find(["strong", "h2", "h3", "h4", "dt"], recursive=False)
        if not title_node:
            return False
        has_body = bool(node.find(["ul", "ol", "p", "dl", "table"], recursive=False))
        has_action = bool(node.find(["a", "button"], recursive=False))
        if not (has_body or has_action):
            return False

        title = safe_text(title_node)
        if not title or is_noise_text(title):
            return False

        fragment = BeautifulSoup(str(node), "lxml")
        copied = fragment.body.find(node.name) if fragment.body else None
        if not copied:
            return False

        copied_title = copied.find(["strong", "h2", "h3", "h4", "dt"], recursive=False)
        if copied_title:
            copied_title.decompose()

        # 이동 URL은 actions/links에서 구조화하므로 본문에는 '바로가기' 글자를 남기지 않습니다.
        for action_node in copied.find_all(["a", "button", "input"]):
            action_node.decompose()

        child_parser = StructureParser("static_page")
        for child in copied.find_all(recursive=False):
            child_parser.process_node(child)

        content_blocks = child_parser.blocks
        self.add({
            "type": "resource_group",
            "title": title,
            "content_blocks": content_blocks,
            "content_text": blocks_text(content_blocks),
        })
        return True

    def process_node(self, node: Any) -> None:
        if isinstance(node, Comment):
            return
        if isinstance(node, NavigableString):
            text = norm(str(node))
            if text and not is_noise_text(text):
                self.add({"type": "paragraph", "text": text})
            return
        if not isinstance(node, Tag):
            return
        name = node.name.lower()
        classes = set(node.get("class", []))
        if name in {"script", "style", "noscript", "template"} or classes & {"floatTop", "floatBottom", "paging", "pagination", "sr-only"}:
            return
        if "item" in classes and self.parse_resource_card(node):
            return
        if name in {"button", "input"}:
            return
        if name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
            text = safe_text(node)
            if text and not is_noise_text(text):
                self.add({"type": "heading", "level": int(name[1]), "text": text})
            return
        if name == "p":
            text = safe_text(node)
            if text and not is_noise_text(text):
                self.add({"type": "paragraph", "text": text})
            return
        if name == "table":
            headers, rows = expand_table(node)
            if headers or rows:
                self.add({"type": "table", "headers": headers, "rows": rows, "row_count": len(rows)})
            return
        if name in {"ul", "ol"}:
            items: list[str] = []
            for li in node.find_all("li", recursive=False):
                fragment = BeautifulSoup(str(li), "lxml")
                copied = fragment.body.find("li") if fragment.body else None
                if not copied:
                    continue
                for nested in copied.find_all(["ul", "ol"]):
                    nested.decompose()

                # 사이트가 CSS용 번호 span과 실제 텍스트 번호를 동시에 두는 경우가 있습니다.
                # 예: <li><span>1</span> 1. 첫 번째 절차</li>
                # 번호 전용 직계 자식 태그를 먼저 제거한 뒤, 텍스트 번호도 반복 제거합니다.
                if name == "ol":
                    for marker in copied.find_all(
                        ["span", "em", "strong", "i", "b"],
                        recursive=False,
                    ):
                        marker_text = norm(marker.get_text(" ", strip=True))
                        marker_classes = " ".join(marker.get("class", [])).lower()
                        if (
                            re.fullmatch(r"(?:\d{1,3}|[①-⑳])(?:[.)])?", marker_text)
                            or any(token in marker_classes for token in ["num", "number", "step"])
                            and re.fullmatch(r"\d{1,3}", marker_text)
                        ):
                            marker.decompose()

                text = safe_text(copied)
                if name == "ol":
                    prefix_pattern = re.compile(
                        r"^\s*(?:\(?\d{1,3}\)?[.)]|[①-⑳]|\d{1,3}(?=\s+\D))\s*"
                    )
                    # 최대 3회 반복 제거해 '1 1. 내용', '1. 1. 내용'을 모두 정리합니다.
                    for _ in range(3):
                        cleaned = norm(prefix_pattern.sub("", text, count=1))
                        if cleaned == text:
                            break
                        text = cleaned
                if text and not is_noise_text(text):
                    items.append(text)
            # 단순 탭 메뉴는 제거합니다.
            if set(items).issubset({"착오송금인", "착오송금수취인"}):
                items = []
            if items:
                self.add({"type": "list", "ordered": name == "ol", "items": items})
            for li in node.find_all("li", recursive=False):
                for nested in li.find_all(["ul", "ol"], recursive=False):
                    self.process_node(nested)
            return
        if name == "dl":
            self.parse_definition_list(node)
            return
        if name == "figure":
            caption = node.find("figcaption")
            if caption:
                text = safe_text(caption)
                if text and not is_noise_text(text):
                    self.add({"type": "paragraph", "text": text})
            return

        inline: list[str] = []
        def flush() -> None:
            nonlocal inline
            text = norm(" ".join(inline))
            inline = []
            if text and not is_noise_text(text):
                self.add({"type": "paragraph", "text": text})

        for child in node.children:
            if isinstance(child, Comment):
                continue
            if isinstance(child, NavigableString):
                value = norm(str(child))
                if value:
                    inline.append(value)
            elif isinstance(child, Tag):
                if child.name.lower() == "br":
                    inline.append(" ")
                elif child.name.lower() in BLOCK_TAGS:
                    flush()
                    self.process_node(child)
                elif child.name.lower() in {"button", "input"}:
                    continue
                else:
                    value = safe_text(child)
                    if value:
                        inline.append(value)
        flush()


def render_blocks(blocks: list[dict[str, Any]]) -> str:
    lines: list[str] = []
    for block in blocks:
        kind = block.get("type")
        if kind == "heading":
            level = min(max(int(block.get("level", 2)), 2), 6)
            lines.append(f"{'#' * level} {block.get('text', '')}")
        elif kind == "paragraph":
            lines.append(block.get("text", ""))
        elif kind == "list":
            for index, item in enumerate(block.get("items", []), start=1):
                prefix = f"{index}. " if block.get("ordered") else "- "
                lines.append(prefix + item)
        elif kind == "resource_group":
            lines.append("### " + block.get("title", ""))
            content = render_blocks(block.get("content_blocks", []))
            if content:
                lines.append(content)
        elif kind == "definition":
            lines.append("### " + block.get("term", ""))
            lines.append(render_blocks(block.get("definition_blocks", [])))
        elif kind == "faq":
            lines.append("### Q. " + block.get("question", ""))
            lines.append(render_blocks(block.get("answer_blocks", [])))
        elif kind == "table":
            headers = [norm(value).replace("|", "\\|") for value in block.get("headers", [])]
            if headers:
                lines.append("| " + " | ".join(headers) + " |")
                lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
                for row in block.get("rows", []):
                    values = [norm(value).replace("|", "\\|") for value in row]
                    values.extend([""] * (len(headers) - len(values)))
                    lines.append("| " + " | ".join(values[:len(headers)]) + " |")
        lines.append("")
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def render_document_markdown(document: dict[str, Any]) -> str:
    """검수용 MD에는 페이지 큰 제목과 공식 버튼·링크를 명시적으로 표시합니다."""
    lines: list[str] = []
    title = norm(document.get("display_title") or document.get("page_heading") or document.get("title"))
    if title:
        lines.extend([f"# {title}", ""])
    body = document.get("content_markdown", "").strip()
    if body:
        lines.extend([body, ""])

    actions = [item for item in document.get("actions", []) if item.get("url") and item.get("label")]
    if actions:
        lines.extend(["## 공식 바로가기", ""])
        seen: set[tuple[str, str]] = set()
        for action in actions:
            key = (action["url"], action["label"])
            if key in seen:
                continue
            seen.add(key)
            suffix = f" — {action.get('action_type')}" if action.get("action_type") else ""
            lines.append(f"- [{action['label']}]({action['url']}){suffix}")
        lines.append("")

    action_urls = {item.get("url") for item in actions}
    related = [
        item for item in document.get("links", [])
        if item.get("link_role") == "content" and item.get("url") not in action_urls
        and item.get("url") and item.get("anchor_text")
    ]
    if related:
        lines.extend(["## 관련 링크", ""])
        seen_urls: set[tuple[str, str]] = set()
        for item in related:
            key = (item["url"], item["anchor_text"])
            if key in seen_urls:
                continue
            seen_urls.add(key)
            lines.append(f"- [{item['anchor_text']}]({item['url']})")
        lines.append("")

    supplementary = [item for item in document.get("supplementary_links", []) if item.get("url")]
    if supplementary:
        lines.extend(["## 참고 자료", ""])
        for item in supplementary:
            lines.append(f"- [{item.get('label', '참고 자료')}]({item['url']})")
        lines.append("")

    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def blocks_text(blocks: list[dict[str, Any]]) -> str:
    values: list[str] = []
    for block in blocks:
        kind = block.get("type")
        if kind in {"heading", "paragraph"}:
            values.append(block.get("text", ""))
        elif kind == "list":
            values.extend(block.get("items", []))
        elif kind == "resource_group":
            values.extend([block.get("title", ""), block.get("content_text", "")])
        elif kind == "definition":
            values.extend([block.get("term", ""), block.get("definition_text", "")])
        elif kind == "faq":
            values.extend([block.get("question", ""), block.get("answer_text", "")])
        elif kind == "table":
            values.extend(block.get("headers", []))
            for row in block.get("rows", []):
                values.extend(row)
    return norm(" ".join(values))


def iter_blocks_recursive(blocks: list[dict[str, Any]]) -> Iterable[dict[str, Any]]:
    for block in blocks:
        yield block
        for key in ("definition_blocks", "answer_blocks", "content_blocks"):
            children = block.get(key, [])
            if isinstance(children, list):
                yield from iter_blocks_recursive(children)


def count_block_type(blocks: list[dict[str, Any]], target: str) -> int:
    return sum(1 for block in iter_blocks_recursive(blocks) if block.get("type") == target)


def block_fingerprint(block: dict[str, Any]) -> str:
    return sha256_text(json.dumps(block, ensure_ascii=False, sort_keys=True))


def apply_document_filters(document: dict[str, Any]) -> None:
    excluded: list[dict[str, Any]] = []
    cleaned: list[dict[str, Any]] = []
    for block in document.get("blocks", []):
        text = blocks_text([block])
        if document.get("policy", {}).get("webtoon_policy") == "reference_only_official_page" and any(
            token in text for token in ["예튼이", "예솜이", "예툰이", "웹툰"]
        ):
            excluded.append({"reason": "웹툰 참고 링크 전용 정책", "content": text})
            continue
        if document["doc_id"] == "MT-009" and any(token in text for token in ["5천만 원", "5천만원", "1천만원 이하", "1천만원 초과"]):
            excluded.append({"reason": "구버전 지원금액이 포함된 웹툰 설명", "content": text})
            continue
        if document["doc_id"] == "HP-001" and block.get("type") == "table":
            headers = block.get("headers", [])
            rows = block.get("rows", [])
            if "회수기여도" in headers and rows and any("예상실회수금액" in norm(cell) for row in rows for cell in row):
                excluded.append({"reason": "포상금 자동계산기 실행 전 초기값 표", "content": text})
                continue
        if text in {".", "-", "·", "|"}:
            continue
        if is_noise_text(text):
            continue
        cleaned.append(block)
    document["blocks"] = cleaned
    if excluded:
        document["excluded_content"] = excluded
        document["content_filter"] = "legacy_amount_removed"


def refresh_document(document: dict[str, Any]) -> None:
    document["content_markdown"] = render_blocks(document.get("blocks", []))
    document["body_markdown"] = document["content_markdown"]
    document["content_text"] = blocks_text(document.get("blocks", []))
    document["markdown_export"] = render_document_markdown(document)
    document["export_markdown"] = document["markdown_export"]
    parsed_hash = sha256_text(document["content_markdown"])
    document["parsed_content_sha256"] = parsed_hash
    document["content_hash"] = parsed_hash


def select_content_root(soup: BeautifulSoup) -> Tag:
    root = (
        soup.select_one(".contents") or soup.select_one("#contents") or
        soup.select_one("#content") or soup.select_one("main") or soup.body
    )
    if not root:
        raise ValueError("본문 루트를 찾지 못했습니다.")
    return root


def extract_page_heading(soup: BeautifulSoup, fallback: str) -> str:
    selectors = [
        ".pageTit h1", ".pageTit h2", ".page-title h1", ".page-title h2",
        ".subTitle h1", ".subTitle h2", "main h1",
    ]
    for selector in selectors:
        node = soup.select_one(selector)
        if node:
            value = safe_text(node)
            if value and not is_noise_text(value):
                return value
    parts = [norm(part) for part in re.split(r"\s*>\s*", fallback or "") if norm(part)]
    return parts[-1] if parts else norm(fallback)


def clone_clean_content_root(root: Tag) -> Tag:
    fragment = BeautifulSoup(str(root), "lxml")
    cloned = fragment.body.find() if fragment.body else fragment.find()
    if not cloned:
        raise ValueError("본문 복제에 실패했습니다.")
    for selector in NOISE_SELECTORS:
        for node in cloned.select(selector):
            node.decompose()
    return cloned



def remove_reference_only_webtoon(root: Tag, manifest_row: dict[str, str]) -> list[dict[str, str]]:
    """웹툰은 공식 페이지 링크만 남기고 DOM 본문·청킹에서는 완전히 제외합니다."""
    if manifest_row.get("webtoon_policy") != "reference_only_official_page":
        return []
    removed: list[dict[str, str]] = []
    targets: list[Tag] = []
    for image in root.find_all("img"):
        value = " ".join([
            image.get("src", ""), image.get("data-src", ""),
            image.get("alt", ""), image.get("title", ""),
        ]).lower()
        if "webtoon" not in value and "웹툰" not in value:
            continue
        target: Tag = image
        for parent in image.parents:
            if not isinstance(parent, Tag) or parent is root:
                break
            tokens = " ".join(parent.get("class", [])).lower() + " " + norm(parent.get("id", "")).lower()
            target = parent
            if any(token in tokens for token in ["sliderwrap", "swiper", "webtoon"]):
                if "sliderwrap" in tokens:
                    break
        targets.append(target)
    for heading in root.find_all(["h2", "h3", "h4"]):
        text = safe_text(heading)
        if any(token in text for token in ["예툰이와 예솜이", "예튼이와 예솜이"]):
            targets.append(heading.find_parent(class_=re.compile(r"topTit", re.I)) or heading)
    seen: set[int] = set()
    for target in targets:
        if id(target) in seen or target.parent is None:
            continue
        seen.add(id(target))
        text = norm(target.get_text(" ", strip=True))
        removed.append({"reason": "웹툰 참고 링크 전용 정책", "content": text[:1000]})
        target.decompose()
    return removed


def synchronize_document_navigation(document: dict[str, Any]) -> None:
    """모든 Action을 links와 검수용 Markdown에 같은 순서로 반영합니다."""
    content_links = [item for item in document.get("links", []) if item.get("link_role") == "content"]
    document["links"] = merge_links_with_actions(content_links, document.get("actions", []))
    document["markdown_export"] = render_document_markdown(document)
    document["export_markdown"] = document["markdown_export"]


def parse_html_document(
    html_bytes: bytes,
    final_url: str,
    manifest_row: dict[str, str],
    encoding: str,
) -> dict[str, Any]:
    soup = BeautifulSoup(html_bytes.decode(encoding, errors="replace"), "lxml")
    original_root = select_content_root(soup)
    page_heading = extract_page_heading(soup, manifest_row["page_title"])
    display_title = resolve_display_title(manifest_row, page_heading)
    root = clone_clean_content_root(original_root)

    attachments = extract_attachments(root, final_url, manifest_row["url_id"])
    actions = extract_actions(root, final_url, manifest_row, attachments)
    if not actions and manifest_row.get("page_type") == "dynamic_lookup":
        actions = [{
            "action_id": sha256_text(f"{manifest_row['url_id']}|{manifest_row['source_url']}|lookup")[:16],
            "label": f"{manifest_row['page_title']} 조회 바로가기",
            "url": manifest_row["source_url"],
            "action_type": "lookup",
            "source_tag": "source_page",
            "auth_required": manifest_row.get("action_auth", "불필요"),
            "official_domain": True,
            "requires_review": False,
        }]
    videos = extract_videos(root, final_url, manifest_row)
    supplementary_links = extract_supplementary_links(root, final_url, manifest_row, videos)
    poster_urls = {item.get("poster_url") for item in videos if item.get("poster_url")}
    images = extract_images(root, final_url, manifest_row, poster_urls)
    content_links = extract_links(root, final_url)
    links = merge_links_with_actions(content_links, actions)

    source_fragment = BeautifulSoup(str(root), "lxml")
    source_root = source_fragment.body.find() if source_fragment.body else source_fragment.find()
    if source_root:
        for selector in NOISE_SELECTORS:
            for node in source_root.select(selector):
                node.decompose()
        source_text = norm(source_root.get_text(" ", strip=True))
    else:
        source_text = ""

    for media_node in root.find_all(["video", "source", "iframe", "embed", "object"]):
        media_node.decompose()
    removed_webtoon = remove_reference_only_webtoon(root, manifest_row)

    parser = StructureParser(manifest_row.get("page_type", "static_page"))
    blocks = parser.parse(root)
    document = {
        "doc_id": manifest_row["url_id"],
        "parent_doc_id": manifest_row["url_id"],
        "record_type": "page",
        "indexable": manifest_row.get("rag_index_mode") != "none",
        "rag_index_mode": manifest_row.get("rag_index_mode", "full"),
        "title": display_title,
        "manifest_title": manifest_row["page_title"],
        "page_heading": page_heading,
        "display_title": display_title,
        "html_title": norm(soup.title.get_text(" ", strip=True)) if soup.title else "",
        "source_url": manifest_row["source_url"],
        "final_url": final_url,
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get("target_business_function", manifest_row["business_function"]),
        "page_type": manifest_row["page_type"],
        "parser_profile": manifest_row["parser_profile"],
        "breadcrumb": [manifest_row["business_function"], manifest_row["page_title"]],
        "blocks": blocks,
        "links": links,
        "actions": actions,
        "attachments": attachments,
        "images": images,
        "videos": videos,
        "supplementary_links": supplementary_links,
        "policy": {
            "normalized_decision": manifest_row["normalized_decision"],
            "content_scope": manifest_row["content_scope"],
            "rag_index_label": manifest_row["rag_index_label"],
            "action_policy": manifest_row["action_policy"],
            "expected_action_types": manifest_row["expected_action_types"],
            "pagination_policy": manifest_row["pagination_policy"],
            "auth_required": manifest_row["auth_required"],
            "attachment_download_policy": manifest_row["attachment_download_policy"],
            "attachment_rag_policy": manifest_row["attachment_rag_policy"],
            "attachment_user_delivery_policy": manifest_row["attachment_user_delivery_policy"],
            "video_policy": manifest_row["video_policy"],
            "webtoon_policy": manifest_row["webtoon_policy"],
            "image_policy": manifest_row["image_policy"],
        },
        "parsed_at": now_kst_iso(),
    }
    if removed_webtoon:
        document.setdefault("excluded_content", []).extend(removed_webtoon)
    apply_document_filters(document)
    refresh_document(document)
    synchronize_document_navigation(document)
    document["quality"] = build_quality(document, source_text, manifest_row)
    return document


def build_quality(document: dict[str, Any], source_text: str, row: dict[str, str]) -> dict[str, Any]:
    blocks = document.get("blocks", [])
    content = document.get("content_text", "")
    faq_count = count_block_type(blocks, "faq")
    table_count = count_block_type(blocks, "table")
    markdown_lines = {norm(line.lstrip("#-0123456789. ")) for line in document.get("content_markdown", "").splitlines()}
    noise_hits = [value for value in NOISE_EXACT_TEXTS if value in markdown_lines]
    reasons: list[str] = []
    if document.get("indexable", True) and len(content) < 80:
        reasons.append("본문이 80자 미만")
    retention = round(len(content) / max(1, len(source_text)), 3)
    if document.get("indexable", True) and retention < 0.55:
        reasons.append("본문 보존율 55% 미만")
    if noise_hits:
        reasons.append("공통 UI 문구 잔존")
    if row.get("page_type") == "faq" and faq_count == 0:
        reasons.append("FAQ 질문-답변 추출 실패")
    if row.get("action_policy", "").startswith("O") and not document.get("actions"):
        reasons.append("예상 Action 링크 미검출")
    status = "fail" if any(value in reasons for value in ["공통 UI 문구 잔존", "FAQ 질문-답변 추출 실패"]) else ("review" if reasons else "pass")
    return {
        "status": status,
        "reasons": reasons,
        "source_text_chars": len(source_text),
        "parsed_text_chars": len(content),
        "retention_ratio": retention,
        "faq_count": faq_count,
        "table_count": table_count,
        "attachment_button_count": len(document.get("attachments", [])),
        "action_count": len(document.get("actions", [])),
        "video_count": len(document.get("videos", [])),
        "supplementary_link_count": len(document.get("supplementary_links", [])),
        "noise_hits": noise_hits,
    }


# ============================================================
# 공통 페이지네이션 수집
# ============================================================

@dataclass
class PaginationPlan:
    method: str
    endpoint: str
    page_param: str
    page_size_param: str | None
    page_size: int
    last_internal_index: int
    index_base: int
    base_payload: dict[str, str]
    detection_method: str


def extract_form_payload(form: Tag | None) -> dict[str, str]:
    payload: dict[str, str] = {}
    if not form:
        return payload
    for field in form.select("input[name], select[name], textarea[name]"):
        name = norm(field.get("name", ""))
        if not name:
            continue
        if field.name == "select":
            selected = field.find("option", selected=True) or field.find("option")
            value = selected.get("value", "") if selected else ""
        elif field.name == "textarea":
            value = field.get_text("", strip=False)
        else:
            field_type = norm(field.get("type", "text")).lower()
            if field_type in {"checkbox", "radio"} and not field.has_attr("checked"):
                continue
            value = field.get("value", "")
        payload[name] = str(value or "")
    return payload


def extract_script_text(soup: BeautifulSoup) -> str:
    return "\n".join(script.get_text("\n", strip=True) for script in soup.find_all("script") if not script.get("src"))


def detect_pagination_plan(
    html_bytes: bytes,
    encoding: str,
    final_url: str,
    row: dict[str, str],
    config: PipelineConfig,
) -> PaginationPlan | None:
    if not config.enable_generic_pagination:
        return None
    policy = row.get("pagination_policy", "")
    if "제외" in policy:
        return None

    soup = BeautifulSoup(html_bytes.decode(encoding, errors="replace"), "lxml")
    onclick_indexes: list[int] = []
    for node in soup.select(".paging [onclick*='chgPagingNo'], .pagination [onclick*='chgPagingNo']"):
        match = re.search(r"chgPagingNo\(\s*(\d+)\s*\)", node.get("onclick", ""))
        if match:
            onclick_indexes.append(int(match.group(1)))

    if onclick_indexes:
        last_index = max(onclick_indexes)
        if last_index <= 0:
            return None
        form = soup.select_one("form#srchForm") or soup.select_one("form[name='srchForm']")
        payload = extract_form_payload(form)
        scripts = extract_script_text(soup)
        action = norm(form.get("action", "")) if form else ""
        if not action:
            match = re.search(r"[\"']#srchForm[\"']\)\.attr\(\s*[\"']action[\"']\s*,\s*[\"']([^\"']+)[\"']", scripts)
            if not match:
                match = re.search(r"attr\(\s*[\"']action[\"']\s*,\s*[\"']([^\"']+)[\"']", scripts)
            action = match.group(1) if match else ""
        endpoint = urljoin(final_url, action) if action else final_url
        page_param = "curPage"
        for candidate in ["curPage", "pageIndex", "pageNo", "currentPage", "page"]:
            if candidate in payload or re.search(rf"name\s*[:=]\s*['\"]{candidate}['\"]", scripts):
                page_param = candidate
                break
        page_size_param = "pageSize" if "pageSize" in payload or "pageSize" in scripts else None
        return PaginationPlan(
            method="POST", endpoint=endpoint, page_param=page_param,
            page_size_param=page_size_param, page_size=config.pagination_page_size,
            last_internal_index=last_index, index_base=0, base_payload=payload,
            detection_method="chgPagingNo",
        )

    # query string 기반 페이지 링크도 지원합니다.
    page_links: list[tuple[str, str, int]] = []
    for anchor in soup.select(".paging a[href], .pagination a[href]"):
        href = anchor.get("href", "")
        if not href or href.startswith("javascript:"):
            continue
        absolute = urljoin(final_url, href)
        query = parse_qs(urlparse(absolute).query)
        for param in ["curPage", "pageIndex", "pageNo", "currentPage", "page"]:
            if param in query and query[param] and str(query[param][0]).isdigit():
                page_links.append((absolute, param, int(query[param][0])))
    if page_links:
        _, param, max_page = max(page_links, key=lambda item: item[2])
        if max_page <= 1:
            return None
        return PaginationPlan(
            method="GET", endpoint=final_url, page_param=param, page_size_param=None,
            page_size=config.pagination_page_size, last_internal_index=max_page,
            index_base=1, base_payload={}, detection_method="query_link",
        )
    return None


def displayed_page_number(html_bytes: bytes, encoding: str) -> int | None:
    soup = BeautifulSoup(html_bytes.decode(encoding, errors="replace"), "lxml")
    current = soup.select_one(".paging strong[title*='현재 페이지'] span") or soup.select_one(".pagination .active")
    if not current:
        return None
    text = norm(current.get_text(" ", strip=True))
    return int(text) if text.isdigit() else None


def repeatable_signature(document: dict[str, Any]) -> str:
    selected = [
        block for block in document.get("blocks", [])
        if block.get("type") in {"faq", "table", "list", "definition"}
    ]
    if not selected:
        selected = document.get("blocks", [])
    return sha256_text(json.dumps(selected, ensure_ascii=False, sort_keys=True))


def merge_table_blocks(target: dict[str, Any], incoming: dict[str, Any]) -> None:
    existing = {tuple(row) for row in target.get("rows", [])}
    for row in incoming.get("rows", []):
        key = tuple(row)
        if key not in existing:
            existing.add(key)
            target.setdefault("rows", []).append(row)
    target["row_count"] = len(target.get("rows", []))


def merge_paginated_documents(documents: list[dict[str, Any]], row: dict[str, str]) -> dict[str, Any]:
    merged = documents[0]
    page_type = row.get("page_type", "")

    if page_type == "faq":
        base_non_faq = [block for block in merged["blocks"] if block.get("type") != "faq"]
        faq_blocks: list[dict[str, Any]] = []
        seen: set[tuple[str, str]] = set()
        for document in documents:
            for block in document.get("blocks", []):
                if block.get("type") != "faq":
                    continue
                key = (norm(block.get("question", "")), norm(block.get("answer_text", "")))
                if key in seen:
                    continue
                seen.add(key)
                faq_blocks.append(block)
        merged["blocks"] = base_non_faq + faq_blocks
    else:
        # 같은 헤더를 가진 표는 행을 합치고, 나머지 새 블록은 중복 없이 추가합니다.
        table_by_headers: dict[tuple[str, ...], dict[str, Any]] = {
            tuple(block.get("headers", [])): block
            for block in merged.get("blocks", []) if block.get("type") == "table"
        }
        fingerprints = {block_fingerprint(block) for block in merged.get("blocks", [])}
        for document in documents[1:]:
            for block in document.get("blocks", []):
                if block.get("type") == "table":
                    key = tuple(block.get("headers", []))
                    if key in table_by_headers:
                        merge_table_blocks(table_by_headers[key], block)
                        continue
                fingerprint = block_fingerprint(block)
                if fingerprint not in fingerprints:
                    fingerprints.add(fingerprint)
                    merged["blocks"].append(block)

    for field, keys in [
        ("actions", ("url", "label")), ("attachments", ("attachment_id",)),
        ("images", ("url",)), ("videos", ("media_url",)),
        ("supplementary_links", ("url", "link_type")),
        ("links", ("url", "anchor_text")),
    ]:
        combined: list[dict[str, Any]] = []
        for document in documents:
            combined.extend(document.get(field, []))
        merged[field] = unique_dicts(combined, keys)

    apply_document_filters(merged)
    refresh_document(merged)
    return merged


def collect_paginated_document(
    session: requests.Session,
    first_result: FetchResult,
    first_document: dict[str, Any],
    row: dict[str, str],
    paths: OutputPaths,
    config: PipelineConfig,
) -> tuple[dict[str, Any], dict[str, Any]]:
    plan = detect_pagination_plan(first_result.body, first_result.encoding, first_result.final_url, row, config)
    if plan is None:
        collection = {
            "collection_scope": "single_public_page",
            "is_complete": True,
            "pagination_detected": False,
            "expected_page_count": 1,
            "fetched_page_count": 1,
            "failures": [],
        }
        first_document["pagination_collection"] = collection
        return first_document, collection

    if plan.last_internal_index + 1 > config.pagination_max_pages:
        raise ValueError(
            f"{row['url_id']} 페이지 수가 안전 한도 초과: {plan.last_internal_index + 1}"
        )

    page_dir = paths.pagination / row["url_id"]
    page_dir.mkdir(parents=True, exist_ok=True)
    first_page_path = page_dir / "page_001.html"
    first_page_path.write_bytes(first_result.body)

    page_documents = [first_document]
    page_records: list[dict[str, Any]] = [{
        "internal_page_index": 0 if plan.index_base == 0 else 1,
        "displayed_page_number": displayed_page_number(first_result.body, first_result.encoding),
        "status_code": first_result.status_code,
        "request_method": "GET",
        "final_url": first_result.final_url,
        "signature": repeatable_signature(first_document),
        "raw_sha256": first_result.raw_sha256,
        "raw_html_path": str(first_page_path.relative_to(paths.root)),
    }]
    failures: list[dict[str, Any]] = []
    seen_signatures = {page_records[0]["signature"]: page_records[0]["internal_page_index"]}

    if plan.index_base == 0:
        page_indexes = range(1, plan.last_internal_index + 1)
    else:
        page_indexes = range(2, plan.last_internal_index + 1)

    for page_index in page_indexes:
        try:
            started = time.perf_counter()
            if plan.method == "POST":
                payload = dict(plan.base_payload)
                payload[plan.page_param] = str(page_index)
                if plan.page_size_param:
                    payload[plan.page_size_param] = str(plan.page_size)
                response = session.post(
                    plan.endpoint, data=payload, headers={"Referer": first_result.final_url},
                    timeout=config.request_timeout_seconds, allow_redirects=True,
                )
                request_payload: dict[str, Any] = payload
            else:
                parsed = urlparse(plan.endpoint)
                query = parse_qs(parsed.query)
                query[plan.page_param] = [str(page_index)]
                query_string = urlencode(query, doseq=True)
                url = urlunparse(parsed._replace(query=query_string))
                response = session.get(url, timeout=config.request_timeout_seconds, allow_redirects=True)
                request_payload = {plan.page_param: page_index}
            response.raise_for_status()
            result = response_to_fetch_result(response, plan.endpoint, time.perf_counter() - started)
            page_document = parse_html_document(result.body, result.final_url, row, result.encoding)
            signature = repeatable_signature(page_document)
            page_number = page_index + 1 if plan.index_base == 0 else page_index
            page_path = page_dir / f"page_{page_number:03d}.html"
            page_path.write_bytes(result.body)
            record = {
                "internal_page_index": page_index,
                "displayed_page_number": displayed_page_number(result.body, result.encoding),
                "status_code": result.status_code,
                "request_method": plan.method,
                "request_payload": request_payload,
                "final_url": result.final_url,
                "signature": signature,
                "raw_sha256": result.raw_sha256,
                "raw_html_path": str(page_path.relative_to(paths.root)),
            }
            if signature in seen_signatures:
                failures.append({
                    "internal_page_index": page_index,
                    "reason": "다른 페이지와 같은 반복 콘텐츠",
                    "same_as": seen_signatures[signature],
                })
            else:
                seen_signatures[signature] = page_index
            expected_display = page_number
            actual_display = record["displayed_page_number"]
            if actual_display is not None and actual_display != expected_display:
                failures.append({
                    "internal_page_index": page_index,
                    "reason": "요청 페이지와 표시 페이지 불일치",
                    "expected": expected_display,
                    "actual": actual_display,
                })
            page_records.append(record)
            page_documents.append(page_document)
        except Exception as error:
            failures.append({
                "internal_page_index": page_index,
                "reason": "페이지 요청 또는 파싱 실패",
                "error_type": type(error).__name__,
                "error": str(error),
            })
        time.sleep(config.request_delay_seconds)

    expected_count = plan.last_internal_index + 1 if plan.index_base == 0 else plan.last_internal_index
    is_complete = not failures and len(page_documents) == expected_count
    merged = merge_paginated_documents(page_documents, row)
    collection = {
        "collection_scope": "all_public_pages" if is_complete else "partial_public_pages",
        "is_complete": is_complete,
        "pagination_detected": True,
        "detection_method": plan.detection_method,
        "method": plan.method,
        "endpoint": plan.endpoint,
        "page_param": plan.page_param,
        "index_base": plan.index_base,
        "expected_page_count": expected_count,
        "fetched_page_count": len(page_documents),
        "pages": page_records,
        "failures": failures,
        "collected_at": now_kst_iso(),
    }
    merged["pagination_collection"] = collection
    merged["dynamic_collection"] = collection if row["url_id"] == "BI-004" else None
    return merged, collection


# ============================================================
# 자산 다운로드
# ============================================================



def ensure_playwright() -> None:
    try:
        import playwright  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"], check=True)
    subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)




# ============================================================
# 문서 저장과 청킹
# ============================================================

def save_document(paths: OutputPaths, document: dict[str, Any], row: dict[str, str]) -> tuple[Path, Path]:
    folder = safe_name(row["business_function"])
    md_path = paths.parsed_markdown / folder / f"{row['url_id']}.md"
    json_path = paths.parsed_json / folder / f"{row['url_id']}.json"
    ensure_parent(md_path)
    markdown = document.get("markdown_export") or render_document_markdown(document)
    md_path.write_text(markdown, encoding="utf-8")
    write_json(json_path, document)
    return md_path, json_path


def render_table_chunk(headers: list[str], rows: list[list[str]]) -> str:
    return render_blocks([{"type": "table", "headers": headers, "rows": rows}])


def split_table(block: dict[str, Any], max_chars: int, heading_prefix: str = "") -> list[str]:
    headers, rows = block.get("headers", []), block.get("rows", [])
    prefix = f"### {heading_prefix}\n\n" if heading_prefix else ""
    complete = prefix + render_table_chunk(headers, rows)
    if len(complete) <= max_chars:
        return [complete]
    chunks: list[str] = []
    current: list[list[str]] = []
    for row in rows:
        candidate = prefix + render_table_chunk(headers, current + [row])
        if current and len(candidate) > max_chars:
            chunks.append(prefix + render_table_chunk(headers, current))
            current = [row]
        else:
            current.append(row)
    if current:
        chunks.append(prefix + render_table_chunk(headers, current))
    return chunks


def meaningful_chunk(text: str, min_chars: int) -> bool:
    plain = re.sub(r"[#|\-\s]", "", text)
    if len(plain) < min_chars:
        return False
    if any(fragment in text for fragment in NOISE_CONTAINS_TEXTS):
        return False
    return True




def build_link_only_document(row: dict[str, str]) -> dict[str, Any]:
    expected = [value.strip() for value in row.get("expected_action_types", "").split("|") if value.strip()]
    action_type = expected[0] if expected else "related_service"
    display_title = resolve_display_title(row, "")
    label_suffix = {
        "self_check": "자가진단", "lookup": "조회", "apply": "신청",
        "progress": "진행현황 조회", "consult": "상담",
    }.get(action_type, "바로가기")
    label = display_title if label_suffix in display_title else f"{display_title} {label_suffix}"
    action = {
        "action_id": sha256_text(f"{row['url_id']}|{row['source_url']}")[:16],
        "label": label,
        "raw_label": row["page_title"],
        "url": row["source_url"],
        "action_type": action_type,
        "source_tag": "source_page",
        "auth_required": row.get("action_auth", ""),
        "official_domain": allowed_action_domain(row["source_url"]),
        "requires_review": False,
    }
    document = {
        "doc_id": row["url_id"], "parent_doc_id": row["url_id"],
        "record_type": "link_only", "indexable": False, "rag_index_mode": "none",
        "title": display_title, "manifest_title": row["page_title"],
        "page_heading": "", "display_title": display_title,
        "source_url": row["source_url"],
        "site_name": row["site_name"], "business_function": row["business_function"],
        "target_business_function": row["target_business_function"],
        "page_type": row["page_type"], "content_markdown": "", "body_markdown": "",
        "content_text": "", "blocks": [], "attachments": [], "images": [], "videos": [],
        "supplementary_links": [], "links": [], "actions": [action],
        "policy": {
            "normalized_decision": row["normalized_decision"],
            "content_scope": row["content_scope"], "action_policy": row["action_policy"],
            "auth_required": row["auth_required"],
            "attachment_download_policy": row.get("attachment_download_policy", "X"),
            "attachment_rag_policy": row.get("attachment_rag_policy", "none"),
            "attachment_user_delivery_policy": row.get("attachment_user_delivery_policy", "none"),
            "video_policy": row.get("video_policy", "none"),
            "webtoon_policy": row.get("webtoon_policy", "none"),
            "image_policy": row.get("image_policy", "metadata_only_nondecorative"),
        },
        "quality": {"status": "pass", "reasons": [], "action_count": 1},
        "parsed_at": now_kst_iso(),
    }
    synchronize_document_navigation(document)
    return document


# ============================================================
# 전체 실행
# ============================================================

# ============================================================
# 첨부파일 하이브리드 처리 및 무손실 청킹 확장
# ============================================================

import csv
import io
import struct
import xml.etree.ElementTree as ET
import zipfile
import zlib
from email.message import Message

FORM_NO_INDEX_KEYWORDS = {
    "신청서", "청구서", "지급청구서", "위임장", "동의서", "개인정보",
    "서약서", "확인서", "철회서", "신고서", "계좌신고서", "통지서",
    "양식", "서식", "작성용", "채권양도", "인감증명",
}
ATTACHMENT_INDEX_KEYWORDS = {
    "안내", "작성요령", "작성 방법", "설명", "매뉴얼", "가이드", "리플릿",
    "절차", "기준", "목록", "현황", "대상 금융회사", "구비서류 안내",
    "주요내용", "업무편람", "FAQ", "자주 묻는 질문",
}
SUPPORTED_ATTACHMENT_TEXT_FORMATS = {
    "PDF", "HWP", "HWPX", "DOCX", "XLSX", "PPTX", "CSV", "TXT",
}
OLE_MAGIC = bytes.fromhex("D0CF11E0A1B11AE1")


def extract_attachments(root: Tag, base_url: str, parent_doc_id: str) -> list[dict[str, Any]]:
    """다운로드 방식과 공식 링크를 함께 보존합니다."""
    attachments: list[dict[str, Any]] = []
    seen: set[Any] = set()

    for button in root.find_all(["button", "a"], onclick=True):
        match = DOWNLOAD_CALL_RE.search(button.get("onclick", ""))
        if not match:
            continue
        key = (match.group(1), match.group(2))
        if key in seen:
            continue
        seen.add(key)
        fmt = infer_file_format(button)
        label = re.sub(r"다운로드$", "", norm(button.get_text(" ", strip=True))).strip()
        row_context = ""
        parent_row = button.find_parent("tr")
        if parent_row:
            values = []
            for cell in parent_row.find_all(["th", "td"], recursive=False):
                if button in cell.descendants:
                    continue
                value = cell_text(cell)
                if value and "다운로드" not in value and value not in values:
                    values.append(value)
            row_context = " | ".join(values[:4])
        display_name = label or row_context or f"{fmt} 첨부파일"
        if fmt not in display_name.upper():
            display_name = f"{display_name} ({fmt})"
        attachments.append({
            "attachment_id": sha256_text(
                f"{parent_doc_id}|{base_url}|{match.group(1)}|{match.group(2)}|{display_name}|{fmt}"
            )[:16],
            "display_name": display_name,
            "file_format": fmt,
            "download_method": "gfn_downloadFile",
            "token1": match.group(1),
            "token2": match.group(2),
            "row_context": row_context,
            "button_text": norm(button.get_text(" ", strip=True)),
            "source_page_url": base_url,
            "source_url": base_url,
            "official_download_url": None,
            "user_action_url": base_url,
            "delivery_mode": "official_download_page",
            "download_status": "metadata_only",
            "processing_policy": "metadata_only",
            "indexable": False,
            "needs_review": False,
        })

    for anchor in root.find_all("a", href=True):
        url = absolute_url(base_url, anchor.get("href"))
        if not url:
            continue
        ext = Path(urlparse(url).path).suffix.lower()
        if ext not in ATTACHMENT_EXTENSIONS:
            continue
        key = ("href", url)
        if key in seen:
            continue
        seen.add(key)
        attachments.append({
            "attachment_id": sha256_text(f"{parent_doc_id}|{url}")[:16],
            "display_name": norm(anchor.get_text(" ", strip=True)) or Path(urlparse(url).path).name,
            "file_format": ext.lstrip(".").upper(),
            "download_method": "href",
            "url": url,
            "source_page_url": base_url,
            "source_url": base_url,
            "official_download_url": url,
            "user_action_url": url,
            "delivery_mode": "direct_official_download",
            "download_status": "metadata_only",
            "processing_policy": "metadata_only",
            "indexable": False,
            "needs_review": False,
        })
    # PC·모바일 DOM에 같은 파일 버튼이 중복된 경우 표시명·형식 기준으로 하나만 유지합니다.
    deduped: list[dict[str, Any]] = []
    semantic_seen: set[tuple[str, str, str]] = set()
    for item in attachments:
        semantic_name = re.sub(r"\s*\([A-Z0-9]+\)\s*$", "", norm(item.get("display_name", ""))).lower()
        semantic_key = (semantic_name, item.get("file_format", ""), item.get("download_method", ""))
        if semantic_key in semantic_seen:
            continue
        semantic_seen.add(semantic_key)
        deduped.append(item)
    return deduped


def _filename_from_content_disposition(value: str) -> str | None:
    if not value:
        return None
    message = Message()
    message["content-disposition"] = value
    filename = message.get_filename()
    return filename.strip() if filename else None


def detect_attachment_format(path: Path, declared: str = "") -> str:
    data = path.read_bytes()[:8192]
    declared = (declared or path.suffix.lstrip(".")).upper()
    if data.startswith(b"%PDF-"):
        return "PDF"
    if data.startswith(OLE_MAGIC):
        return "HWP" if declared == "HWP" else declared or "OLE"
    if data.startswith(b"PK\x03\x04"):
        try:
            with zipfile.ZipFile(path) as archive:
                names = set(archive.namelist())
                lower_names = {name.lower() for name in names}
                if "mimetype" in names:
                    mime = archive.read("mimetype").decode("utf-8", errors="ignore")
                    if "hwp+zip" in mime:
                        return "HWPX"
                if any(name.startswith("word/") for name in lower_names):
                    return "DOCX"
                if any(name.startswith("xl/") for name in lower_names):
                    return "XLSX"
                if any(name.startswith("ppt/") for name in lower_names):
                    return "PPTX"
                if any(name.startswith("contents/section") for name in lower_names):
                    return "HWPX"
        except Exception:
            pass
        return declared or "ZIP"
    if declared in {"CSV", "TXT"}:
        return declared
    return declared or "UNKNOWN"


def inspect_downloaded_attachment(path: Path, declared: str = "") -> dict[str, Any]:
    data = path.read_bytes()[:8192]
    lower = data.lstrip().lower()
    html_like = lower.startswith((b"<!doctype html", b"<html", b"<script"))
    detected = detect_attachment_format(path, declared)
    return {
        "detected_format": detected,
        "signature_valid": not html_like and path.stat().st_size > 0,
        "html_error_response": html_like,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_bytes(path.read_bytes()),
    }


def _clean_extracted_text(text: str, max_chars: int) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[\u0000-\u0008\u000b\u000c\u000e-\u001f]", " ", text)
    lines = []
    for line in text.splitlines():
        value = norm(line)
        if value and (not lines or value != lines[-1]):
            lines.append(value)
    return "\n".join(lines)[:max_chars].strip()


def extract_pdf_text(path: Path, config: PipelineConfig) -> str:
    from pypdf import PdfReader
    reader = PdfReader(str(path))
    texts = []
    for page in reader.pages[: config.attachment_pdf_max_pages]:
        texts.append(page.extract_text() or "")
    return _clean_extracted_text("\n".join(texts), config.attachment_max_text_chars)


def extract_docx_or_pptx_text(path: Path, config: PipelineConfig) -> str:
    values: list[str] = []
    with zipfile.ZipFile(path) as archive:
        targets = [
            name for name in archive.namelist()
            if (
                name.startswith("word/") and name.endswith(".xml")
            ) or (
                name.startswith("ppt/slides/") and name.endswith(".xml")
            )
        ]
        for name in sorted(targets):
            try:
                root = ET.fromstring(archive.read(name))
                text = " ".join(value.strip() for value in root.itertext() if value.strip())
                if text:
                    values.append(text)
            except Exception:
                continue
    return _clean_extracted_text("\n".join(values), config.attachment_max_text_chars)


def extract_hwpx_text(path: Path, config: PipelineConfig) -> str:
    values: list[str] = []
    with zipfile.ZipFile(path) as archive:
        targets = [
            name for name in archive.namelist()
            if name.lower().startswith("contents/section") and name.lower().endswith(".xml")
        ]
        for name in sorted(targets):
            try:
                root = ET.fromstring(archive.read(name))
                text = " ".join(value.strip() for value in root.itertext() if value.strip())
                if text:
                    values.append(text)
            except Exception:
                continue
    return _clean_extracted_text("\n".join(values), config.attachment_max_text_chars)


def extract_xlsx_text(path: Path, config: PipelineConfig) -> str:
    ns = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    values: list[str] = []
    with zipfile.ZipFile(path) as archive:
        shared: list[str] = []
        if "xl/sharedStrings.xml" in archive.namelist():
            root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
            for si in root.findall("m:si", ns):
                shared.append("".join(si.itertext()).strip())
        sheets = sorted(
            name for name in archive.namelist()
            if name.startswith("xl/worksheets/sheet") and name.endswith(".xml")
        )
        for sheet_no, name in enumerate(sheets, start=1):
            root = ET.fromstring(archive.read(name))
            values.append(f"[시트 {sheet_no}]")
            for row in root.findall(".//m:row", ns):
                row_values = []
                for cell in row.findall("m:c", ns):
                    cell_type = cell.get("t", "")
                    value_node = cell.find("m:v", ns)
                    inline = cell.find("m:is", ns)
                    value = ""
                    if cell_type == "s" and value_node is not None:
                        try:
                            value = shared[int(value_node.text or "0")]
                        except Exception:
                            value = value_node.text or ""
                    elif cell_type == "inlineStr" and inline is not None:
                        value = "".join(inline.itertext())
                    elif value_node is not None:
                        value = value_node.text or ""
                    row_values.append(norm(value))
                if any(row_values):
                    values.append(" | ".join(row_values))
    return _clean_extracted_text("\n".join(values), config.attachment_max_text_chars)


def extract_hwp_text(path: Path, config: PipelineConfig) -> str:
    import olefile
    ole = olefile.OleFileIO(str(path))
    try:
        header = ole.openstream("FileHeader").read()
        compressed = len(header) > 36 and bool(header[36] & 1)
        sections = []
        for entry in ole.listdir():
            joined = "/".join(entry)
            if joined.startswith("BodyText/Section"):
                try:
                    number = int(re.search(r"Section(\d+)", joined).group(1))
                except Exception:
                    number = 0
                sections.append((number, joined))
        texts: list[str] = []
        for _, stream_name in sorted(sections):
            data = ole.openstream(stream_name).read()
            if compressed:
                try:
                    data = zlib.decompress(data, -15)
                except Exception:
                    continue
            offset = 0
            while offset + 4 <= len(data):
                header_value = struct.unpack_from("<I", data, offset)[0]
                offset += 4
                tag_id = header_value & 0x3FF
                size = (header_value >> 20) & 0xFFF
                if size == 0xFFF:
                    if offset + 4 > len(data):
                        break
                    size = struct.unpack_from("<I", data, offset)[0]
                    offset += 4
                payload = data[offset: offset + size]
                offset += size
                if tag_id == 67:
                    text = payload.decode("utf-16le", errors="ignore")
                    text = re.sub(r"[\x00-\x1f]", " ", text)
                    if norm(text):
                        texts.append(norm(text))
        return _clean_extracted_text("\n".join(texts), config.attachment_max_text_chars)
    finally:
        ole.close()


def extract_attachment_text(path: Path, detected_format: str, config: PipelineConfig) -> tuple[str, str, str]:
    if not config.extract_attachment_text:
        return "", "disabled", ""
    try:
        fmt = detected_format.upper()
        if fmt == "PDF":
            text = extract_pdf_text(path, config)
        elif fmt in {"DOCX", "PPTX"}:
            text = extract_docx_or_pptx_text(path, config)
        elif fmt == "HWPX":
            text = extract_hwpx_text(path, config)
        elif fmt == "XLSX":
            text = extract_xlsx_text(path, config)
        elif fmt == "HWP":
            text = extract_hwp_text(path, config)
        elif fmt in {"CSV", "TXT"}:
            raw = path.read_bytes()
            for encoding in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
                try:
                    text = raw.decode(encoding)
                    break
                except Exception:
                    text = ""
            text = _clean_extracted_text(text, config.attachment_max_text_chars)
        else:
            return "", "unsupported", f"지원하지 않는 텍스트 추출 형식: {fmt}"
        if not text:
            return "", "empty_or_scanned", "텍스트가 없거나 스캔 문서일 수 있음"
        return text, "success", ""
    except Exception as error:
        return "", "failed", f"{type(error).__name__}: {error}"


def classify_attachment_policy(attachment: dict[str, Any], config: PipelineConfig) -> dict[str, Any]:
    name = " ".join([
        attachment.get("display_name", ""),
        attachment.get("row_context", ""),
        attachment.get("filename", ""),
    ])
    text = attachment.get("extracted_text", "")
    status = attachment.get("text_extraction_status", "")
    downloaded = attachment.get("download_status") == "downloaded"
    detected = attachment.get("detected_format", attachment.get("file_format", "")).upper()

    if not downloaded:
        return {
            "processing_policy": "metadata_only",
            "indexable": False,
            "classification_reason": "원본 파일 미다운로드",
            "needs_review": attachment.get("download_status") == "failed",
        }
    if any(keyword in name for keyword in FORM_NO_INDEX_KEYWORDS):
        return {
            "processing_policy": "download_no_index",
            "indexable": False,
            "classification_reason": "신청서·동의서·위임장 등 작성용 서식",
            "needs_review": False,
        }
    if status != "success":
        return {
            "processing_policy": "download_no_index" if detected in SUPPORTED_ATTACHMENT_TEXT_FORMATS else "metadata_only",
            "indexable": False,
            "classification_reason": "텍스트 추출 실패·미지원 또는 스캔 문서",
            "needs_review": True,
        }
    if any(keyword in name for keyword in ATTACHMENT_INDEX_KEYWORDS):
        return {
            "processing_policy": "download_and_index",
            "indexable": True,
            "classification_reason": "안내·절차·목록형 자료",
            "needs_review": False,
        }
    if len(text) >= max(config.attachment_index_min_chars, 400):
        return {
            "processing_policy": "download_and_index",
            "indexable": True,
            "classification_reason": "충분한 설명 텍스트가 있는 공개 자료",
            "needs_review": True,
        }
    return {
        "processing_policy": "download_no_index",
        "indexable": False,
        "classification_reason": "짧은 서식 또는 설명 가치가 낮은 파일",
        "needs_review": len(text) >= config.attachment_index_min_chars,
    }


def _write_stream_to_file(response: requests.Response, output: Path, config: PipelineConfig) -> int:
    ensure_parent(output)
    written = 0
    with output.open("wb") as file:
        for chunk in response.iter_content(64 * 1024):
            if not chunk:
                continue
            written += len(chunk)
            if written > config.max_asset_bytes:
                raise ValueError("첨부파일 제한 용량 초과")
            file.write(chunk)
    return written


def download_direct_assets(
    session: requests.Session,
    document: dict[str, Any],
    row: dict[str, str],
    paths: OutputPaths,
    config: PipelineConfig,
) -> dict[str, list[dict[str, Any]]]:
    result = {"attachments": [], "images": [], "failures": []}
    base_dir = paths.raw_assets / safe_name(row["business_function"]) / row["url_id"]

    attachment_download_enabled = (
        document.get("policy", {}).get("attachment_download_policy") == "O"
    )
    if config.download_direct_attachments and attachment_download_enabled:
        for attachment in document.get("attachments", []):
            if attachment.get("download_method") != "href" or not attachment.get("url"):
                continue
            try:
                response = session.get(
                    attachment["url"], timeout=config.request_timeout_seconds,
                    stream=True, allow_redirects=True,
                )
                response.raise_for_status()
                content_type = response.headers.get("Content-Type", "")
                filename = (
                    _filename_from_content_disposition(response.headers.get("Content-Disposition", ""))
                    or Path(urlparse(response.url).path).name
                    or attachment["display_name"]
                )
                filename = f"{attachment['attachment_id']}_{safe_name(filename)}"
                output = base_dir / "attachments" / filename
                _write_stream_to_file(response, output, config)
                inspection = inspect_downloaded_attachment(output, attachment.get("file_format", ""))
                if not inspection["signature_valid"]:
                    raise ValueError("파일 대신 HTML 오류 페이지 또는 빈 응답을 받음")
                attachment.update({
                    "download_status": "downloaded",
                    "filename": filename,
                    "local_path": str(output.relative_to(paths.root)),
                    "mime_type": content_type.split(";", 1)[0].strip(),
                    "official_download_url": response.url,
                    "download_url_final": response.url,
                    "user_action_url": response.url,
                    "delivery_mode": "direct_official_download",
                    **inspection,
                })
                result["attachments"].append(attachment)
            except Exception as error:
                attachment.update({
                    "download_status": "failed",
                    "processing_policy": "metadata_only",
                    "indexable": False,
                    "needs_review": True,
                    "error": f"{type(error).__name__}: {error}",
                })
                result["failures"].append({
                    "type": "attachment", "name": attachment.get("display_name", ""),
                    "error": attachment["error"],
                })

    if config.download_images:
        for image in document.get("images", []):
            if image.get("image_role") == "decorative":
                continue
            try:
                response = session.get(image["url"], timeout=config.request_timeout_seconds, stream=True)
                response.raise_for_status()
                filename = image.get("filename") or Path(urlparse(response.url).path).name
                output = base_dir / "images" / safe_name(filename)
                _write_stream_to_file(response, output, config)
                image.update({
                    "download_status": "downloaded",
                    "download_policy": "download_original_no_index",
                    "local_path": str(output.relative_to(paths.root)),
                    "size_bytes": output.stat().st_size,
                    "sha256": sha256_bytes(output.read_bytes()),
                })
                result["images"].append(dict(image))
            except Exception as error:
                result["failures"].append({
                    "type": "image", "url": image.get("url", ""),
                    "error": f"{type(error).__name__}: {error}",
                })
    return result


def download_js_attachments(
    documents: list[dict[str, Any]], paths: OutputPaths, config: PipelineConfig
) -> list[dict[str, Any]]:
    records = []
    for document in documents:
        if document.get("policy", {}).get("attachment_download_policy") != "O":
            continue
        for attachment in document.get("attachments", []):
            if attachment.get("download_method") == "gfn_downloadFile":
                records.append((document, attachment))
    if not records or not config.download_js_attachments_with_playwright:
        return []
    ensure_playwright()
    from playwright.sync_api import sync_playwright
    results: list[dict[str, Any]] = []
    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        context = browser.new_context(accept_downloads=True, user_agent=config.user_agent, locale="ko-KR")
        grouped: dict[str, list[tuple[dict[str, Any], dict[str, Any]]]] = {}
        for document, attachment in records:
            grouped.setdefault(document["source_url"], []).append((document, attachment))
        for source_url, items in grouped.items():
            page = context.new_page()
            try:
                page.goto(source_url, wait_until="domcontentloaded", timeout=config.request_timeout_seconds * 1000)
                page.wait_for_function("typeof gfn_downloadFile === 'function'", timeout=config.request_timeout_seconds * 1000)
                for document, attachment in items:
                    try:
                        with page.expect_download(timeout=60_000) as info:
                            page.evaluate(
                                "([a,b]) => gfn_downloadFile(a,b)",
                                [attachment["token1"], attachment["token2"]],
                            )
                        download = info.value
                        filename = f"{attachment['attachment_id']}_{safe_name(download.suggested_filename)}"
                        output = (
                            paths.raw_assets / safe_name(document["business_function"]) /
                            document["doc_id"] / "attachments" / filename
                        )
                        ensure_parent(output)
                        download.save_as(str(output))
                        inspection = inspect_downloaded_attachment(output, attachment.get("file_format", ""))
                        if not inspection["signature_valid"]:
                            raise ValueError("파일 대신 HTML 오류 페이지 또는 빈 응답을 받음")
                        attachment.update({
                            "download_status": "downloaded",
                            "filename": filename,
                            "local_path": str(output.relative_to(paths.root)),
                            "source_page_url": source_url,
                            "official_download_url": None,
                            "user_action_url": source_url,
                            "delivery_mode": "official_download_page",
                            **inspection,
                        })
                        results.append(attachment)
                    except Exception as error:
                        attachment.update({
                            "download_status": "failed",
                            "processing_policy": "metadata_only",
                            "indexable": False,
                            "needs_review": True,
                            "error": f"{type(error).__name__}: {error}",
                        })
            finally:
                page.close()
        context.close()
        browser.close()
    return results


def ensure_source_page_action(document: dict[str, Any], row: dict[str, str]) -> None:
    if document.get("actions") or not row.get("action_policy", "").startswith("O"):
        return
    expected = [value for value in row.get("expected_action_types", "").split("|") if value]
    action_type = expected[0] if expected else "related_service"
    label_suffix = {
        "self_check": "자가진단", "lookup": "조회", "apply": "신청",
        "progress": "진행상황 조회", "consult": "상담",
    }.get(action_type, "바로가기")
    display_title = document.get("display_title") or resolve_display_title(row, "")
    label = display_title if label_suffix in display_title else f"{display_title} {label_suffix}"
    document["actions"] = [{
        "action_id": sha256_text(f"{row['url_id']}|source-page|{action_type}")[:16],
        "label": label,
        "raw_label": row["page_title"],
        "url": row["source_url"],
        "action_type": action_type,
        "source_tag": "source_page",
        "auth_required": row.get("action_auth", row.get("auth_required", "")),
        "official_domain": allowed_action_domain(row["source_url"]),
        "requires_review": False,
    }]


def sync_attachment_actions(document: dict[str, Any]) -> None:
    actions = [action for action in document.get("actions", []) if action.get("source_tag") != "attachment"]
    if document.get("policy", {}).get("attachment_user_delivery_policy") == "none":
        document["actions"] = actions
        return
    seen = {(action.get("url"), action.get("label")) for action in actions}
    for attachment in document.get("attachments", []):
        direct = bool(attachment.get("official_download_url"))
        url = attachment.get("official_download_url") or attachment.get("source_page_url") or document["source_url"]
        # 안정적인 공식 파일 URL이 있으면 직접 다운로드, JS 토큰 방식이면 공식 게시 페이지를 제공합니다.
        suffix = "공식 파일 다운로드" if direct else "공식 다운로드 페이지"
        label = f"{attachment.get('display_name', '첨부파일')} {suffix}"
        key = (url, label)
        if key in seen:
            continue
        seen.add(key)
        actions.append({
            "action_id": attachment["attachment_id"],
            "label": label,
            "url": url,
            "action_type": "download",
            "source_tag": "attachment",
            "attachment_id": attachment["attachment_id"],
            "download_method": attachment.get("download_method"),
            "delivery_mode": "direct_official_download" if direct else "official_download_page",
            "auth_required": "불필요",
            "official_domain": allowed_action_domain(url),
            "requires_review": attachment.get("needs_review", False),
        })
    document["actions"] = actions


def attachment_to_document(
    parent: dict[str, Any], attachment: dict[str, Any], paths: OutputPaths, config: PipelineConfig
) -> dict[str, Any]:
    local_path = paths.root / attachment["local_path"]
    detected = attachment.get("detected_format") or detect_attachment_format(local_path, attachment.get("file_format", ""))
    text, extraction_status, extraction_error = extract_attachment_text(local_path, detected, config)
    attachment.update({
        "detected_format": detected,
        "extracted_text": text,
        "extracted_text_chars": len(text),
        "text_extraction_status": extraction_status,
        "text_extraction_error": extraction_error,
    })
    attachment.update(classify_attachment_policy(attachment, config))

    title = attachment.get("display_name") or attachment.get("filename") or "첨부파일"
    markdown = f"# {title}\n\n{text}".strip() if text else f"# {title}"
    doc_id = f"{parent['doc_id']}-ATT-{attachment['attachment_id'][:8]}"
    document = {
        "doc_id": doc_id,
        "parent_doc_id": parent["doc_id"],
        "attachment_id": attachment["attachment_id"],
        "record_type": "attachment",
        "indexable": attachment["indexable"],
        "rag_index_mode": "attachment" if attachment["indexable"] else "none",
        "title": title,
        "source_url": attachment.get("source_page_url") or parent["source_url"],
        "official_download_url": attachment.get("official_download_url"),
        "user_action_url": attachment.get("user_action_url") or attachment.get("source_page_url"),
        "site_name": parent.get("site_name", ""),
        "business_function": parent["business_function"],
        "target_business_function": parent.get("target_business_function", parent["business_function"]),
        "page_type": "attachment",
        "file_format": detected,
        "mime_type": attachment.get("mime_type", ""),
        "filename": attachment.get("filename", ""),
        "local_path": attachment.get("local_path", ""),
        "size_bytes": attachment.get("size_bytes", 0),
        "sha256": attachment.get("sha256", ""),
        "processing_policy": attachment["processing_policy"],
        "classification_reason": attachment["classification_reason"],
        "needs_review": attachment["needs_review"],
        "text_extraction_status": extraction_status,
        "content_text": text,
        "content_markdown": markdown,
        "blocks": [{"type": "paragraph", "text": text}] if text else [],
        "actions": [],
        "attachments": [],
        "images": [],
        "videos": [],
        "supplementary_links": [],
        "links": [],
        "quality": {
            "status": "review" if attachment["needs_review"] else "pass",
            "reasons": [attachment["classification_reason"]] if attachment["needs_review"] else [],
            "parsed_text_chars": len(text),
            "faq_count": 0,
            "table_count": 0,
        },
        "parsed_at": now_kst_iso(),
    }
    folder = paths.parsed_attachments / safe_name(parent["business_function"]) / parent["doc_id"]
    md_path = folder / f"{doc_id}.md"
    json_path = folder / f"{doc_id}.json"
    ensure_parent(md_path)
    md_path.write_text(markdown, encoding="utf-8")
    write_json(json_path, document)
    attachment["parsed_markdown_path"] = str(md_path.relative_to(paths.root))
    attachment["parsed_json_path"] = str(json_path.relative_to(paths.root))
    return document


def process_attachment_documents(
    page_documents: list[dict[str, Any]], paths: OutputPaths, config: PipelineConfig
) -> tuple[list[dict[str, Any]], pd.DataFrame]:
    attachment_documents: list[dict[str, Any]] = []
    manifest_rows: list[dict[str, Any]] = []
    for parent in page_documents:
        parent_rag_policy = parent.get("policy", {}).get("attachment_rag_policy", "none")
        for attachment in parent.get("attachments", []):
            if (
                attachment.get("download_status") == "downloaded"
                and config.create_attachment_documents
                and parent_rag_policy == "auto_classify_guide_only"
            ):
                try:
                    attachment_document = attachment_to_document(parent, attachment, paths, config)
                    attachment_documents.append(attachment_document)
                except Exception as error:
                    attachment.update({
                        "processing_policy": "metadata_only", "indexable": False,
                        "needs_review": True,
                        "text_extraction_status": "failed",
                        "text_extraction_error": f"{type(error).__name__}: {error}",
                    })
            else:
                attachment.update(classify_attachment_policy(attachment, config))

            if parent_rag_policy != "auto_classify_guide_only":
                attachment.update({
                    "processing_policy": "download_no_index" if attachment.get("download_status") == "downloaded" else "metadata_only",
                    "indexable": False,
                    "classification_reason": "URL 정책표에서 첨부파일 RAG 인덱싱을 사용하지 않음",
                })
            manifest_rows.append({
                "parent_doc_id": parent["doc_id"],
                "business_function": parent["business_function"],
                "attachment_download_policy": parent.get("policy", {}).get("attachment_download_policy", "X"),
                "attachment_rag_policy": parent.get("policy", {}).get("attachment_rag_policy", "none"),
                "attachment_user_delivery_policy": parent.get("policy", {}).get("attachment_user_delivery_policy", "none"),
                "attachment_id": attachment.get("attachment_id", ""),
                "display_name": attachment.get("display_name", ""),
                "file_format": attachment.get("file_format", ""),
                "detected_format": attachment.get("detected_format", ""),
                "download_method": attachment.get("download_method", ""),
                "download_status": attachment.get("download_status", ""),
                "processing_policy": attachment.get("processing_policy", ""),
                "indexable": attachment.get("indexable", False),
                "needs_review": attachment.get("needs_review", False),
                "classification_reason": attachment.get("classification_reason", ""),
                "text_extraction_status": attachment.get("text_extraction_status", ""),
                "extracted_text_chars": attachment.get("extracted_text_chars", 0),
                "source_page_url": attachment.get("source_page_url", parent["source_url"]),
                "official_download_url": attachment.get("official_download_url"),
                "user_action_url": attachment.get("user_action_url", parent["source_url"]),
                "local_path": attachment.get("local_path", ""),
                "size_bytes": attachment.get("size_bytes", 0),
                "sha256": attachment.get("sha256", ""),
                "error": attachment.get("error") or attachment.get("text_extraction_error", ""),
            })
        sync_attachment_actions(parent)
    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(paths.processed / "attachment_manifest.csv", index=False, encoding="utf-8-sig")
    with (paths.processed / "attachment_manifest.jsonl").open("w", encoding="utf-8") as file:
        for row in manifest_rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")
    if not manifest_df.empty:
        manifest_df[manifest_df["needs_review"].fillna(False)].to_csv(
            paths.quality / "attachment_review.csv", index=False, encoding="utf-8-sig"
        )
    return attachment_documents, manifest_df


def _chunk_plain_len(text: str) -> int:
    return len(re.sub(r"[#|\-\s]", "", text))


def _noise_chunk(text: str) -> bool:
    return any(fragment in text for fragment in NOISE_CONTAINS_TEXTS)


def split_text_lossless(text: str, max_chars: int) -> list[str]:
    text = text.strip()
    if len(text) <= max_chars:
        return [text] if text else []
    paragraphs = [value.strip() for value in re.split(r"\n\s*\n", text) if value.strip()]
    chunks: list[str] = []
    current = ""
    for paragraph in paragraphs:
        if len(paragraph) > max_chars:
            sentences = [value.strip() for value in re.split(r"(?<=[.!?다요])\s+", paragraph) if value.strip()]
        else:
            sentences = [paragraph]
        for sentence in sentences:
            candidate = f"{current}\n\n{sentence}".strip() if current else sentence
            if current and len(candidate) > max_chars:
                chunks.append(current)
                current = sentence
            elif len(sentence) > max_chars and not current:
                for start in range(0, len(sentence), max_chars):
                    chunks.append(sentence[start:start + max_chars])
                current = ""
            else:
                current = candidate
    if current:
        chunks.append(current)
    return chunks


def structure_aware_chunks(document: dict[str, Any], config: PipelineConfig) -> list[dict[str, Any]]:
    """짧은 FAQ와 짧은 본문을 삭제하지 않는 무손실 청킹입니다."""
    if not document.get("indexable", True):
        return []

    intermediate: list[dict[str, str]] = []
    current_heading = ""
    current_parts: list[str] = []

    def add_section(content: str, title: str) -> None:
        content = content.strip()
        if not content or _noise_chunk(content):
            return
        for part in split_text_lossless(content, config.chunk_max_chars):
            intermediate.append({"section_title": title, "chunk_type": "section", "content": part})

    def current_is_heading_only() -> bool:
        content = "\n\n".join(part for part in current_parts if part).strip()
        return bool(current_heading and content == f"### {current_heading}")

    def flush() -> None:
        nonlocal current_parts
        content = "\n\n".join(part for part in current_parts if part).strip()
        current_parts = []
        add_section(content, current_heading)

    for block in document.get("blocks", []):
        kind = block.get("type")
        if kind == "heading":
            flush()
            current_heading = block.get("text", "")
            current_parts = [render_blocks([block])]
            continue
        if kind == "faq":
            if current_is_heading_only():
                current_parts = []
            else:
                flush()
            content = render_blocks([block]).strip()
            if content and not _noise_chunk(content):
                intermediate.append({
                    "section_title": block.get("question", ""),
                    "chunk_type": "faq", "content": content,
                })
            continue
        if kind == "table":
            pending_heading = current_heading
            # 표 앞의 설명·목록은 먼저 보존하되, 제목만 있는 경우에는
            # 제목을 별도 청크로 만들지 않고 표 청크에 붙입니다.
            if current_is_heading_only():
                current_parts = []
            else:
                flush()
            for content in split_table(block, config.chunk_max_chars, pending_heading):
                if content.strip() and not _noise_chunk(content):
                    intermediate.append({
                        "section_title": pending_heading,
                        "chunk_type": "table", "content": content.strip(),
                    })
            continue
        block_text = render_blocks([block])
        candidate = "\n\n".join(current_parts + [block_text]).strip()
        if current_parts and len(candidate) > config.chunk_max_chars:
            flush()
            if current_heading:
                current_parts = [f"### {current_heading}"]
        current_parts.append(block_text)
    flush()

    # 짧은 일반 섹션은 삭제하지 않고 앞 섹션과 결합하거나 그대로 유지합니다.
    merged: list[dict[str, str]] = []
    for item in intermediate:
        if (
            item["chunk_type"] == "section"
            and _chunk_plain_len(item["content"]) < config.min_chunk_chars
            and merged
            and merged[-1]["chunk_type"] == "section"
            and len(merged[-1]["content"] + "\n\n" + item["content"]) <= config.chunk_max_chars
        ):
            merged[-1]["content"] += "\n\n" + item["content"]
        else:
            merged.append(item)

    # 문서 맨 앞의 짧은 제목 청크는 다음 일반 섹션과 결합합니다.
    if (
        len(merged) >= 2
        and merged[0]["chunk_type"] == "section"
        and _chunk_plain_len(merged[0]["content"]) < config.min_chunk_chars
        and merged[1]["chunk_type"] == "section"
        and len(merged[0]["content"] + "\n\n" + merged[1]["content"]) <= config.chunk_max_chars
    ):
        merged[1]["content"] = merged[0]["content"] + "\n\n" + merged[1]["content"]
        merged.pop(0)

    # 인덱싱 문서가 내용이 있는데 청크 0개가 되는 것을 방지합니다.
    if not merged and document.get("content_markdown", "").strip():
        for part in split_text_lossless(document["content_markdown"], config.chunk_max_chars):
            merged.append({"section_title": "", "chunk_type": "section", "content": part})

    records: list[dict[str, Any]] = []
    seen_hashes: set[str] = set()
    action_types = sorted({
        action.get("action_type", "") for action in document.get("actions", [])
        if action.get("action_type")
    })
    for item in merged:
        content = item["content"].strip()
        content_hash = sha256_text(content)
        if content_hash in seen_hashes:
            continue
        seen_hashes.add(content_hash)
        records.append({
            "chunk_id": f"{document['doc_id']}_chunk_{len(records):03d}",
            "parent_doc_id": document.get("parent_doc_id", document["doc_id"]),
            "document_id": document["doc_id"],
            "attachment_id": document.get("attachment_id"),
            "record_type": document.get("record_type", "page"),
            "chunk_index": len(records),
            "title": document["title"],
            "section_title": item["section_title"],
            "chunk_type": item["chunk_type"],
            "business_function": document["business_function"],
            "target_business_function": document["target_business_function"],
            "page_type": document["page_type"],
            "rag_index_mode": document.get("rag_index_mode", "full"),
            "source_url": document["source_url"],
            "official_download_url": document.get("official_download_url"),
            "available_action_types": action_types,
            "content": content,
            "content_hash": content_hash,
        })
    return records


def _normalized_coverage_text(value: str) -> str:
    return re.sub(r"\s+", "", re.sub(r"[#|`*\-]", "", value or ""))


def atomic_index_units(document: dict[str, Any]) -> list[str]:
    units: list[str] = []
    for block in iter_blocks_recursive(document.get("blocks", [])):
        kind = block.get("type")
        if kind == "heading":
            continue
        if kind == "faq":
            value = render_blocks([block])
            if value:
                units.append(value)
        elif kind == "table":
            headers = block.get("headers", [])
            for row in block.get("rows", []):
                units.append(" ".join(headers + row))
        else:
            value = blocks_text([block])
            if value:
                units.append(value)
    if not units and document.get("content_text"):
        units.append(document["content_text"])
    return units


def calculate_chunk_coverage(document: dict[str, Any], chunks: list[dict[str, Any]]) -> float:
    """문서의 고유 단어·숫자가 청크에 보존됐는지 계산합니다."""
    token_pattern = re.compile(r"[가-힣A-Za-z]{2,}|[0-9]+")
    source_tokens = set(token_pattern.findall(document.get("content_text", "").lower()))
    if not source_tokens:
        return 1.0
    chunk_text = "\n".join(chunk.get("content", "") for chunk in chunks).lower()
    chunk_tokens = set(token_pattern.findall(chunk_text))
    return round(len(source_tokens & chunk_tokens) / len(source_tokens), 4)


def run_pipeline(
    review_csv_path: str | Path,
    runtime_root: str | Path,
    config: PipelineConfig | None = None,
) -> dict[str, Any]:
    config = config or PipelineConfig()
    validate_media_policy(config)
    runtime_root = Path(runtime_root)
    paths = create_output_paths(runtime_root)
    session = create_session(config)

    review_df = pd.read_csv(review_csv_path, encoding="utf-8-sig", dtype=str).fillna("")
    manifest_df = normalize_review_manifest(review_df)
    manifest_df.to_csv(paths.manifest / "manifest_full_42.csv", index=False, encoding="utf-8-sig")
    review_df.to_csv(paths.manifest / "review_policy_42.csv", index=False, encoding="utf-8-sig")

    target_df = manifest_df.copy()
    if config.select_business_functions:
        target_df = target_df[target_df["business_function"].isin(config.select_business_functions)]
    if config.run_only_url_ids:
        unknown = sorted(set(config.run_only_url_ids) - set(manifest_df["url_id"]))
        if unknown:
            raise ValueError(f"Manifest에 없는 URL ID: {unknown}")
        target_df = target_df[target_df["url_id"].isin(config.run_only_url_ids)]
    if config.max_urls is not None:
        target_df = target_df.head(config.max_urls)
    target_df.to_csv(paths.manifest / "manifest_run.csv", index=False, encoding="utf-8-sig")

    page_documents: list[dict[str, Any]] = []
    embedded_documents: list[dict[str, Any]] = []
    run_results: list[dict[str, Any]] = []
    crawl_jsonl = paths.logs / "crawl_results.jsonl"

    for _, series in tqdm(target_df.iterrows(), total=len(target_df), desc="KDIC 42개 정책 기반 수집"):
        row = row_to_dict(series)
        started = now_kst_iso()
        try:
            if row["normalized_decision"] == "link_only":
                document = build_link_only_document(row)
                save_document(paths, document, row)
                page_documents.append(document)
                result = {
                    "url_id": row["url_id"], "business_function": row["business_function"],
                    "page_title": row["page_title"], "status": "link_metadata_created",
                    "status_code": None, "content_chars": 0, "faq_count": 0,
                    "table_count": 0, "action_count": 1, "attachment_count": 0,
                    "downloaded_attachment_count": 0,
                    "pagination_detected": False, "pagination_is_complete": True,
                    "pagination_page_count": 0, "error_type": "", "error": "",
                }
            else:
                first = fetch_url(session, row["source_url"], config)
                html_path, meta_path = save_fetch_result(paths, row, first)
                if not first.ok:
                    raise requests.HTTPError(f"HTTP {first.status_code}: {first.final_url}")
                if "html" not in first.content_type.lower():
                    raise ValueError(f"HTML 응답이 아님: {first.content_type}")
                first_document = parse_html_document(first.body, first.final_url, row, first.encoding)
                document, pagination = collect_paginated_document(
                    session, first, first_document, row, paths, config
                )
                child_documents = document.pop("_embedded_documents", [])
                for child_document in child_documents:
                    save_embedded_document(paths, child_document, row)
                embedded_documents.extend(child_documents)
                ensure_source_page_action(document, row)
                synchronize_document_navigation(document)
                document["quality"] = build_quality(document, document.get("content_text", ""), row)
                assets = download_direct_assets(session, document, row, paths, config)
                document["downloaded_assets"] = assets
                md_path, json_path = save_document(paths, document, row)
                page_documents.append(document)
                result = {
                    "url_id": row["url_id"], "business_function": row["business_function"],
                    "page_title": document["title"], "status": (
                        "paginated_full_collection_created" if pagination.get("pagination_detected") and pagination.get("is_complete")
                        else "pagination_collection_incomplete" if pagination.get("pagination_detected")
                        else "parse_success"
                    ),
                    "status_code": first.status_code,
                    "content_chars": len(document.get("content_text", "")),
                    "quality_status": document.get("quality", {}).get("status", ""),
                    "quality_reasons": " | ".join(document.get("quality", {}).get("reasons", [])),
                    "retention_ratio": document.get("quality", {}).get("retention_ratio", 0),
                    "faq_count": document.get("quality", {}).get("faq_count", 0),
                    "table_count": document.get("quality", {}).get("table_count", 0),
                    "action_count": len(document.get("actions", [])),
                    "attachment_count": len(document.get("attachments", [])),
                    "downloaded_attachment_count": len(assets.get("attachments", [])),
                    "image_count": len(document.get("images", [])),
                    "video_count": len(document.get("videos", [])),
                    "supplementary_link_count": len(document.get("supplementary_links", [])),
                    "asset_failure_count": len(assets.get("failures", [])),
                    "pagination_detected": pagination.get("pagination_detected", False),
                    "pagination_is_complete": pagination.get("is_complete", True),
                    "pagination_page_count": pagination.get("fetched_page_count", 1),
                    "pagination_failure_count": len(pagination.get("failures", [])),
                    "raw_html_path": str(html_path.relative_to(paths.root)),
                    "response_meta_path": str(meta_path.relative_to(paths.root)),
                    "parsed_markdown_path": str(md_path.relative_to(paths.root)),
                    "parsed_json_path": str(json_path.relative_to(paths.root)),
                    "error_type": "", "error": "",
                }
        except Exception as error:
            result = {
                "url_id": row["url_id"], "business_function": row["business_function"],
                "page_title": row["page_title"], "status": "failed", "status_code": None,
                "content_chars": 0, "faq_count": 0, "table_count": 0, "action_count": 0,
                "attachment_count": 0, "downloaded_attachment_count": 0,
                "pagination_detected": False, "pagination_is_complete": False,
                "pagination_page_count": 0,
                "error_type": type(error).__name__, "error": str(error),
            }
        result["started_at"] = started
        result["finished_at"] = now_kst_iso()
        run_results.append(result)
        append_jsonl(crawl_jsonl, result)
        if row["normalized_decision"] != "link_only":
            time.sleep(config.request_delay_seconds)

    # JavaScript 기반 공개 첨부파일도 다운로드합니다.
    download_js_attachments(page_documents, paths, config)

    # 모든 첨부파일을 검증·텍스트 추출·정책 분류합니다.
    attachment_documents, attachment_manifest_df = process_attachment_documents(
        page_documents, paths, config
    )
    documents = page_documents + embedded_documents + attachment_documents

    # 영상·웹툰은 다운로드하지 않고 공식 게시 페이지 참고 링크 목록으로 저장합니다.
    supplementary_rows: list[dict[str, Any]] = []
    for parent in page_documents:
        for link in parent.get("supplementary_links", []):
            supplementary_rows.append({
                "parent_doc_id": parent["doc_id"],
                "business_function": parent["business_function"],
                "title": parent["title"],
                **link,
            })
    supplementary_df = pd.DataFrame(supplementary_rows)
    supplementary_df.to_csv(
        paths.processed / "supplementary_links.csv",
        index=False,
        encoding="utf-8-sig",
    )
    with (paths.processed / "supplementary_links.jsonl").open("w", encoding="utf-8") as file:
        for row in supplementary_rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")

    # 첨부파일 상태와 Action이 갱신되었으므로 links·Markdown까지 다시 동기화합니다.
    by_id = {row["url_id"]: row_to_dict(row) for _, row in target_df.iterrows()}
    for document in page_documents:
        synchronize_document_navigation(document)
        save_document(paths, document, by_id[document["doc_id"]])

    results_df = pd.DataFrame(run_results)
    if not results_df.empty:
        downloaded_by_parent = {}
        for parent in page_documents:
            downloaded_by_parent[parent["doc_id"]] = sum(
                1 for item in parent.get("attachments", [])
                if item.get("download_status") == "downloaded"
            )
        results_df["downloaded_attachment_count"] = results_df["url_id"].map(downloaded_by_parent).fillna(0).astype(int)
    results_df.to_csv(paths.logs / "crawl_results.csv", index=False, encoding="utf-8-sig")

    documents_path = paths.processed / "documents.jsonl"
    with documents_path.open("w", encoding="utf-8") as file:
        for document in documents:
            file.write(json.dumps(document, ensure_ascii=False) + "\n")

    chunks: list[dict[str, Any]] = []
    chunks_by_doc: dict[str, list[dict[str, Any]]] = {}
    if config.create_chunks:
        for document in documents:
            doc_chunks = structure_aware_chunks(document, config)
            chunks.extend(doc_chunks)
            chunks_by_doc[document["doc_id"]] = doc_chunks
    chunks_path = paths.processed / "chunks.jsonl"
    with chunks_path.open("w", encoding="utf-8") as file:
        for chunk in chunks:
            file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    quality_rows: list[dict[str, Any]] = []
    validations: list[dict[str, Any]] = []
    for document in documents:
        quality = document.get("quality", {})
        pagination = document.get("pagination_collection", {}) or {}
        doc_chunks = chunks_by_doc.get(document["doc_id"], [])
        faq_count = count_block_type(document.get("blocks", []), "faq")
        faq_chunk_count = sum(1 for chunk in doc_chunks if chunk["chunk_type"] == "faq")
        coverage = calculate_chunk_coverage(document, doc_chunks) if document.get("indexable", True) else 1.0
        chunk_complete = (
            (not document.get("indexable", True))
            or (
                len(doc_chunks) >= 1
                and faq_count == faq_chunk_count
                and coverage >= 0.98
            )
        )
        reasons = list(quality.get("reasons", []))
        if not chunk_complete:
            reasons.append("청킹 데이터 손실 가능성")
        quality_status = "fail" if not chunk_complete else quality.get("status", "pass")
        quality_rows.append({
            "url_id": document["doc_id"],
            "parent_doc_id": document.get("parent_doc_id", document["doc_id"]),
            "record_type": document.get("record_type", "page"),
            "business_function": document["business_function"],
            "title": document["title"],
            "indexable": document.get("indexable", True),
            "rag_index_mode": document.get("rag_index_mode", ""),
            "quality_status": quality_status,
            "quality_reasons": " | ".join(dict.fromkeys(reasons)),
            "content_chars": len(document.get("content_text", "")),
            "chunk_count": len(doc_chunks),
            "chunk_coverage_ratio": coverage,
            "faq_count": faq_count,
            "faq_chunk_count": faq_chunk_count,
            "table_count": count_block_type(document.get("blocks", []), "table"),
            "action_count": len(document.get("actions", [])),
            "attachment_count": len(document.get("attachments", [])),
            "video_count": len(document.get("videos", [])),
            "supplementary_link_count": len(document.get("supplementary_links", [])),
            "pagination_detected": pagination.get("pagination_detected", False),
            "pagination_is_complete": pagination.get("is_complete", True),
            "pagination_page_count": pagination.get("fetched_page_count", 1),
            "processing_policy": document.get("processing_policy", ""),
            "needs_review": document.get("needs_review", False),
        })
        if document.get("indexable", True):
            validations.extend([
                {
                    "validation": f"{document['doc_id']} 인덱싱 문서 청크 존재",
                    "passed": len(doc_chunks) >= 1,
                },
                {
                    "validation": f"{document['doc_id']} FAQ 청크 무손실",
                    "passed": faq_count == faq_chunk_count,
                },
                {
                    "validation": f"{document['doc_id']} 청크 본문 보존율",
                    "passed": coverage >= 0.98,
                    "value": coverage,
                },
            ])

    quality_df = pd.DataFrame(quality_rows)
    quality_df.to_csv(paths.quality / "quality_report.csv", index=False, encoding="utf-8-sig")

    by_id = {document["doc_id"]: document for document in page_documents}
    if "DP-001" in by_id:
        validations.append({"validation": "DP-001 1억원", "passed": "1억원" in by_id["DP-001"].get("content_text", "")})
    if "UN-003" in by_id:
        text = by_id["UN-003"].get("content_text", "")
        validations.append({"validation": "UN-003 미수령금 종류", "passed": all(k in text for k in ["예금보험금", "개산지급금", "파산배당금"])})
    if "BI-004" in by_id:
        pc = by_id["BI-004"].get("pagination_collection", {})
        validations.append({"validation": "BI-004 전체 페이지", "passed": pc.get("is_complete", False)})
    if "MT-001" in by_id:
        validations.append({
            "validation": "MT-001 영상 공식 페이지 참고 링크",
            "passed": any(item.get("link_type") == "video" for item in by_id["MT-001"].get("supplementary_links", []))
            and all(not item.get("download", False) and not item.get("indexable", False) for item in by_id["MT-001"].get("videos", [])),
        })
    if "MT-009" in by_id:
        text = by_id["MT-009"].get("content_text", "")
        validations.append({"validation": "MT-009 구버전 금액 제외", "passed": "5천만원" not in text and "5천만 원" not in text})
        validations.append({
            "validation": "MT-009 웹툰 공식 페이지 참고 링크",
            "passed": any(item.get("link_type") == "webtoon" for item in by_id["MT-009"].get("supplementary_links", []))
            and not any("webtoon" in image.get("url", "").lower() for image in by_id["MT-009"].get("images", []))
            and not any(token in by_id["MT-009"].get("content_text", "") for token in ["예튼이", "예솜이", "예툰이"]),
        })

    # 구조적 회귀 검사: 링크·제목·노이즈·웹툰·Action 라벨
    generic_titles = {value.lower() for value in GENERIC_PAGE_HEADINGS}
    for document in page_documents:
        action_urls = {item.get("url") for item in document.get("actions", []) if item.get("url")}
        link_urls = {item.get("url") for item in document.get("links", []) if item.get("url")}
        export_md = document.get("markdown_export", "")
        display_title = norm(document.get("display_title", ""))
        all_text = document.get("content_text", "") + " " + export_md
        validations.extend([
            {
                "validation": f"{document['doc_id']} Action URL links 동기화",
                "passed": action_urls.issubset(link_urls),
            },
            {
                "validation": f"{document['doc_id']} Action URL Markdown 동기화",
                "passed": all(url in export_md for url in action_urls),
            },
            {
                "validation": f"{document['doc_id']} 맥락 포함 표시 제목",
                "passed": bool(display_title) and display_title.lower() not in generic_titles,
                "value": display_title,
            },
            {
                "validation": f"{document['doc_id']} Adobe Reader 노이즈 제거",
                "passed": not any(pattern.search(all_text) for pattern in NOISE_REGEXES),
            },
            {
                "validation": f"{document['doc_id']} Action 라벨 길이",
                "passed": all(len(norm(action.get('label', ''))) <= 80 for action in document.get('actions', [])),
            },
            {
                "validation": f"{document['doc_id']} 목록 번호 중복 제거",
                "passed": not bool(re.search(r"(?:^|\\n)\\s*\\d+[.]\\s+\\d+[.]\\s+", export_md)),
            },
        ])
    if "MT-009" in by_id:
        mt009_text = by_id["MT-009"].get("content_text", "")
        mt009_chunks = " ".join(chunk.get("content", "") for chunk in chunks_by_doc.get("MT-009", []))
        validations.append({
            "validation": "MT-009 웹툰 본문·청크 제외",
            "passed": not any(token in mt009_text + " " + mt009_chunks for token in ["예튼이", "예솜이", "예툰이"]),
        })

    validation_df = pd.DataFrame(validations)
    validation_df.to_csv(paths.quality / "regression_tests.csv", index=False, encoding="utf-8-sig")

    service_action_count = sum(
        1 for document in page_documents for action in document.get("actions", [])
        if action.get("action_type") != "download"
    )
    download_action_count = sum(
        1 for document in page_documents for action in document.get("actions", [])
        if action.get("action_type") == "download"
    )
    pagination_detected_count = int(quality_df["pagination_detected"].fillna(False).sum()) if not quality_df.empty else 0
    pagination_complete_count = int(
        quality_df.loc[quality_df["pagination_detected"].fillna(False), "pagination_is_complete"].fillna(False).sum()
    ) if not quality_df.empty else 0

    summary = {
        "run_id": paths.root.name,
        "manifest_count": len(manifest_df),
        "target_count": len(target_df),
        "page_document_count": len(page_documents),
        "embedded_document_count": len(embedded_documents),
        "attachment_document_count": len(attachment_documents),
        "document_count": len(documents),
        "chunk_count": len(chunks),
        "status_counts": results_df["status"].value_counts().to_dict() if not results_df.empty else {},
        "quality_counts": quality_df["quality_status"].value_counts().to_dict() if not quality_df.empty else {},
        "service_action_count": service_action_count,
        "download_action_count": download_action_count,
        "attachment_count": len(attachment_manifest_df),
        "downloaded_attachment_count": int((attachment_manifest_df["download_status"] == "downloaded").sum()) if not attachment_manifest_df.empty else 0,
        "indexed_attachment_count": int(attachment_manifest_df["indexable"].fillna(False).sum()) if not attachment_manifest_df.empty else 0,
        "video_reference_count": sum(len(document.get("videos", [])) for document in page_documents),
        "supplementary_link_count": sum(len(document.get("supplementary_links", [])) for document in page_documents),
        "attachment_review_count": int(attachment_manifest_df["needs_review"].fillna(False).sum()) if not attachment_manifest_df.empty else 0,
        "pagination_detected_count": pagination_detected_count,
        "pagination_complete_count": pagination_complete_count,
        "non_paginated_count": len(page_documents) - pagination_detected_count,
        "regression_failed_count": int((~validation_df["passed"].fillna(False)).sum()) if not validation_df.empty else 0,
    }
    write_json(paths.quality / "quality_summary.json", summary)

    archive_path = Path(shutil.make_archive(
        base_name=str(paths.root), format="zip",
        root_dir=paths.root.parent, base_dir=paths.root.name,
    ))
    return {
        "paths": paths,
        "manifest_df": manifest_df,
        "target_df": target_df,
        "results_df": results_df,
        "quality_df": quality_df,
        "validation_df": validation_df,
        "attachment_manifest_df": attachment_manifest_df,
        "supplementary_df": supplementary_df,
        "documents": documents,
        "page_documents": page_documents,
        "embedded_documents": embedded_documents,
        "attachment_documents": attachment_documents,
        "chunks": chunks,
        "summary": summary,
        "archive_path": archive_path,
    }


# 정책 불변식: 영상과 웹툰은 공식 페이지 참고 링크 전용입니다.
def validate_media_policy(config: PipelineConfig) -> None:
    if config.download_videos or config.download_webtoons:
        raise ValueError("영상·웹툰 다운로드는 이 파이프라인 정책에서 허용하지 않습니다.")

# ============================================================
# V4.2: 안전한 의미 그룹화·모달 문서·검색 단위 보강
# ============================================================

_normalize_review_manifest_v41 = normalize_review_manifest
_extract_actions_v41 = extract_actions
_StructureParserV41 = StructureParser


def normalize_review_manifest(review_df: pd.DataFrame) -> pd.DataFrame:
    """V4 정책표를 유지하면서 구조형 페이지의 명시적 선택자를 보강합니다."""
    manifest = _normalize_review_manifest_v41(review_df)
    defaults = {
        "resource_group_selector": "",
        "resource_group_chunk_policy": "page",
        "modal_content_policy": "none",
    }
    for column, default in defaults.items():
        if column not in manifest.columns:
            manifest[column] = default

    # .item 같은 범용 클래스가 아니라 확인된 부모-자식 경계를 명시합니다.
    mask = manifest["url_id"] == "MT-010"
    manifest.loc[mask, "resource_group_selector"] = ".ruleBox > .item"
    manifest.loc[mask, "resource_group_chunk_policy"] = "resource_group"
    manifest.loc[mask, "modal_content_policy"] = "extract_referenced_layers"
    manifest.loc[mask, "parser_profile"] = "resource_card_with_modal_v1"
    return manifest


def resource_group_selector_for(manifest_row: dict[str, str]) -> str:
    selector = norm(manifest_row.get("resource_group_selector", ""))
    if selector:
        return selector
    # 안전한 자동 탐지는 부모가 ruleBox이고 직계 제목·본문/동작을 가진 경우로 제한합니다.
    return ".ruleBox > .item"


def _fn_layer_target(onclick: str) -> str:
    match = re.search(
        r"\bfn_layer\s*\(\s*['\"]([^'\"]+)['\"]",
        onclick or "",
        flags=re.I,
    )
    return norm(match.group(1)) if match else ""


def _safe_fragment_id(value: str) -> str:
    cleaned = re.sub(r"[^0-9A-Za-z가-힣_-]+", "_", value or "").strip("_")
    return cleaned or sha256_text(value or "fragment")[:12]


def _resource_card_descriptors(
    root: Tag,
    base_url: str,
    manifest_row: dict[str, str],
) -> list[dict[str, Any]]:
    selector = resource_group_selector_for(manifest_row)
    descriptors: list[dict[str, Any]] = []
    for index, card in enumerate(root.select(selector), start=1):
        title_node = card.find(["strong", "h2", "h3", "h4", "dt"], recursive=False)
        title = safe_text(title_node) if title_node else ""
        if not title:
            continue
        group_id = f"{manifest_row['url_id']}_resource_{index:03d}"
        descriptor: dict[str, Any] = {
            "resource_group_id": group_id,
            "title": title,
            "index": index,
            "external_url": "",
            "target_layer_id": "",
            "target_document_id": "",
        }
        action_node = card.find(["a", "button"], recursive=False)
        if action_node:
            onclick = action_node.get("onclick", "")
            target_layer_id = _fn_layer_target(onclick)
            if target_layer_id:
                descriptor["target_layer_id"] = target_layer_id
                descriptor["target_document_id"] = (
                    f"{manifest_row['url_id']}_{_safe_fragment_id(target_layer_id)}"
                )
            else:
                href = action_node.get("href", "")
                descriptor["external_url"] = absolute_url(base_url, href) or ""
        descriptors.append(descriptor)
    return descriptors


def merge_links_with_actions(
    content_links: list[dict[str, Any]],
    actions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """동일 URL의 일반 링크·Action 중복을 제거하되 서로 다른 모달은 보존합니다."""
    action_external_urls = {
        item.get("url")
        for item in actions
        if item.get("url") and not item.get("target_layer_id")
    }

    merged: list[dict[str, Any]] = []
    seen_content: set[tuple[str, str]] = set()
    for item in content_links:
        url = item.get("url")
        text = item.get("anchor_text")
        if not url or not text:
            continue
        # 같은 URL이 Action으로 승격되면 의미 없는 '바로가기' content link는 제거합니다.
        if url in action_external_urls:
            continue
        key = (url, text)
        if key in seen_content:
            continue
        seen_content.add(key)
        merged.append(item)

    seen_actions: set[tuple[str, str, str]] = set()
    for action in actions:
        url = action.get("url")
        label = action.get("label")
        target_layer_id = action.get("target_layer_id", "")
        if not url or not label:
            continue
        key = (url, target_layer_id, label)
        if key in seen_actions:
            continue
        seen_actions.add(key)
        merged.append({
            "url": url,
            "anchor_text": label,
            "link_role": "action",
            "action_type": action.get("action_type", "related_service"),
            "source_tag": action.get("source_tag", ""),
            "action_id": action.get("action_id", ""),
            "resource_group_id": action.get("resource_group_id", ""),
            "interaction_type": action.get("interaction_type", "navigate"),
            "target_layer_id": target_layer_id,
            "target_document_id": action.get("target_document_id", ""),
        })
    return merged


def extract_actions(
    root: Tag,
    base_url: str,
    manifest_row: dict[str, str],
    attachments: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    actions = _extract_actions_v41(root, base_url, manifest_row, attachments)
    descriptors = _resource_card_descriptors(root, base_url, manifest_row)

    # 외부 URL Action에 그룹 참조를 붙입니다.
    for descriptor in descriptors:
        external_url = descriptor.get("external_url")
        if not external_url:
            continue
        for action in actions:
            if action.get("url") == external_url:
                action["resource_group_id"] = descriptor["resource_group_id"]
                action["interaction_type"] = "navigate"

    # fn_layer는 외부 URL이 아니라 같은 공식 페이지 안의 규정 전문을 여는 동작입니다.
    for descriptor in descriptors:
        target_layer_id = descriptor.get("target_layer_id")
        if not target_layer_id:
            continue
        title = descriptor["title"]
        label = f"{title} 내용 보기"
        action_id = sha256_text(
            f"{manifest_row['url_id']}|{target_layer_id}|{label}"
        )[:16]
        actions.append({
            "action_id": action_id,
            "label": label,
            "raw_label": "바로가기",
            "url": manifest_row["source_url"],
            "action_type": "legal_reference",
            "source_tag": "a",
            "javascript_function": "fn_layer",
            "interaction_type": "open_modal",
            "delivery_mode": "official_page_modal",
            "target_layer_id": target_layer_id,
            "target_document_id": descriptor["target_document_id"],
            "resource_group_id": descriptor["resource_group_id"],
            "auth_required": "불필요",
            "official_domain": True,
            "requires_review": False,
        })

    unique: list[dict[str, Any]] = []
    seen: set[tuple[str, str, str]] = set()
    for action in actions:
        key = (
            action.get("url", ""),
            action.get("target_layer_id", ""),
            action.get("label", ""),
        )
        if key in seen:
            continue
        seen.add(key)
        unique.append(action)
    return unique


class StructureParser(_StructureParserV41):
    """명시적으로 확인된 카드 경계에서만 resource_group을 만듭니다."""

    def __init__(
        self,
        page_type: str = "static_page",
        document_id: str = "",
        resource_group_selector: str = "",
    ):
        super().__init__(page_type)
        self.document_id = document_id
        self.resource_group_selector = resource_group_selector
        self._resource_node_ids: set[int] = set()
        self._resource_index = 0

    def parse(self, root: Tag) -> list[dict[str, Any]]:
        selector = self.resource_group_selector
        if selector:
            self._resource_node_ids = {id(node) for node in root.select(selector)}
        else:
            self._resource_node_ids = set()
        return super().parse(root)

    def parse_resource_card(self, node: Tag) -> bool:
        title_node = node.find(["strong", "h2", "h3", "h4", "dt"], recursive=False)
        if not title_node:
            return False
        title = safe_text(title_node)
        if not title or is_noise_text(title):
            return False

        self._resource_index += 1
        group_id = (
            f"{self.document_id}_resource_{self._resource_index:03d}"
            if self.document_id
            else f"resource_{self._resource_index:03d}"
        )

        fragment = BeautifulSoup(str(node), "lxml")
        copied = fragment.body.find(node.name) if fragment.body else None
        if not copied:
            return False
        copied_title = copied.find(["strong", "h2", "h3", "h4", "dt"], recursive=False)
        if copied_title:
            copied_title.decompose()

        target_layer_id = ""
        external_url = ""
        action_node = node.find(["a", "button"], recursive=False)
        if action_node:
            target_layer_id = _fn_layer_target(action_node.get("onclick", ""))
            if not target_layer_id:
                external_url = action_node.get("href", "")

        for action_node_copy in copied.find_all(["a", "button", "input"]):
            action_node_copy.decompose()

        child_parser = StructureParser("static_page")
        for child in copied.find_all(recursive=False):
            child_parser.process_node(child)

        block: dict[str, Any] = {
            "type": "resource_group",
            "resource_group_id": group_id,
            "title": title,
            "content_blocks": child_parser.blocks,
            "content_text": blocks_text(child_parser.blocks),
            "interaction_type": "open_modal" if target_layer_id else "navigate",
        }
        if target_layer_id:
            block["target_layer_id"] = target_layer_id
            block["child_document_id"] = (
                f"{self.document_id}_{_safe_fragment_id(target_layer_id)}"
            )
        if external_url:
            block["external_url_raw"] = external_url
        self.add(block)
        return True

    def process_node(self, node: Any) -> None:
        if isinstance(node, Tag) and id(node) in self._resource_node_ids:
            if self.parse_resource_card(node):
                return

        # 기존 V4.1의 범용 '.item' 자동 그룹화를 차단합니다.
        if isinstance(node, Tag) and "item" in set(node.get("class", [])):
            original_classes = list(node.get("class", []))
            node["class"] = [value for value in original_classes if value != "item"]
            try:
                return super().process_node(node)
            finally:
                node["class"] = original_classes
        return super().process_node(node)


def _modal_content_root(layer: Tag) -> Tag:
    return (
        layer.select_one(".ruleArea")
        or layer.select_one(".cont")
        or layer
    )


def extract_modal_child_documents(
    soup: BeautifulSoup,
    final_url: str,
    manifest_row: dict[str, str],
    actions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """실제로 fn_layer가 참조한 레이어만 별도 검색 문서로 만듭니다."""
    documents: list[dict[str, Any]] = []
    seen_targets: set[str] = set()
    for action in actions:
        target_layer_id = norm(action.get("target_layer_id", ""))
        if not target_layer_id or target_layer_id in seen_targets:
            continue
        seen_targets.add(target_layer_id)
        layer = soup.find(id=target_layer_id)
        if not isinstance(layer, Tag):
            action["requires_review"] = True
            action["modal_resolution_status"] = "target_not_found"
            continue

        title_node = layer.select_one(":scope > .inner > .tit") or layer.select_one(".tit")
        title = safe_text(title_node) if title_node else ""
        if not title:
            title = re.sub(r"\s*내용\s*보기$", "", action.get("label", ""))

        content_node = _modal_content_root(layer)
        fragment = BeautifulSoup(str(content_node), "lxml")
        cloned = fragment.body.find() if fragment.body else fragment.find()
        if not cloned:
            action["requires_review"] = True
            action["modal_resolution_status"] = "content_not_found"
            continue

        for selector in [
            ".btnBottomArea", ".popClose", ".close", "script", "style", "noscript",
        ]:
            for node in cloned.select(selector):
                node.decompose()
        for node in cloned.find_all(["button", "input"]):
            node.decompose()
        for anchor in cloned.find_all("a"):
            if safe_text(anchor) in {"닫기", "바로가기"}:
                anchor.decompose()

        parser = StructureParser(
            "modal_document",
            document_id=action["target_document_id"],
            resource_group_selector="",
        )
        blocks = parser.parse(cloned)
        # 모달 제목이 본문 첫 문단에 중복되면 한 번만 유지합니다.
        while blocks and blocks[0].get("type") in {"paragraph", "heading"}:
            first_text = blocks[0].get("text", "")
            if norm(first_text) == norm(title):
                blocks.pop(0)
            else:
                break

        child = {
            "doc_id": action["target_document_id"],
            "parent_doc_id": manifest_row["url_id"],
            "resource_group_id": action.get("resource_group_id", ""),
            "record_type": "modal_document",
            "indexable": True,
            "rag_index_mode": "full",
            "title": title,
            "manifest_title": manifest_row["page_title"],
            "page_heading": title,
            "display_title": title,
            "html_title": "",
            "source_url": manifest_row["source_url"],
            "final_url": final_url,
            "source_fragment": f"#{target_layer_id}",
            "target_layer_id": target_layer_id,
            "site_name": manifest_row["site_name"],
            "business_function": manifest_row["business_function"],
            "target_business_function": manifest_row.get(
                "target_business_function", manifest_row["business_function"]
            ),
            "page_type": "modal_document",
            "parser_profile": "referenced_modal_document_v1",
            "breadcrumb": [
                manifest_row["business_function"],
                manifest_row["page_title"],
                title,
            ],
            "blocks": blocks,
            "links": [],
            "actions": [],
            "attachments": [],
            "images": [],
            "videos": [],
            "supplementary_links": [],
            "policy": {
                "normalized_decision": "embedded_public_document",
                "content_scope": "fn_layer로 참조된 공개 규정 전문",
                "rag_index_label": "O",
                "action_policy": "source_page_modal",
                "expected_action_types": "legal_reference",
                "pagination_policy": "none",
                "video_policy": "none",
                "webtoon_policy": "none",
                "image_policy": "none",
            },
            "parsed_at": now_kst_iso(),
        }
        refresh_document(child)
        child["quality"] = {
            "status": "pass" if child.get("content_text") else "fail",
            "reasons": [] if child.get("content_text") else ["모달 본문 없음"],
            "parsed_text_chars": len(child.get("content_text", "")),
            "faq_count": count_block_type(blocks, "faq"),
            "table_count": count_block_type(blocks, "table"),
            "action_count": 0,
        }
        action["modal_resolution_status"] = "resolved"
        documents.append(child)
    return documents


def parse_html_document(
    html_bytes: bytes,
    final_url: str,
    manifest_row: dict[str, str],
    encoding: str,
) -> dict[str, Any]:
    soup = BeautifulSoup(html_bytes.decode(encoding, errors="replace"), "lxml")
    original_root = select_content_root(soup)
    page_heading = extract_page_heading(soup, manifest_row["page_title"])
    display_title = resolve_display_title(manifest_row, page_heading)
    root = clone_clean_content_root(original_root)

    attachments = extract_attachments(root, final_url, manifest_row["url_id"])
    actions = extract_actions(root, final_url, manifest_row, attachments)
    if not actions and manifest_row.get("page_type") == "dynamic_lookup":
        actions = [{
            "action_id": sha256_text(
                f"{manifest_row['url_id']}|{manifest_row['source_url']}|lookup"
            )[:16],
            "label": f"{manifest_row['page_title']} 조회 바로가기",
            "raw_label": "",
            "url": manifest_row["source_url"],
            "action_type": "lookup",
            "source_tag": "source_page",
            "interaction_type": "navigate",
            "auth_required": manifest_row.get("action_auth", "불필요"),
            "official_domain": True,
            "requires_review": False,
        }]

    embedded_documents = extract_modal_child_documents(
        soup, final_url, manifest_row, actions
    )

    videos = extract_videos(root, final_url, manifest_row)
    supplementary_links = extract_supplementary_links(
        root, final_url, manifest_row, videos
    )
    poster_urls = {
        item.get("poster_url") for item in videos if item.get("poster_url")
    }
    images = extract_images(root, final_url, manifest_row, poster_urls)
    content_links = extract_links(root, final_url)
    links = merge_links_with_actions(content_links, actions)

    source_fragment = BeautifulSoup(str(root), "lxml")
    source_root = source_fragment.body.find() if source_fragment.body else source_fragment.find()
    if source_root:
        for selector in NOISE_SELECTORS:
            for node in source_root.select(selector):
                node.decompose()
        source_text = norm(source_root.get_text(" ", strip=True))
    else:
        source_text = ""

    for media_node in root.find_all(["video", "source", "iframe", "embed", "object"]):
        media_node.decompose()
    removed_webtoon = remove_reference_only_webtoon(root, manifest_row)

    parser = StructureParser(
        manifest_row.get("page_type", "static_page"),
        document_id=manifest_row["url_id"],
        resource_group_selector=resource_group_selector_for(manifest_row),
    )
    blocks = parser.parse(root)
    document = {
        "doc_id": manifest_row["url_id"],
        "parent_doc_id": manifest_row["url_id"],
        "record_type": "page",
        "indexable": manifest_row.get("rag_index_mode") != "none",
        "rag_index_mode": manifest_row.get("rag_index_mode", "full"),
        "title": display_title,
        "manifest_title": manifest_row["page_title"],
        "page_heading": page_heading,
        "display_title": display_title,
        "html_title": norm(soup.title.get_text(" ", strip=True)) if soup.title else "",
        "source_url": manifest_row["source_url"],
        "final_url": final_url,
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function", manifest_row["business_function"]
        ),
        "page_type": manifest_row["page_type"],
        "parser_profile": manifest_row["parser_profile"],
        "resource_group_selector": manifest_row.get("resource_group_selector", ""),
        "resource_group_chunk_policy": manifest_row.get(
            "resource_group_chunk_policy", "page"
        ),
        "breadcrumb": [manifest_row["business_function"], manifest_row["page_title"]],
        "blocks": blocks,
        "links": links,
        "actions": actions,
        "attachments": attachments,
        "images": images,
        "videos": videos,
        "supplementary_links": supplementary_links,
        "embedded_document_ids": [item["doc_id"] for item in embedded_documents],
        "modal_resources": [{
            "target_layer_id": item["target_layer_id"],
            "target_document_id": item["doc_id"],
            "title": item["title"],
            "resource_group_id": item.get("resource_group_id", ""),
            "status": "resolved",
        } for item in embedded_documents],
        "policy": {
            "normalized_decision": manifest_row["normalized_decision"],
            "content_scope": manifest_row["content_scope"],
            "rag_index_label": manifest_row["rag_index_label"],
            "action_policy": manifest_row["action_policy"],
            "expected_action_types": manifest_row["expected_action_types"],
            "pagination_policy": manifest_row["pagination_policy"],
            "auth_required": manifest_row["auth_required"],
            "attachment_download_policy": manifest_row["attachment_download_policy"],
            "attachment_rag_policy": manifest_row["attachment_rag_policy"],
            "attachment_user_delivery_policy": manifest_row[
                "attachment_user_delivery_policy"
            ],
            "video_policy": manifest_row["video_policy"],
            "webtoon_policy": manifest_row["webtoon_policy"],
            "image_policy": manifest_row["image_policy"],
            "modal_content_policy": manifest_row.get("modal_content_policy", "none"),
        },
        "parsed_at": now_kst_iso(),
        "_embedded_documents": embedded_documents,
    }
    if removed_webtoon:
        document.setdefault("excluded_content", []).extend(removed_webtoon)
    apply_document_filters(document)
    refresh_document(document)
    synchronize_document_navigation(document)
    document["quality"] = build_quality(document, source_text, manifest_row)
    return document


def save_embedded_document(
    paths: OutputPaths,
    document: dict[str, Any],
    row: dict[str, str],
) -> tuple[Path, Path]:
    folder = safe_name(row["business_function"])
    md_path = (
        paths.parsed_markdown / folder / "embedded" / f"{document['doc_id']}.md"
    )
    json_path = (
        paths.parsed_json / folder / "embedded" / f"{document['doc_id']}.json"
    )
    ensure_parent(md_path)
    md_path.write_text(
        document.get("markdown_export") or render_document_markdown(document),
        encoding="utf-8",
    )
    write_json(json_path, document)
    return md_path, json_path


def structure_aware_chunks(
    document: dict[str, Any],
    config: PipelineConfig,
) -> list[dict[str, Any]]:
    """resource_group·법령 조문 경계를 검색 단위로 보존하는 무손실 청킹입니다."""
    if not document.get("indexable", True):
        return []

    intermediate: list[dict[str, Any]] = []
    current_heading = ""
    current_parts: list[str] = []

    def add_unit(
        content: str,
        title: str,
        chunk_type: str,
        resource_group_id: str = "",
    ) -> None:
        content = content.strip()
        if not content or _noise_chunk(content):
            return
        for part in split_text_lossless(content, config.chunk_max_chars):
            intermediate.append({
                "section_title": title,
                "chunk_type": chunk_type,
                "content": part,
                "resource_group_id": resource_group_id,
            })

    def current_is_heading_only() -> bool:
        content = "\n\n".join(
            part for part in current_parts if part
        ).strip()
        return bool(current_heading and content == f"### {current_heading}")

    def flush() -> None:
        nonlocal current_parts
        content = "\n\n".join(
            part for part in current_parts if part
        ).strip()
        current_parts = []
        add_unit(content, current_heading, "section")

    for block in document.get("blocks", []):
        kind = block.get("type")
        if kind == "heading":
            flush()
            current_heading = block.get("text", "")
            current_parts = [render_blocks([block])]
            continue
        if kind == "resource_group":
            flush()
            add_unit(
                render_blocks([block]),
                block.get("title", ""),
                "resource_group",
                block.get("resource_group_id", ""),
            )
            continue
        if kind == "definition":
            flush()
            definition_type = (
                "legal_article"
                if document.get("record_type") == "modal_document"
                else "definition"
            )
            add_unit(
                render_blocks([block]),
                block.get("term", ""),
                definition_type,
                document.get("resource_group_id", ""),
            )
            continue
        if kind == "faq":
            if current_is_heading_only():
                current_parts = []
            else:
                flush()
            add_unit(
                render_blocks([block]),
                block.get("question", ""),
                "faq",
            )
            continue
        if kind == "table":
            pending_heading = current_heading
            if current_is_heading_only():
                current_parts = []
            else:
                flush()
            for content in split_table(
                block, config.chunk_max_chars, pending_heading
            ):
                add_unit(content, pending_heading, "table")
            continue

        block_text = render_blocks([block])
        candidate = "\n\n".join(
            current_parts + [block_text]
        ).strip()
        if current_parts and len(candidate) > config.chunk_max_chars:
            flush()
            if current_heading:
                current_parts = [f"### {current_heading}"]
        current_parts.append(block_text)
    flush()

    merged: list[dict[str, Any]] = []
    for item in intermediate:
        if (
            item["chunk_type"] == "section"
            and _chunk_plain_len(item["content"]) < config.min_chunk_chars
            and merged
            and merged[-1]["chunk_type"] == "section"
            and len(
                merged[-1]["content"] + "\n\n" + item["content"]
            ) <= config.chunk_max_chars
        ):
            merged[-1]["content"] += "\n\n" + item["content"]
        else:
            merged.append(item)

    if not merged and document.get("content_markdown", "").strip():
        for part in split_text_lossless(
            document["content_markdown"], config.chunk_max_chars
        ):
            merged.append({
                "section_title": "",
                "chunk_type": "section",
                "content": part,
                "resource_group_id": "",
            })

    records: list[dict[str, Any]] = []
    seen_hashes: set[str] = set()
    action_types = sorted({
        action.get("action_type", "")
        for action in document.get("actions", [])
        if action.get("action_type")
    })
    for item in merged:
        content = item["content"].strip()
        content_hash = sha256_text(content)
        if content_hash in seen_hashes:
            continue
        seen_hashes.add(content_hash)
        records.append({
            "chunk_id": f"{document['doc_id']}_chunk_{len(records):03d}",
            "parent_doc_id": document.get("parent_doc_id", document["doc_id"]),
            "document_id": document["doc_id"],
            "attachment_id": document.get("attachment_id"),
            "record_type": document.get("record_type", "page"),
            "chunk_index": len(records),
            "title": document["title"],
            "section_title": item["section_title"],
            "chunk_type": item["chunk_type"],
            "resource_group_id": item.get(
                "resource_group_id", document.get("resource_group_id", "")
            ),
            "source_fragment": document.get("source_fragment", ""),
            "business_function": document["business_function"],
            "target_business_function": document["target_business_function"],
            "page_type": document["page_type"],
            "rag_index_mode": document.get("rag_index_mode", "full"),
            "source_url": document["source_url"],
            "official_download_url": document.get("official_download_url"),
            "available_action_types": action_types,
            "content": content,
            "content_hash": content_hash,
        })
    return records

# ============================================================
# V4.2.1: 모달 법령의 장·절 문맥을 다음 조문에 결합
# ============================================================

def structure_aware_chunks(
    document: dict[str, Any],
    config: PipelineConfig,
) -> list[dict[str, Any]]:
    """카드·FAQ·표·법령 조문 경계를 유지하면서 짧은 문맥도 버리지 않습니다."""
    if not document.get("indexable", True):
        return []

    intermediate: list[dict[str, Any]] = []
    current_heading = ""
    current_parts: list[str] = []
    modal_context: list[str] = []
    is_modal = document.get("record_type") == "modal_document"

    def add_unit(
        content: str,
        title: str,
        chunk_type: str,
        resource_group_id: str = "",
    ) -> None:
        content = content.strip()
        if not content or _noise_chunk(content):
            return
        for part in split_text_lossless(content, config.chunk_max_chars):
            intermediate.append({
                "section_title": title,
                "chunk_type": chunk_type,
                "content": part,
                "resource_group_id": resource_group_id,
            })

    def current_is_heading_only() -> bool:
        content = "\n\n".join(
            part for part in current_parts if part
        ).strip()
        return bool(current_heading and content == f"### {current_heading}")

    def flush() -> None:
        nonlocal current_parts
        content = "\n\n".join(
            part for part in current_parts if part
        ).strip()
        current_parts = []
        add_unit(content, current_heading, "section")

    for block in document.get("blocks", []):
        kind = block.get("type")

        # 모달 법령의 제정일·개정일·장·절 제목은 단독 짧은 청크가 아니라
        # 바로 다음 조문의 문맥으로 결합합니다.
        if is_modal and kind in {"paragraph", "heading"}:
            text = render_blocks([block]).strip()
            if text:
                modal_context.append(text)
            continue

        if kind == "heading":
            flush()
            current_heading = block.get("text", "")
            current_parts = [render_blocks([block])]
            continue

        if kind == "resource_group":
            flush()
            add_unit(
                render_blocks([block]),
                block.get("title", ""),
                "resource_group",
                block.get("resource_group_id", ""),
            )
            continue

        if kind == "definition":
            flush()
            rendered = render_blocks([block]).strip()
            if is_modal and modal_context:
                rendered = "\n\n".join(modal_context + [rendered])
                modal_context = []
            add_unit(
                rendered,
                block.get("term", ""),
                "legal_article" if is_modal else "definition",
                document.get("resource_group_id", ""),
            )
            continue

        if kind == "faq":
            if current_is_heading_only():
                current_parts = []
            else:
                flush()
            add_unit(
                render_blocks([block]),
                block.get("question", ""),
                "faq",
            )
            continue

        if kind == "table":
            pending_heading = current_heading
            if current_is_heading_only():
                current_parts = []
            else:
                flush()
            for content in split_table(
                block, config.chunk_max_chars, pending_heading
            ):
                add_unit(content, pending_heading, "table")
            continue

        block_text = render_blocks([block])
        candidate = "\n\n".join(
            current_parts + [block_text]
        ).strip()
        if current_parts and len(candidate) > config.chunk_max_chars:
            flush()
            if current_heading:
                current_parts = [f"### {current_heading}"]
        current_parts.append(block_text)

    flush()
    if modal_context:
        add_unit(
            "\n\n".join(modal_context),
            document.get("title", ""),
            "legal_context",
            document.get("resource_group_id", ""),
        )

    merged: list[dict[str, Any]] = []
    for item in intermediate:
        if (
            item["chunk_type"] == "section"
            and _chunk_plain_len(item["content"]) < config.min_chunk_chars
            and merged
            and merged[-1]["chunk_type"] == "section"
            and len(
                merged[-1]["content"] + "\n\n" + item["content"]
            ) <= config.chunk_max_chars
        ):
            merged[-1]["content"] += "\n\n" + item["content"]
        else:
            merged.append(item)

    if not merged and document.get("content_markdown", "").strip():
        for part in split_text_lossless(
            document["content_markdown"], config.chunk_max_chars
        ):
            merged.append({
                "section_title": "",
                "chunk_type": "section",
                "content": part,
                "resource_group_id": "",
            })

    records: list[dict[str, Any]] = []
    seen_hashes: set[str] = set()
    action_types = sorted({
        action.get("action_type", "")
        for action in document.get("actions", [])
        if action.get("action_type")
    })
    for item in merged:
        content = item["content"].strip()
        content_hash = sha256_text(content)
        if content_hash in seen_hashes:
            continue
        seen_hashes.add(content_hash)
        records.append({
            "chunk_id": f"{document['doc_id']}_chunk_{len(records):03d}",
            "parent_doc_id": document.get("parent_doc_id", document["doc_id"]),
            "document_id": document["doc_id"],
            "attachment_id": document.get("attachment_id"),
            "record_type": document.get("record_type", "page"),
            "chunk_index": len(records),
            "title": document["title"],
            "section_title": item["section_title"],
            "chunk_type": item["chunk_type"],
            "resource_group_id": item.get(
                "resource_group_id", document.get("resource_group_id", "")
            ),
            "source_fragment": document.get("source_fragment", ""),
            "business_function": document["business_function"],
            "target_business_function": document["target_business_function"],
            "page_type": document["page_type"],
            "rag_index_mode": document.get("rag_index_mode", "full"),
            "source_url": document["source_url"],
            "official_download_url": document.get("official_download_url"),
            "available_action_types": action_types,
            "content": content,
            "content_hash": content_hash,
        })
    return records

# V4.2 정책 CSV에 구조 처리 열이 있으면 URL ID 하드코딩보다 우선 적용합니다.
_normalize_review_manifest_v42_default = normalize_review_manifest

def normalize_review_manifest(review_df: pd.DataFrame) -> pd.DataFrame:
    manifest = _normalize_review_manifest_v42_default(review_df)
    lookup = review_df.fillna("").astype(str).set_index("문서_ID")
    column_map = {
        "구조그룹_선택자": "resource_group_selector",
        "구조그룹_청킹정책": "resource_group_chunk_policy",
        "모달콘텐츠_처리정책": "modal_content_policy",
    }
    for source_column, target_column in column_map.items():
        if source_column not in lookup.columns:
            continue
        values = manifest["url_id"].map(lookup[source_column]).fillna("")
        mask = values.str.strip() != ""
        manifest.loc[mask, target_column] = values[mask]
    return manifest


Overwriting kdic_final_pipeline.py


## 3. 42개 URL V4.2 정책 CSV 자동 생성

별도 CSV 파일을 업로드하지 않습니다.

이 셀은 노트북 안에 포함된 정책 CSV를 Colab 런타임에 생성합니다.

V4.2에서는 다음 구조 정책 열도 포함합니다.

```text
구조그룹_선택자
구조그룹_청킹정책
모달콘텐츠_처리정책
```

In [ ]:
import base64
import gzip
from pathlib import Path

RUNTIME_ROOT = (
    Path("/content")
    if Path("/content").exists()
    else Path.cwd()
)

POLICY_CSV_PATH = (
    RUNTIME_ROOT
    / "KDIC_42_URL_POLICY_V4_2.csv"
)

POLICY_GZIP_B64 = """H4sIAHVtWGoC/+1ca08bWZr+3r+ixIdVkKoxtyTdK22v6GQ7jaYzSZNkNd8sYxfEE2M7VeVcVv3BEBORQG/IDAxOYiNnGhJoObvGGOJoaI3UP8dVpfkL+17OKVfZBQFjJ92zLUWhXJdT55z3Oe/7vJdT//jb363qgpOvq1a5bucK4fGLqr02b5VrYetJztpesYtw6cdt52nBe4LutX6cV53/ztvFmv06G7YLJSe/qjr5nL1eDtsLefv100a1YpdW1cb+kr2+Gbb3CvZf5+U5+hO2drON6k/qxNglOKxDu2Fo3/p+x17cUd2jsFVdsQs5dSxqxlPJsPV61pkty1bFyev305q8AR60XxVVa3HD3njq6SH1Cd5q78yrdnXL2s86S0t28SBsv1iGt/N1/xXsWOATc2X7+ba9vmyXCo3dPXlLftZ+MBu2q3nrVVmee/HOWai0nCseWBXslvW/deqZ96K1W7NfVuyf8s78sl0vhWHq7cUCnGrs1Nou8lxY0F51C+Y06zwswfDzjWpZ/mq8PWjsVFR7vwBvCd+Y+EZt7JWxtbd164d3YTtXch4UYSAtp6tvnLl3skc/blmL5eZLvd39ZEi9ePXTwcEhGPxCo74ALUEfAVAwMQCYQ87yD2eVfhhmxIxHw+nItKbGk9FEJqaFpzKJhP8H4AumSr3ie1Zh0Ch2KWdXa+ofVNVZrVmL7xR77aFdXFKt/QVnNWc/X4FLyVRSa/1vRjMjsYgZCaeSifthOBXToikdunNHU7GtPXuxrHCfFZ5IhbE/oOCoVl/+/NbazeHCoE79/BbE39iFe+sVeyNrF+GIZtUuzdqFA+tlQSEB1gbU4cHhc58Onv90aFSxAaelgsLwU76+fvkbhWWn3jTNtPGvodDdu3cHbsXi0YGUPnBLDxnpUCxtmHpaT5mhq/DftfuGiX+/mTGNkKEltKh5LaonB2IpGMUnwyyh4RNJKPjsCUUV2IjSqBSs+WWYq/Wn1pNnuPZelXskQfFGhGoxr9ivF1B6cGQtZWFZtYi0+zJpl8UIy2Kkg9UCfQZ0KV8Q8gp7zvMlUEL4s1LA6eloFYk2fS3auQ3Q7D+/xQkSUrlyBrv0w5KCKvV5DXBs/bnQr8ZSd5OJVCTmEVLz6IoayZipcDQRMYz41P3wdCYOnUApqbG4DrMSTk1NxaPxSCKc0RPh+FQYRjCZ0MJawtDCRiqjRzUezXGF3cfKoLF7oDirW42/gcDXntrzWRjKaqFRzyoBg1Csas55fIAr1avfFbYkYs3apSxYGljyu3sw2QpbC4XvtItZxars2Gsr2BpgBCEDlkGsds+rnNV8Y7ek2KsL1lxN9MV6vURHT54prrWD+6ztiqo0TczacqO6ZD1ewWcRyqBMinXsjb1YFN1QQLErVn4Z7xIX4BmwG4pr/6zNA4Wt1UBf96DOCPcqoWuZGV2gfZTRPnp6tMOx86eFU6IdlPU7oan9zaKJfZYVyO+VMYHl5KxtWN+vkCCoZ9C+vyPW4hYQHcIcGeCBXgrquj4Nf2NRr8TOssTOnkhifYDrRqVo5w6cXAVk9NXYt/A/cpuFvFV6B084D/ec1W17sQTDOoPALtacx3XF2s9Zm3mUM8y7/e6NKscIKwSxD2uL38iv6+9TpyK3jyNv7AFre5D54h6wzEOliiJdyJ+hJ+iW5ooRS5MnvL8zA9Qc6/aK9agG40T91DLMGioKaxPoYJZkTwRccR4UkI3hug+adrwTJh4UwglRMhVPGl6YRGdCk5OSN3wVuf17PTJjjqUT9wUmzjEmzp2ePyAyfn0iBLdlsaQw4QTWMle2XmZxJAOif8AAFevJvPXqJ1q1s2BCgVHndnpAKEA67ZziPMvn/Gnkw6M7A/NqP6v3q7o2pelaEixwNDWTjuhxI5VUtXskJ1de7k2q8OLYOZGW7PGGckXlBiWFBvXb2FsGTYh2j4mz4rbSfK7bqvccWm1J5mltMRuctUtPBQdk65qzFhfEGgQGj0YeDfqDR4r9vAKOkr1QQO0M6tua+x97fwXNOQ9JUsu1ZbLoZLkH1MuRZHxKM8yf30pb7cISXGow4yH2Dt6DhFg8NPlf6clE0lXik1H9Kv6WnP/KHd0QYPhM/XLc446hdNfycCB4x8lOgxZnnY1EAvoKg0W/fzN/YkeAWwHWwmECXDLe9tD6+vlmroAT9svmm0ODRAe9vNFDJ3nB83nU+XATONLC2QCkABQrjWrlN1oZpPAupo2Z8Ulhh66mDONidCYZ4E99znAfPi3cCZ8wjgW7soWoD7zOl1XoYFQzjBN6wYe9rAl5Ot2vRtIwZLUN3525v/wK5rXOi5wrXQ8fUziGJIXYKxnCpLWLb2iQ5TdyOvlhJG09+++dhSfcVkBNg3MxV4GLjfojcSLr/nKjCD0U2PRUMpxKa8kbekKqkrVlUBfSS4CFulh2npSdXF3CCFYoqZv9FXu/0Avp/ce9dCIZILohFt1otyyN5xRNeUBsQo3dT0ZmQMKJVOpWJu2KFRiKCYrdJ+ZwOjOZiEc90mZqKLStP+TxsgJHMDsVVNRnXd2GQqZL/ap4Y1O8gnQiuSzNyhZa6WdnjDOa0a8ChMEA5K3dPeXqlWvXOQzhGwB1lrQvCVjYgbhBlC2hmdq/XdczGhmlVbYIoODXyz1zKb8ErFyCAaBb+VXSMCVOhtUbvydGwl6izy8UID7qku+c0Fs02SGxAL5Q+GnvjSeOD/heTOscmP5f54GawF96j1XZtqqrrasfrlJnQFV46IzQCd8xaL7TtUTE1GJhQ9PvxIEuY/h/HwhPNzQGx7xQL7XrDhS6P5BFSkIgBXAE0yn8UmEEiA2UCkDZe+C8kF85nryN5oCOL2ViQYplhAEz3AXA+OwcmPWHS6jiMdTjk6lLdqVTpAjlA9J2nq2ICRYKrNMY+BFtSlTZ6wuUM6j+BA6HAhjz3NqvRlNJMxI1A9lvZ+yAWqf0Hxg6xdQSAgb4i70h+y81CtmTn4J3LxcU0Q9EjvMga31fwQhHruKJlJ4uHpGJhCIAkWlEx0VEEWqU61ri24xxGGRGGTIjHwEyHYHC3wnSM/SK/JaF3B8TSfkNZyXHv5RgDeTtqdRDfs3THVLiJ4uVJ2Cfc85fHsEr2U7P1ey1N9ICCj2yGRxC7x4wWJmY0zpqk3FTj102b8YOQcdZRsdo7ywQrGv74ffkPhGvwIjCep7vOrHCOKKtFn0BGsK9uYWodEXy0jOll5Jt8HiT5HUKGXr5EwIZl0yxDoYUSB09JykKheRWH9vFpR4YmK/1tPZ13EAbE4CCc+rl6xwZqWzD6rIfvkR5AsdynuWlqE92CQYDTruIbQEIPHd4bxDRL/BQ1DvxmJY6NhL6ZPqSdFBLrlvhYgMKFPgVcd9x42fNKB9J342PNKMgR8EjA3QmrGu3M5phGmEa2eGTpPiHwn1Hyt+oV9SgYeK4JsYuUZykFXTNgXviIKCOYGLmyh6QgerZrhwWtwBx1SmAxwqL+HFXHOBbM7pB/6FWCsDhecbh8MfAYUfxC18bZ63FLaxw8Ecv2ohLgGsMSiFrPd7sZkjDk0oRqlgYHegL5pZelTHD1pHteY9cDwlnfMayHfkIsvVQhJNTVJd2YAPlwwiHe58iSYe8vZCz33m8nmgEpaC23q/QAuwKdW1tWSxyXsxogWTP1jdBrXhYSaMyC/+6QE7eAxCkJUhJAkDyOYNktEOQBCZbD8OFeAZzbq3pG8KAyNwclW4Tqxdl5/F615cpNgY+wpaUO4x0Khy9qUVvqegGgzP8eAP1t7VSsTAKsbhhvcr1OFl3/crVoUFSCxtPrd09azmPCXfwt9GJ2S0B5N0qIAyWk9aC99EhgEHkX8kDbhoSMjr0lEjQi9bJYuxUOk/0vC8Te9mI6ZfuTN7yJGOHBxk9Z7uJHpg1ZWjwuJlZmmPlnwQw2Cx3HygAgYKzgPB+Hp23nKiTmoz3yfh6Kj00KIU7xMI91wX74bMOfgWBvX27hdrzCypEKeaJN705sd1w2/G24rEXIKRqDe8C9+mHJQX6A/6itBp26Smcp1xzJhlrWg2+q9sWw9sqZeM8ffCyBsGxOWNi/Ynw2BExbJV7XA9FYjHd0EJj9GfMNJOXTVMPqNccZhSc7wEK3GgG23Y3jOFLxrqxjSDMqBHTjERvzmhJswOgeN8DSqGImXic3nql3bX9NaSAR8mboZ62JX0pUwt+c2nDqrxAGzIWS01qyoQWiWm616lmK/L/Ng8ctFJkmActIK8XNwkcHOQZHuFF89kvatF0uFz8TRxSM/ErLJgYHjzJcvFXTIpkmKif/22xHLJY0vr7FsooL5TPP+BC8buyJwhChKfiCVPTtZjrqJwRyTPm6P1qH284UnxxASocdfcR0aqp5mCpKLwZR+zdwFIcvARiwkUl7+/rVjTjyPDaSSJrRgaeouV5ZICNJ4BHKCJr/slhckM4bwmpNWeIpwaJMJYZrG5ZyyXpIBHqqfn22NuVCxNozkWrHHas9Tj0duRSMHUDPadrad0MjtUMn6V1AG7PBzQYtaxV2oIzVukdZVIabwu4GS6hTYOEdU0o1TiA4d6xInPUEnid1IxQZ/ReUdLjK2i/cqb1flCeZVhD8vVuOaeLadaHzoNllBrWD3FTYqdcRymFvTe4cqgjnB9gX5SyBgCXubLi611bQinE4BFbELuVWmrFzoSW+CZyNzUxnUm0QGdAzyS0L1P3QJwDcVObUV2xTeupTFrV7pl6BIynO52xcCJyX9ONT4Y5AzE09AERh/sKnm3zJj81EUnG4snpMPQTlGowwGQpjdC30swSkHClUn4KmlS4Tc+VpsKl7UKuty8LKH5+yz5Yv9r0+7/jQAC7gKpMMVKEWRQ+dSdc+BrU4Dymr7fJ2gOESZm9ft0M+jC6FAwrPq9gcKSJMFy2r8pdRtg1M2NeuBmkmDhBMDT8QQ100xvuMKPt9afZ3oDgn9cwWbNYaFSwcv6HA4zFeveUBYAkkMi2IcZltsfCzMfYi0YVOEoKJisevYWlHndBrafuDsg4x8BNQD2GJQXwgDQ2S3R+47eHLZv/NMyJaPowbsv5l6GRHsfPTh854/wKFQRh4hK5LMqgUKI9aiALaxMTnQs4q0fm2LAKA6sx/OUfwK6nwS4ZXSojFjkW2tTs2UUrtlTwDooBRWyUEeRa3k07J06vO2fItxEuzhGxM06uDI1+CAQEJOBw4dUr7r4JlrC9VqZ6v2erGBCIJ8H4AkOAuQ17IuCJePIWq6TmkTDCf5BbW6k8xePX8OLz6lFv5qGFXfe1qFyvVlX7vFdIz7iIQd5nP/xe1sAXZeFin8q4PePWjyNuO9xjBZwiX7fXn0ILnsrWibFLshxAZOpa6khEHWPbLODuncebtMuIZ4GzOuRjtI7U1X3QxLxV2pQZiE2Ybi7uFXUtxRqWE7BSpK6JrUPU9qyzuu2rbzkGmBm9mNz5NjEVHY8lzfsTRkIWzo5wpmfo7EdLJhN4fart6Byi1BPkGbsfSABZOKsFv5Y7VKcFUIJ26987oig3iTHUoTPVbblhp+juzhS5P1hlhVJQupCKm7q09ysorYzlkAFfFhhSL45xfdNODre44VcnVt2q+/eeWy+jxnaVnedawLcf3oMDb4k/+3Tt7bU4p21IeDBrLdaaGIimkkYmYUoAHJ4uPB0AaApHKIEMtlpWE8nqeVBOSGKr0C/+JA0WylqLZXo9b2f0+KViizcxsVgmDTwwYgovkTTTfhYMQ9eQok+GEpMJM/LHGTP0DRyMwQFGP9zj4O9RDPOQh7uJGnLrZVHgPw4eKixMoZeAit/UkibNRrMmvmn0TuCJitJ7bp5WJBAn8mwoxUf6q801lf33PCpBJur1BdYkeRJmjr8qpLjQ6++0Xh8bEm0VaxRU8UybOxrPFvig4skPgJrx5FRqPHn7ohE1kqh3AtAzIhZMR+g5KWn+3cS/XPApEsExyWekfGKjsowevrvN+JeoXIZIudDWT/S+lt2P0bDmUPDkXJ4UxNwWLy4gMcsUL+U7WPVQMdMeEMvc6QqXYvGQDm5u3NRCuK/4d3oyCsKPADDaxT3KQxjtjrKAyX++DauQrCmdVSOxP0ailCpLgbq8r+ranbh29wi60dKAwjabStPRdaY1hNTDV6z4rI50XPJGtvXVPCgU7R5Q82Qk0QwHsDslFYJ8sl5p1LICfV3wreQgsIzuxTJVUou+wfTlaEu5s7aA1sf9BFuI7xA9aSe5PVYPF3QtNhG9o9/HHwFAOctAOdsVoGDUYa4CCxu/RXYihHAZOj8PJptaaH5+SgSpUH5rC7TlxcdNpQwEdjiUckVtpnEwgxXL6Jqste4qLOgLcQrGyTdLoo4exIufBHk3L5M2NDp5Dxsy4SYBqweTt+y6SVXQJjx6znlhmAlG3HOcXDWSXwZ46SPnGB/nuoIPIeXnS/aDYkf4oCebqoNe5X7uQ6Kk1ab84uHhhmMoDoPRQmAczsqStZ5vvK0LevohEDAxHYCA84yA88dHAPuuIKT6AgfkMQIDHHonBwPDpO+L5ZBvxwxnlwTpTDP/xs9DnoZyWpWCuzUZvyuGEnSrmTxF062ss6Wj6BW39lTxpZm/ixtGRmuloKIorklBP0Jk2/18QcHeL2AxH86IDNoE7PXxxmi8Hxzw5H3/maLd3XP8DVg+mBcL4Tr6Khm7qE2aFxPJaeTiwfHvkc94VX127FUVXEzu8UfEHjYsDT6ajLU9c+wKYY9D5m5fk9vdmuWfPf7IU3NDAb256UVKVy2/gTVX1p93iGITgpql5CI0JL4CRTXFnvw0b7vvVZG4x1WTMPhc/Vp8k7WYtRYfweuBAOHqA3EHnztMucJQd+bt9Ryihr82SXqLL3E5A0VOPW1yk0zwseSesHXiBEl7J12l+6SMX1nFz3OIC+078+GsmxgR6WqBL11Lp3SzmSDpmbPndlPBGCWM4sa4p+CIaz7dXZQ0Y7S11reLtm3/fnc2qrQqlgvJ6MzFaEIPViqjg4ym4WOjKViptEHk9F+Ca91pAI1KCX+8HQUefOLnJitZ1Ijrm4BH8dlkJIxkMrxbSki80lB1eYsByveqnjZRxlKoQyzUkUOESpl8itMCdVmA5YUrCdxTMcen1RftbbboC1Gq0KTzkrS1GCFDA86GYdVYaiYSTza/RMfBWOkuA3Pr875TvisoEq0EzAiVTNCGwj4ic20tHQd2H7UewYzo05oZnswY8STWXcL6dzc9vE/c3mg2rJX8gbO6Lb+MB8oX3YjfGNxxFO34HWMqkriR1BPjienEESp3mFfn6LFVbsDaY/Pn5HNN86ec8W7MFyqFw+aU6xRfGTHcYnWZ5wlISfta4oelyRXv9LgEHvl5MqSgs9v72K+6tlm2yjFzTp8rfo/IP57TheZ9RkCUiAWMUuE6MoniV2UEMlDA0iPnMcZcZJpQdkYWY7Z+LoXcH7bnXULcnVQ0hB9EADCFLkR0TdMntOm4YYrCyQQcIrr+D8S8yiH0YQAA"""

POLICY_CSV_PATH.write_bytes(
    gzip.decompress(
        base64.b64decode(POLICY_GZIP_B64)
    )
)

print("42개 V4.2 정책 CSV 생성:", POLICY_CSV_PATH)
print("파일 크기:", POLICY_CSV_PATH.stat().st_size)

42개 V4.2 정책 CSV 생성: /content/KDIC_42_URL_POLICY_V4_2.csv
파일 크기: 25076


## 4. 실행 설정

수정하지 않고 그대로 실행합니다.

```text
42개 URL 전체 처리
MT-014 link_only
MT-010 카드별 그룹화
MT-010 팝업 규정 전문 별도 문서화
일반 이미지 자동 다운로드
영상·웹툰 파일 다운로드 안 함
JavaScript 첨부파일 Playwright 다운로드 안 함
```

In [ ]:
from kdic_final_pipeline import PipelineConfig, run_pipeline

CONFIG = PipelineConfig(
    select_business_functions=[],
    run_only_url_ids=[],
    max_urls=None,

    request_delay_seconds=1.2,
    request_timeout_seconds=30,
    max_retries=3,
    respect_robots_txt=True,

    enable_generic_pagination=True,
    pagination_max_pages=100,
    pagination_page_size=10,

    download_direct_attachments=True,
    download_js_attachments_with_playwright=False,

    extract_attachment_text=True,
    create_attachment_documents=True,

    collect_supplementary_media_links=True,
    download_videos=False,
    download_webtoons=False,

    download_images=True,

    create_chunks=True,
    chunk_max_chars=1600,
    min_chunk_chars=80,
)

print("현재 실행 설정")
print("- JavaScript 첨부파일:", CONFIG.download_js_attachments_with_playwright)
print("- 일반 이미지:", CONFIG.download_images)
print("- 영상:", CONFIG.download_videos)
print("- 웹툰:", CONFIG.download_webtoons)

현재 실행 설정
- JavaScript 첨부파일: False
- 일반 이미지: True
- 영상: False
- 웹툰: False


## 5. 실행 전 구조·정책 자체 테스트

실제 사이트를 수집하기 전에 작은 HTML 예제로 다음을 검증합니다.

```text
MT-014 link_only
일반 제목 공통 보정
일반 .item 오분류 방지
안전한 카드 선택자
외부 법령 링크 중복 제거
fn_layer 대상 해석
팝업 규정 전문 별도 문서 생성
카드별 resource_group 청크
조문별 legal_article 청크
순서 목록 중복 번호 제거
FAQ 답변 라벨 제거
Adobe Reader 안내 제거
웹툰 본문 제외
```

In [ ]:
import re
import pandas as pd

from kdic_final_pipeline import (
    PipelineConfig,
    build_link_only_document,
    normalize_review_manifest,
    parse_html_document,
    structure_aware_chunks,
)

policy_df = pd.read_csv(
    POLICY_CSV_PATH,
    encoding="utf-8-sig",
    dtype=str,
).fillna("")

manifest_df = normalize_review_manifest(policy_df)

assert len(manifest_df) == 42
assert manifest_df["url_id"].nunique() == 42

# ------------------------------------------------------------
# MT-014 link_only
# ------------------------------------------------------------
mt014_row = (
    manifest_df[
        manifest_df["url_id"] == "MT-014"
    ]
    .iloc[0]
    .to_dict()
)

assert mt014_row["normalized_decision"] == "link_only"
assert mt014_row["rag_index_mode"] == "none"

mt014_doc = build_link_only_document(mt014_row)

assert mt014_doc["record_type"] == "link_only"
assert mt014_doc["indexable"] is False
assert mt014_doc["blocks"] == []
assert len(mt014_doc["actions"]) == 1
assert len(mt014_doc["links"]) == 1

# ------------------------------------------------------------
# 일반적인 제목 공통 보정
# ------------------------------------------------------------
expected_titles = {
    "DP-003": "보호대상 금융회사 개요",
    "DP-004": "보호대상 금융상품 개요",
    "DP-005": "예금자보호제도 FAQ",
    "MT-004": "착오송금 반환지원 FAQ",
    "MT-005": "착오송금 반환지원 주요 FAQ",
    "MT-006": "착오송금 수취인 유의사항",
    "MT-007": "착오송금 수취인 구비서류 안내",
    "MT-008": "착오송금인 구비서류 안내",
    "MT-013": "착오송금인 유의사항",
    "DA-008": "채무정보조회 FAQ",
    "HP-002": "은닉재산 신고 FAQ",
}

def simple_page(heading: str) -> bytes:
    return f"""
    <html>
      <body>
        <div class="pageTit">
          <h2>{heading}</h2>
        </div>
        <div class="contents">
          <p>본문 테스트입니다.</p>
        </div>
      </body>
    </html>
    """.encode("utf-8")

for url_id, expected_title in expected_titles.items():
    row = (
        manifest_df[
            manifest_df["url_id"] == url_id
        ]
        .iloc[0]
        .to_dict()
    )

    if "FAQ" in expected_title:
        heading = "TOP 10"
    elif expected_title.endswith("개요"):
        heading = "개요"
    elif expected_title.endswith("유의사항"):
        heading = "유의사항"
    else:
        heading = "구비서류안내"

    document = parse_html_document(
        simple_page(heading),
        row["source_url"],
        row,
        "utf-8",
    )

    document.pop("_embedded_documents", None)

    assert document["display_title"] == expected_title, (
        url_id,
        document["display_title"],
        expected_title,
    )

# ------------------------------------------------------------
# 일반 .item은 그룹으로 오분류하지 않음
# ------------------------------------------------------------
generic_item_html = """
<html>
  <body>
    <div class="contents">
      <div class="item">
        <strong>일반 안내</strong>
        <p>일반적인 목록 카드입니다.</p>
      </div>
    </div>
  </body>
</html>
""".encode("utf-8")

generic_row = (
    manifest_df[
        manifest_df["url_id"] == "DP-001"
    ]
    .iloc[0]
    .to_dict()
)

generic_doc = parse_html_document(
    generic_item_html,
    generic_row["source_url"],
    generic_row,
    "utf-8",
)
generic_doc.pop("_embedded_documents", None)

assert not any(
    block.get("type") == "resource_group"
    for block in generic_doc["blocks"]
)

# ------------------------------------------------------------
# 실제 구조와 같은 법령 카드 + fn_layer 팝업
# ------------------------------------------------------------
resource_html = """
<html>
  <body>
    <div class="pageTit">
      <h2>관련 법령 및 규정</h2>
    </div>

    <div class="contents">
      <div class="ruleBox">
        <div class="item">
          <strong>예금자보호법</strong>
          <ul>
            <li>[시행 2024. 9. 10.]</li>
            <li>[법률 제20431호, 일부개정]</li>
          </ul>
          <a href="https://www.law.go.kr/test1">
            바로가기
          </a>
        </div>

        <div class="item">
          <strong>예금자보호법 시행령</strong>
          <ul>
            <li>[시행 2025. 1. 31.]</li>
            <li>[대통령령 제35228호, 타법개정]</li>
          </ul>
          <a href="https://www.law.go.kr/test2">
            바로가기
          </a>
        </div>

        <div class="item">
          <strong>착오송금반환지원 규정</strong>
          <p>규정의 주요 내용을 확인할 수 있습니다.</p>
          <a
            href="javascript:void(0);"
            onclick="fn_layer('rule_layer01', this, 750)"
          >
            바로가기
          </a>
        </div>

        <div class="item">
          <strong>착오송금반환지원 시행세칙</strong>
          <p>시행세칙의 주요 내용을 확인할 수 있습니다.</p>
          <a
            href="javascript:void(0);"
            onclick="fn_layer('operation_layer01', this, 750)"
          >
            바로가기
          </a>
        </div>
      </div>
    </div>

    <div
      class="layerPop medium"
      id="rule_layer01"
    >
      <div class="inner">
        <strong class="tit">
          착오송금 반환지원 등에 관한 규정
        </strong>
        <div class="cont">
          <div class="ruleArea">
            <p>제 정 2021. 06. 09</p>
            <p>제1장 총칙</p>
            <dl>
              <dt>제1조(목적)</dt>
              <dd>이 규정의 목적을 정합니다.</dd>
              <dt>제2조(적용 범위)</dt>
              <dd>적용 범위를 정합니다.</dd>
            </dl>
          </div>
        </div>
      </div>
    </div>

    <div
      class="layerPop medium"
      id="operation_layer01"
    >
      <div class="inner">
        <strong class="tit">
          착오송금 반환지원 시행세칙
        </strong>
        <div class="cont">
          <div class="ruleArea">
            <p>제 정 2021. 06. 02</p>
            <dl>
              <dt>제1조(목적)</dt>
              <dd>시행세칙의 목적을 정합니다.</dd>
              <dt>제2조(서류)</dt>
              <dd>
                <ol>
                  <li>
                    <span>1</span>
                    1. 신청서를 제출합니다.
                  </li>
                </ol>
              </dd>
            </dl>
          </div>
        </div>
      </div>
    </div>
  </body>
</html>
""".encode("utf-8")

mt010_row = (
    manifest_df[
        manifest_df["url_id"] == "MT-010"
    ]
    .iloc[0]
    .to_dict()
)

resource_doc = parse_html_document(
    resource_html,
    mt010_row["source_url"],
    mt010_row,
    "utf-8",
)

embedded_docs = resource_doc.pop("_embedded_documents")

resource_groups = [
    block
    for block in resource_doc["blocks"]
    if block.get("type") == "resource_group"
]

assert len(resource_groups) == 4
assert [
    block["title"]
    for block in resource_groups
] == [
    "예금자보호법",
    "예금자보호법 시행령",
    "착오송금반환지원 규정",
    "착오송금반환지원 시행세칙",
]

# 외부 URL은 content/action으로 중복되지 않음
law_links = [
    link
    for link in resource_doc["links"]
    if "law.go.kr" in link.get("url", "")
]

assert len(law_links) == 2
assert all(
    link["link_role"] == "action"
    for link in law_links
)

# 같은 공식 페이지 URL을 사용해도 서로 다른 모달 두 개는 유지
modal_actions = [
    action
    for action in resource_doc["actions"]
    if action.get("interaction_type") == "open_modal"
]

assert len(modal_actions) == 2
assert {
    action["target_layer_id"]
    for action in modal_actions
} == {
    "rule_layer01",
    "operation_layer01",
}

assert all(
    action["modal_resolution_status"] == "resolved"
    for action in modal_actions
)

# 팝업 전문이 별도 검색 문서로 생성
assert len(embedded_docs) == 2
assert {
    document["doc_id"]
    for document in embedded_docs
} == {
    "MT-010_rule_layer01",
    "MT-010_operation_layer01",
}

assert all(
    document["record_type"] == "modal_document"
    for document in embedded_docs
)
assert all(
    document["parent_doc_id"] == "MT-010"
    for document in embedded_docs
)

# 카드별 검색 단위
parent_chunks = structure_aware_chunks(
    resource_doc,
    PipelineConfig(),
)

resource_chunks = [
    chunk
    for chunk in parent_chunks
    if chunk["chunk_type"] == "resource_group"
]

assert len(resource_chunks) == 4
assert len({
    chunk["resource_group_id"]
    for chunk in resource_chunks
}) == 4

# 팝업은 조문 단위 검색
for child_document in embedded_docs:
    child_chunks = structure_aware_chunks(
        child_document,
        PipelineConfig(),
    )
    assert child_chunks
    assert any(
        chunk["chunk_type"] == "legal_article"
        for chunk in child_chunks
    )

# 번호 중복과 단독 바로가기 문구 제거
all_markdown = (
    resource_doc["markdown_export"]
    + "\n"
    + "\n".join(
        document["markdown_export"]
        for document in embedded_docs
    )
)

assert not re.search(
    r"(?m)^\s*\d+[.)]\s+\d+[.)]\s+",
    all_markdown,
)
assert "\n바로가기\n" not in (
    "\n" + resource_doc["content_markdown"] + "\n"
)

# ------------------------------------------------------------
# FAQ 답변 UI 라벨 제거
# ------------------------------------------------------------
faq_html = """
<html>
  <body>
    <div class="contents">
      <div class="accoWrap">
        <div class="accodian">
          <dl>
            <dt>질문 신청 기한은 언제인가요? 열기</dt>
            <dd>
              <p>답변</p>
              <p>신청 기한은 1년 이내입니다.</p>
            </dd>
          </dl>
        </div>
      </div>
    </div>
  </body>
</html>
""".encode("utf-8")

faq_row = (
    manifest_df[
        manifest_df["url_id"] == "DP-005"
    ]
    .iloc[0]
    .to_dict()
)

faq_doc = parse_html_document(
    faq_html,
    faq_row["source_url"],
    faq_row,
    "utf-8",
)
faq_doc.pop("_embedded_documents", None)

faq_chunks = structure_aware_chunks(
    faq_doc,
    PipelineConfig(),
)

assert sum(
    chunk["chunk_type"] == "faq"
    for chunk in faq_chunks
) == 1
assert "\n답변\n" not in (
    "\n" + faq_doc["content_markdown"] + "\n"
)

# ------------------------------------------------------------
# Adobe Reader 문장 제거
# ------------------------------------------------------------
noise_html = """
<html>
  <body>
    <div class="contents">
      <p>정상 본문입니다.</p>
      <p>
        PDF파일 내용이 보이지 않으시면
        Adobe Reader를 설치하시기 바랍니다.
      </p>
    </div>
  </body>
</html>
""".encode("utf-8")

noise_row = (
    manifest_df[
        manifest_df["url_id"] == "MT-007"
    ]
    .iloc[0]
    .to_dict()
)

noise_doc = parse_html_document(
    noise_html,
    noise_row["source_url"],
    noise_row,
    "utf-8",
)
noise_doc.pop("_embedded_documents", None)

assert "Adobe Reader" not in noise_doc["content_text"]
assert "PDF파일 내용이 보이지" not in noise_doc["content_text"]

# ------------------------------------------------------------
# 웹툰 본문 제외
# ------------------------------------------------------------
webtoon_html = """
<html>
  <body>
    <div class="contents">
      <p>현재 반환지원 절차 설명입니다.</p>
      <div class="sliderWrap">
        <img
          src="/assets/webtoon/webtoon01.jpg"
          alt="안내 웹툰"
        >
        <p>예튼이 : 잘못 송금했어.</p>
        <p>예솜이 : 반환지원을 이용해.</p>
      </div>
    </div>
  </body>
</html>
""".encode("utf-8")

webtoon_row = (
    manifest_df[
        manifest_df["url_id"] == "MT-009"
    ]
    .iloc[0]
    .to_dict()
)

webtoon_doc = parse_html_document(
    webtoon_html,
    webtoon_row["source_url"],
    webtoon_row,
    "utf-8",
)
webtoon_doc.pop("_embedded_documents", None)

webtoon_chunks = structure_aware_chunks(
    webtoon_doc,
    PipelineConfig(),
)

rag_text = (
    webtoon_doc["content_text"]
    + "\n"
    + "\n".join(
        chunk["content"]
        for chunk in webtoon_chunks
    )
)

assert "현재 반환지원 절차 설명" in rag_text
assert "예튼이" not in rag_text
assert "예솜이" not in rag_text

print("Manifest 42개: 통과")
print("MT-014 link_only: 통과")
print("일반 제목 공통 보정: 통과")
print("일반 .item 오분류 방지: 통과")
print("법령 카드 4개 그룹화: 통과")
print("외부 링크 중복 제거: 통과")
print("fn_layer 모달 Action 2개: 통과")
print("팝업 규정 전문 별도 문서화: 통과")
print("카드별 resource_group 청크: 통과")
print("조문별 legal_article 청크: 통과")
print("순서 목록 중복 번호 제거: 통과")
print("FAQ 답변 라벨 제거: 통과")
print("Adobe Reader 문장 제거: 통과")
print("웹툰 본문·청크 제외: 통과")

display(
    manifest_df[
        [
            "url_id",
            "page_title",
            "normalized_decision",
            "rag_index_mode",
            "resource_group_selector",
            "resource_group_chunk_policy",
            "modal_content_policy",
        ]
    ]
)

Manifest 42개: 통과
MT-014 link_only: 통과
일반 제목 공통 보정: 통과
일반 .item 오분류 방지: 통과
법령 카드 4개 그룹화: 통과
외부 링크 중복 제거: 통과
fn_layer 모달 Action 2개: 통과
팝업 규정 전문 별도 문서화: 통과
카드별 resource_group 청크: 통과
조문별 legal_article 청크: 통과
순서 목록 중복 번호 제거: 통과
FAQ 답변 라벨 제거: 통과
Adobe Reader 문장 제거: 통과
웹툰 본문·청크 제외: 통과


,url_id,page_title,normalized_decision,rag_index_mode,resource_group_selector,resource_group_chunk_policy,modal_content_policy
0,DP-001,보호한도,include_full,full,,page,none
1,DP-002,예금자보호제도,include_full,full,,page,none
2,DP-003,보호대상 > 금융회사 > 개요,include_full,full,,page,none
3,DP-004,보호대상 > 금융상품 > 개요,include_full,full,,page,none
4,DP-005,"고객센터 > FAQ > 미수령금통합신청(사이트 분류상 명칭, 실제 내용은 예금보호)",include_full,full,,page,none
5,DP-006,예금자보호제도 FAQ,include_full,full,,page,none
6,DP-007,예금자보호한도(해외),include_reference,reference,,page,none
7,BI-001,예금보험금 안내 > 신청시 구비서류,include_full,full,,page,none
8,BI-002,예금보험금 신청 절차 > 예금보험금 신청절차,include_full,full,,page,none
9,BI-003,예금보험금이란?,include_full,full,,page,none


## 6. 42개 전체 파이프라인 실행

다음 셀을 실행하면 실제 사이트의 42개 정책을 처리합니다.

MT-010에서는 부모 페이지 외에 다음 내장 문서도 생성될 수 있습니다.

```text
MT-010_rule_layer01
MT-010_operation_layer01
```

In [ ]:
RESULT = run_pipeline(
    review_csv_path=POLICY_CSV_PATH,
    runtime_root=RUNTIME_ROOT,
    config=CONFIG,
)

print(RESULT['summary'])
display(RESULT['results_df'])
display(RESULT['quality_df'])
display(RESULT['validation_df'])

KDIC 42개 정책 기반 수집:   0%|          | 0/42 [00:00<?, ?it/s]

{'run_id': 'kdic_crawl_output_20260716_172301', 'manifest_count': 42, 'target_count': 42, 'page_document_count': 42, 'embedded_document_count': 2, 'attachment_document_count': 0, 'document_count': 44, 'chunk_count': 302, 'status_counts': {'parse_success': 36, 'paginated_full_collection_created': 4, 'link_metadata_created': 2}, 'quality_counts': {'pass': 44}, 'service_action_count': 29, 'download_action_count': 43, 'attachment_count': 43, 'downloaded_attachment_count': 0, 'indexed_attachment_count': 0, 'video_reference_count': 1, 'supplementary_link_count': 2, 'attachment_review_count': 0, 'pagination_detected_count': 4, 'pagination_complete_count': 4, 'non_paginated_count': 38, 'regression_failed_count': 0}


,url_id,business_function,page_title,status,status_code,content_chars,quality_status,quality_reasons,retention_ratio,faq_count,...,pagination_page_count,pagination_failure_count,raw_html_path,response_meta_path,parsed_markdown_path,parsed_json_path,error_type,error,started_at,finished_at
0,DP-001,예금자보호제도,보호한도,parse_success,200.0,1110,pass,,1.0,0,...,1,0.0,raw/html/예금자보호제도/DP-001.html,raw/response_meta/예금자보호제도/DP-001.json,parsed/markdown/예금자보호제도/DP-001.md,parsed/json/예금자보호제도/DP-001.json,,,2026-07-16T17:23:01+09:00,2026-07-16T17:23:07+09:00
1,DP-002,예금자보호제도,예금자보호제도,parse_success,200.0,740,pass,,1.0,0,...,1,0.0,raw/html/예금자보호제도/DP-002.html,raw/response_meta/예금자보호제도/DP-002.json,parsed/markdown/예금자보호제도/DP-002.md,parsed/json/예금자보호제도/DP-002.json,,,2026-07-16T17:23:08+09:00,2026-07-16T17:23:15+09:00
2,DP-003,예금자보호제도,보호대상 금융회사 개요,parse_success,200.0,1073,pass,,1.0,0,...,1,0.0,raw/html/예금자보호제도/DP-003.html,raw/response_meta/예금자보호제도/DP-003.json,parsed/markdown/예금자보호제도/DP-003.md,parsed/json/예금자보호제도/DP-003.json,,,2026-07-16T17:23:16+09:00,2026-07-16T17:23:21+09:00
3,DP-004,예금자보호제도,보호대상 금융상품 개요,parse_success,200.0,2532,pass,,1.0,0,...,1,0.0,raw/html/예금자보호제도/DP-004.html,raw/response_meta/예금자보호제도/DP-004.json,parsed/markdown/예금자보호제도/DP-004.md,parsed/json/예금자보호제도/DP-004.json,,,2026-07-16T17:23:22+09:00,2026-07-16T17:23:27+09:00
4,DP-005,예금자보호제도,예금자보호제도 FAQ,paginated_full_collection_created,200.0,12519,pass,,1.0,38,...,4,0.0,raw/html/예금자보호제도/DP-005.html,raw/response_meta/예금자보호제도/DP-005.json,parsed/markdown/예금자보호제도/DP-005.md,parsed/json/예금자보호제도/DP-005.json,,,2026-07-16T17:23:29+09:00,2026-07-16T17:23:42+09:00
5,DP-006,예금자보호제도,예금자보호제도 FAQ,parse_success,200.0,3932,pass,,1.0,17,...,1,0.0,raw/html/예금자보호제도/DP-006.html,raw/response_meta/예금자보호제도/DP-006.json,parsed/markdown/예금자보호제도/DP-006.md,parsed/json/예금자보호제도/DP-006.json,,,2026-07-16T17:23:43+09:00,2026-07-16T17:23:48+09:00
6,DP-007,예금자보호제도,예금자보호한도(해외),parse_success,200.0,313,pass,,1.0,0,...,1,0.0,raw/html/예금자보호제도/DP-007.html,raw/response_meta/예금자보호제도/DP-007.json,parsed/markdown/예금자보호제도/DP-007.md,parsed/json/예금자보호제도/DP-007.json,,,2026-07-16T17:23:49+09:00,2026-07-16T17:23:54+09:00
7,BI-001,예금보험금 안내,신청시 구비서류,parse_success,200.0,2334,pass,,1.0,0,...,1,0.0,raw/html/예금보험금 안내/BI-001.html,raw/response_meta/예금보험금 안내/BI-001.json,parsed/markdown/예금보험금 안내/BI-001.md,parsed/json/예금보험금 안내/BI-001.json,,,2026-07-16T17:23:56+09:00,2026-07-16T17:24:00+09:00
8,BI-002,예금보험금 안내,예금보험금 신청절차,parse_success,200.0,1185,pass,,1.0,0,...,1,0.0,raw/html/예금보험금 안내/BI-002.html,raw/response_meta/예금보험금 안내/BI-002.json,parsed/markdown/예금보험금 안내/BI-002.md,parsed/json/예금보험금 안내/BI-002.json,,,2026-07-16T17:24:01+09:00,2026-07-16T17:24:06+09:00
9,BI-003,예금보험금 안내,예금보험금이란?,parse_success,200.0,2354,pass,,1.0,0,...,1,0.0,raw/html/예금보험금 안내/BI-003.html,raw/response_meta/예금보험금 안내/BI-003.json,parsed/markdown/예금보험금 안내/BI-003.md,parsed/json/예금보험금 안내/BI-003.json,,,2026-07-16T17:24:07+09:00,2026-07-16T17:24:12+09:00


,url_id,parent_doc_id,record_type,business_function,title,indexable,rag_index_mode,quality_status,quality_reasons,content_chars,...,table_count,action_count,attachment_count,video_count,supplementary_link_count,pagination_detected,pagination_is_complete,pagination_page_count,processing_policy,needs_review
0,DP-001,DP-001,page,예금자보호제도,보호한도,True,full,pass,,1110,...,0,0,0,0,0,False,True,1,,False
1,DP-002,DP-002,page,예금자보호제도,예금자보호제도,True,full,pass,,740,...,0,0,0,0,0,False,True,1,,False
2,DP-003,DP-003,page,예금자보호제도,보호대상 금융회사 개요,True,full,pass,,1073,...,1,2,2,0,0,False,True,1,,False
3,DP-004,DP-004,page,예금자보호제도,보호대상 금융상품 개요,True,full,pass,,2532,...,1,0,0,0,0,False,True,1,,False
4,DP-005,DP-005,page,예금자보호제도,예금자보호제도 FAQ,True,full,pass,,12519,...,0,0,0,0,0,True,True,4,,False
5,DP-006,DP-006,page,예금자보호제도,예금자보호제도 FAQ,True,full,pass,,3932,...,0,0,0,0,0,False,True,1,,False
6,DP-007,DP-007,page,예금자보호제도,예금자보호한도(해외),True,reference,pass,,313,...,1,0,0,0,0,False,True,1,,False
7,BI-001,BI-001,page,예금보험금 안내,신청시 구비서류,True,full,pass,,2334,...,1,10,10,0,0,False,True,1,,False
8,BI-002,BI-002,page,예금보험금 안내,예금보험금 신청절차,True,full,pass,,1185,...,0,1,0,0,0,False,True,1,,False
9,BI-003,BI-003,page,예금보험금 안내,예금보험금이란?,True,full,pass,,2354,...,0,1,0,0,0,False,True,1,,False


,validation,passed,value
0,DP-001 인덱싱 문서 청크 존재,True,NaN
1,DP-001 FAQ 청크 무손실,True,NaN
2,DP-001 청크 본문 보존율,True,1.0
3,DP-002 인덱싱 문서 청크 존재,True,NaN
4,DP-002 FAQ 청크 무손실,True,NaN
...,...,...,...
380,HP-004 맥락 포함 표시 제목,True,부실책임조사 진행현황 조회 (개인정보 수집이용 동의)
381,HP-004 Adobe Reader 노이즈 제거,True,NaN
382,HP-004 Action 라벨 길이,True,NaN
383,HP-004 목록 번호 중복 제거,True,NaN


## 7. 실제 수집 결과 구조 회귀 검사

합성 HTML 테스트만 통과했다고 완료로 판정하지 않습니다.

실제로 수집된 결과를 대상으로 다음을 다시 확인합니다.

```text
MT-010 카드 4개
MT-010 resource_group 청크 4개
외부 법령 링크 중복 없음
fn_layer 모달 2개 해결
규정 전문 내장 문서 2개
규정 전문 조문 청크 생성
MT-010 외 일반 .item 오분류 없음
MT-014 link_only
중복 번호 없음
웹툰 대사 RAG 유입 없음
```

In [ ]:
import re
from collections import Counter

documents = RESULT["documents"]
chunks = RESULT["chunks"]

documents_by_id = {
    document["doc_id"]: document
    for document in documents
}

assert "MT-010" in documents_by_id
assert "MT-014" in documents_by_id

mt010 = documents_by_id["MT-010"]
mt014 = documents_by_id["MT-014"]

# MT-014
assert mt014["record_type"] == "link_only"
assert mt014["indexable"] is False
assert not [
    chunk
    for chunk in chunks
    if chunk["document_id"] == "MT-014"
]

# MT-010 카드 그룹
groups = [
    block
    for block in mt010["blocks"]
    if block.get("type") == "resource_group"
]

assert len(groups) == 4
assert len({
    block["resource_group_id"]
    for block in groups
}) == 4

group_chunks = [
    chunk
    for chunk in chunks
    if (
        chunk["document_id"] == "MT-010"
        and chunk["chunk_type"] == "resource_group"
    )
]

assert len(group_chunks) == 4
assert len({
    chunk["resource_group_id"]
    for chunk in group_chunks
}) == 4

# 외부 법령 URL 중복 없음
law_links = [
    link
    for link in mt010["links"]
    if "law.go.kr" in link.get("url", "")
]

assert len(law_links) == 2
assert all(
    link["link_role"] == "action"
    for link in law_links
)

law_url_counts = Counter(
    link["url"]
    for link in law_links
)

assert all(
    count == 1
    for count in law_url_counts.values()
)

# 모달 Action과 내장 문서
modal_actions = [
    action
    for action in mt010["actions"]
    if action.get("interaction_type") == "open_modal"
]

assert len(modal_actions) == 2
assert all(
    action.get("modal_resolution_status") == "resolved"
    for action in modal_actions
)

expected_embedded_ids = {
    "MT-010_rule_layer01",
    "MT-010_operation_layer01",
}

assert set(
    mt010.get("embedded_document_ids", [])
) == expected_embedded_ids

assert expected_embedded_ids.issubset(
    documents_by_id.keys()
)

for document_id in expected_embedded_ids:
    embedded = documents_by_id[document_id]

    assert embedded["record_type"] == "modal_document"
    assert embedded["parent_doc_id"] == "MT-010"
    assert len(embedded["content_text"]) >= 500

    embedded_chunks = [
        chunk
        for chunk in chunks
        if chunk["document_id"] == document_id
    ]

    assert embedded_chunks
    assert any(
        chunk["chunk_type"] == "legal_article"
        for chunk in embedded_chunks
    )

# 현재 42개 정책에서 resource_group은 명시된 문서에만 허용
configured_group_docs = set(
    RESULT["manifest_df"]
    .loc[
        RESULT["manifest_df"][
            "resource_group_selector"
        ].str.strip() != "",
        "url_id",
    ]
)

actual_group_docs = {
    document["doc_id"]
    for document in RESULT["page_documents"]
    if any(
        block.get("type") == "resource_group"
        for block in document.get("blocks", [])
    )
}

assert actual_group_docs == configured_group_docs

# 전체 Markdown·청크 노이즈 검사
all_export_markdown = "\n".join(
    document.get("markdown_export", "")
    for document in documents
)

assert not re.search(
    r"(?m)^\s*\d+[.)]\s+\d+[.)]\s+",
    all_export_markdown,
)

all_rag_text = "\n".join(
    chunk.get("content", "")
    for chunk in chunks
)

assert "예튼이" not in all_rag_text
assert "예솜이" not in all_rag_text
assert "PDF파일 내용이 보이지 않으시면" not in all_rag_text

print("실제 MT-010 카드 4개: 통과")
print("실제 카드별 청크 4개: 통과")
print("국가법령정보센터 링크 중복 제거: 통과")
print("실제 fn_layer 모달 2개 해결: 통과")
print("실제 규정 전문 문서 2개: 통과")
print("실제 조문 청크 생성: 통과")
print("명시되지 않은 .item 오분류 없음: 통과")
print("MT-014 link_only: 통과")
print("중복 번호·웹툰·Adobe 노이즈: 통과")

print("내장 문서 수:", RESULT["summary"]["embedded_document_count"])
print("전체 문서 수:", RESULT["summary"]["document_count"])
print("전체 청크 수:", RESULT["summary"]["chunk_count"])

실제 MT-010 카드 4개: 통과
실제 카드별 청크 4개: 통과
국가법령정보센터 링크 중복 제거: 통과
실제 fn_layer 모달 2개 해결: 통과
실제 규정 전문 문서 2개: 통과
실제 조문 청크 생성: 통과
명시되지 않은 .item 오분류 없음: 통과
MT-014 link_only: 통과
중복 번호·웹툰·Adobe 노이즈: 통과
내장 문서 수: 2
전체 문서 수: 44
전체 청크 수: 302


## 8. 영상·웹툰 참고 링크 확인

In [ ]:
supplementary_df = RESULT['supplementary_df']
if supplementary_df.empty:
    print('생성된 참고 링크가 없습니다.')
else:
    display(supplementary_df[[
        'parent_doc_id', 'link_type', 'label', 'url',
        'display_condition', 'content_status'
    ]])

,parent_doc_id,link_type,label,url,display_condition,content_status
0,MT-001,video,착오송금 반환지원 제도 소개 영상 보기,https://www.kdic.or.kr/sp/kmrs/kmrsItrd/selectScrn.do,user_requests_video,reference_only
1,MT-009,webtoon,착오송금 반환지원 절차 웹툰 보기,https://fins.kdic.or.kr/ir/aplygudn/MtrsGvbkSprtProc/selectScrn.do,user_requests_visual_guide,reference_only


## 9. 첨부파일 다운로드·인덱싱 정책 확인

In [ ]:
attachment_df = RESULT['attachment_manifest_df']
if attachment_df.empty:
    print('발견된 첨부파일이 없습니다.')
else:
    display(attachment_df[[
        'parent_doc_id', 'display_name', 'download_method',
        'download_status', 'processing_policy', 'indexable',
        'official_download_url', 'user_action_url', 'sha256',
        'needs_review'
    ]])

    print('첨부파일 처리 요약')
    display(
        attachment_df.groupby(
            ['download_status', 'processing_policy'],
            dropna=False,
        ).size().rename('count').reset_index()
    )

,parent_doc_id,display_name,download_method,download_status,processing_policy,indexable,official_download_url,user_action_url,sha256,needs_review
0,DP-003,엑셀 다운받기 (XLSX),gfn_downloadFile,metadata_only,metadata_only,False,None,https://www.kdic.or.kr/sp/dpstrprot/selectProtSystProtSumr.do,,False
1,DP-003,한글 다운받기 (FILE),gfn_downloadFile,metadata_only,metadata_only,False,None,https://www.kdic.or.kr/sp/dpstrprot/selectProtSystProtSumr.do,,False
2,BI-001,예금보험금 등 지급청구 및 동의서 (HWP),gfn_downloadFile,metadata_only,metadata_only,False,None,https://www.kdic.or.kr/sp/dpstrprot/DpsmIbamtAplyPossDcmnt/selectScrn.do,,False
3,BI-001,예금보험금 등 지급청구 및 동의서 (PDF),gfn_downloadFile,metadata_only,metadata_only,False,None,https://www.kdic.or.kr/sp/dpstrprot/DpsmIbamtAplyPossDcmnt/selectScrn.do,,False
4,BI-001,위임장 (HWP),gfn_downloadFile,metadata_only,metadata_only,False,None,https://www.kdic.or.kr/sp/dpstrprot/DpsmIbamtAplyPossDcmnt/selectScrn.do,,False
5,BI-001,위임장 (PDF),gfn_downloadFile,metadata_only,metadata_only,False,None,https://www.kdic.or.kr/sp/dpstrprot/DpsmIbamtAplyPossDcmnt/selectScrn.do,,False
6,BI-001,친권자합의서 (HWP),gfn_downloadFile,metadata_only,metadata_only,False,None,https://www.kdic.or.kr/sp/dpstrprot/DpsmIbamtAplyPossDcmnt/selectScrn.do,,False
7,BI-001,친권자합의서 (PDF),gfn_downloadFile,metadata_only,metadata_only,False,None,https://www.kdic.or.kr/sp/dpstrprot/DpsmIbamtAplyPossDcmnt/selectScrn.do,,False
8,BI-001,지분신고서 및 수령권한 위임 동의서 (HWP),gfn_downloadFile,metadata_only,metadata_only,False,None,https://www.kdic.or.kr/sp/dpstrprot/DpsmIbamtAplyPossDcmnt/selectScrn.do,,False
9,BI-001,지분신고서 및 수령권한 위임 동의서 (PDF),gfn_downloadFile,metadata_only,metadata_only,False,None,https://www.kdic.or.kr/sp/dpstrprot/DpsmIbamtAplyPossDcmnt/selectScrn.do,,False


첨부파일 처리 요약


,download_status,processing_policy,count
0,metadata_only,metadata_only,43


## 10. HCX API 연동 설정

### API 키 등록

Colab 왼쪽 메뉴에서 열쇠 모양의 **Secrets**를 열고 다음 이름으로 키를 등록합니다.

```text
HCX_API_KEY
```

노트북 코드나 GitHub에 실제 키를 직접 입력하지 않습니다.

### 기본 모델

```text
Chat 모델: HCX-005
Embedding 모델: bge-m3
```

모델 사용 권한이 다른 경우 아래 설정 셀의 모델명만 변경합니다.

### 실행 범위

기본 설정은 다음과 같습니다.

```text
문서 메타데이터 생성: 실행
청크 임베딩 생성: 실행
RAG 예시 질문: 실행
```

API 호출 결과는 중간 저장되므로 같은 런타임에서 셀을 다시 실행할 때 성공한 항목은 재사용합니다.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata
except ImportError:
    userdata = None

from openai import OpenAI


def load_hcx_api_key() -> str:
    """Colab Secret 또는 환경 변수에서 HCX API 키를 읽습니다."""
    key = None

    if userdata is not None:
        try:
            key = userdata.get("HCX_API_KEY")
        except Exception:
            key = None

    if not key:
        key = os.environ.get("HCX_API_KEY")

    if not key:
        raise RuntimeError(
            "HCX API 키가 없습니다.\n"
            "Colab 왼쪽의 Secrets에 HCX_API_KEY를 등록한 뒤 "
            "이 셀을 다시 실행하세요."
        )

    return key.strip()


HCX_API_KEY = load_hcx_api_key()

HCX_BASE_URL = (
    "https://clovastudio.stream.ntruss.com/v1/openai"
)

# 사용 권한에 따라 HCX-005, HCX-DASH-002, HCX-007 등으로 변경할 수 있습니다.
HCX_CHAT_MODEL = "HCX-005"
HCX_EMBEDDING_MODEL = "bge-m3"

# 실행 기능
HCX_RUN_METADATA = True
HCX_RUN_EMBEDDING = True
HCX_RUN_RAG_DEMO = True

# None이면 전체 처리합니다.
HCX_DOCUMENT_LIMIT = None
HCX_CHUNK_LIMIT = None

# API 호출 설정
HCX_METADATA_MAX_INPUT_CHARS = 14000
HCX_EMBEDDING_BATCH_SIZE = 16
HCX_REQUEST_TIMEOUT_SECONDS = 120
HCX_MAX_RETRIES = 4

# RAG 예시 질문
HCX_RAG_QUESTION = (
    "카카오뱅크로 돈을 잘못보냈는데 어떻게 해야해?"
)
HCX_RAG_TOP_K = 5

hcx_client = OpenAI(
    api_key=HCX_API_KEY,
    base_url=HCX_BASE_URL,
    timeout=HCX_REQUEST_TIMEOUT_SECONDS,
    max_retries=HCX_MAX_RETRIES,
)

print("HCX 설정 완료")
print("- Base URL:", HCX_BASE_URL)
print("- Chat 모델:", HCX_CHAT_MODEL)
print("- Embedding 모델:", HCX_EMBEDDING_MODEL)
print("- 메타데이터:", HCX_RUN_METADATA)
print("- 임베딩:", HCX_RUN_EMBEDDING)
print("- RAG 예시:", HCX_RUN_RAG_DEMO)
print("- API 키:", "등록됨")

HCX 설정 완료
- Base URL: https://clovastudio.stream.ntruss.com/v1/openai
- Chat 모델: HCX-005
- Embedding 모델: bge-m3
- 메타데이터: True
- 임베딩: True
- RAG 예시: True
- API 키: 등록됨


## 11. HCX 연결 테스트

API 키와 Chat 모델이 실제로 호출되는지 한 번 확인합니다.

키 값은 출력하지 않습니다. 실패하면 오류 코드와 모델 사용 권한을 확인합니다.

In [ ]:
def hcx_chat_text(
    *,
    system_prompt: str,
    user_prompt: str,
    max_tokens: int = 500,
    temperature: float = 0.1,
) -> str:
    response = hcx_client.chat.completions.create(
        model=HCX_CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    text = response.choices[0].message.content

    if not text or not text.strip():
        raise RuntimeError(
            "HCX 응답 본문이 비어 있습니다."
        )

    return text.strip()


connection_result = hcx_chat_text(
    system_prompt=(
        "당신은 API 연결 상태 확인 도우미입니다. "
        "요청받은 짧은 문장만 출력하세요."
    ),
    user_prompt=(
        "다음 문장만 그대로 출력하세요: "
        "HCX API 연결 성공"
    ),
    max_tokens=50,
    temperature=0.0,
)

print(connection_result)

HCX API 연결 성공


## 12. HCX 문서 메타데이터 생성

크롤링 원문과 청크는 수정하지 않습니다.

HCX는 문서별로 다음 파생 메타데이터만 생성합니다.

```text
summary
sub_category
user_roles
keywords
question_types
answerable_questions
needs_human_review
review_reason
```

### 저장 파일

```text
processed/hcx_metadata.jsonl
processed/documents_hcx_enriched.jsonl
quality/hcx_metadata_review.csv
processed/hcx/cache/metadata/
```

동일한 문서 본문 해시와 프롬프트 버전의 결과가 이미 있으면 API를 다시 호출하지 않습니다.

In [ ]:
import copy
import hashlib
import json
import re
import time
from datetime import datetime, timezone

import pandas as pd
from tqdm.auto import tqdm


HCX_METADATA_PROMPT_VERSION = "kdic-metadata-v1"

ALLOWED_QUESTION_TYPES = {
    "정의형",
    "대상형",
    "조건형",
    "절차형",
    "서류형",
    "금액형",
    "기한형",
    "예외형",
    "비교형",
    "링크안내형",
    "파일안내형",
}

ALLOWED_USER_ROLES = {
    "일반 예금자",
    "예금보험금 신청인",
    "상속인",
    "미수령금 신청인",
    "착오송금인",
    "착오송금 수취인",
    "대리인",
    "채무자",
    "은닉재산 신고자",
    "일반 사용자",
}


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_text(value: str) -> str:
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()


def extract_json_object(text: str) -> dict:
    """마크다운 코드 펜스가 있어도 첫 JSON 객체를 복구합니다."""
    cleaned = text.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )

    try:
        value = json.loads(cleaned)
        if isinstance(value, dict):
            return value
    except json.JSONDecodeError:
        pass

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(
            "HCX 응답에서 JSON 객체를 찾지 못했습니다."
        )

    value = json.loads(cleaned[start:end + 1])

    if not isinstance(value, dict):
        raise ValueError(
            "HCX 메타데이터 결과가 JSON 객체가 아닙니다."
        )

    return value


def normalize_string_list(
    value,
    *,
    max_items: int,
) -> list[str]:
    if not isinstance(value, list):
        return []

    result = []
    seen = set()

    for item in value:
        text = str(item).strip()

        if not text or text in seen:
            continue

        seen.add(text)
        result.append(text)

        if len(result) >= max_items:
            break

    return result


def validate_hcx_metadata(
    raw: dict,
    document: dict,
) -> dict:
    summary = str(
        raw.get("summary", "")
    ).strip()

    sub_category = str(
        raw.get("sub_category", "")
    ).strip()

    user_roles = [
        role
        for role in normalize_string_list(
            raw.get("user_roles"),
            max_items=4,
        )
        if role in ALLOWED_USER_ROLES
    ]

    question_types = [
        question_type
        for question_type in normalize_string_list(
            raw.get("question_types"),
            max_items=6,
        )
        if question_type in ALLOWED_QUESTION_TYPES
    ]

    keywords = normalize_string_list(
        raw.get("keywords"),
        max_items=10,
    )

    answerable_questions = normalize_string_list(
        raw.get("answerable_questions"),
        max_items=5,
    )

    needs_human_review = bool(
        raw.get("needs_human_review", False)
    )

    review_reason = str(
        raw.get("review_reason", "")
    ).strip()

    if not summary:
        raise ValueError("summary가 비어 있습니다.")

    if not sub_category:
        sub_category = (
            document.get("display_title")
            or document.get("title")
            or "미분류"
        )

    return {
        "summary": summary,
        "sub_category": sub_category,
        "user_roles": user_roles,
        "keywords": keywords,
        "question_types": question_types,
        "answerable_questions": answerable_questions,
        "needs_human_review": needs_human_review,
        "review_reason": review_reason,
    }


def build_metadata_prompt(
    document: dict,
) -> str:
    content = (
        document.get("content_text")
        or document.get("body_markdown")
        or document.get("content_markdown")
        or ""
    )

    content = content[:HCX_METADATA_MAX_INPUT_CHARS]

    return f"""
다음은 예금보험공사 또는 금융안심포털의 공식 문서입니다.

[문서 정보]
문서 ID: {document.get("doc_id", "")}
업무: {document.get("business_function", "")}
제목: {document.get("display_title") or document.get("title", "")}
문서 유형: {document.get("record_type", "")}
출처: {document.get("source_url", "")}

[공식 원문]
{content}

아래 규칙을 모두 지키세요.

1. 공식 원문에 없는 금액, 기간, 조건, 기관명은 만들지 마세요.
2. 원문의 문장이나 수치를 수정하지 마세요.
3. summary는 2~3문장으로 작성하세요.
4. sub_category는 업무 안에서 이 문서를 구분할 수 있는 짧은 한국어 분류명으로 작성하세요.
5. user_roles는 다음 값 중에서만 선택하세요.
   {sorted(ALLOWED_USER_ROLES)}
6. question_types는 다음 값 중에서만 선택하세요.
   {sorted(ALLOWED_QUESTION_TYPES)}
7. keywords는 최대 10개, answerable_questions는 최대 5개입니다.
8. 정보가 모호하거나 동적 조회 결과가 필요한 문서면 needs_human_review를 true로 표시하세요.
9. 설명 문장 없이 JSON 객체 하나만 출력하세요.

반환 형식:
{{
  "summary": "공식 문서의 핵심 내용",
  "sub_category": "하위 분류",
  "user_roles": ["허용된 역할"],
  "keywords": ["핵심어"],
  "question_types": ["허용된 질문 유형"],
  "answerable_questions": ["이 문서로 답할 수 있는 질문"],
  "needs_human_review": false,
  "review_reason": ""
}}
""".strip()


def deterministic_link_only_metadata(
    document: dict,
) -> dict:
    title = (
        document.get("display_title")
        or document.get("title")
        or document.get("doc_id")
    )

    return {
        "summary": (
            f"{title} 서비스를 이용할 수 있는 "
            "공식 바로가기 문서입니다."
        ),
        "sub_category": title,
        "user_roles": ["일반 사용자"],
        "keywords": [
            title,
            "공식 바로가기",
        ],
        "question_types": ["링크안내형"],
        "answerable_questions": [
            f"{title} 페이지로 이동하려면 어떻게 하나요?"
        ],
        "needs_human_review": False,
        "review_reason": "",
    }


hcx_processed_dir = RESULT["paths"].processed / "hcx"
hcx_metadata_cache_dir = (
    hcx_processed_dir / "cache" / "metadata"
)
hcx_metadata_cache_dir.mkdir(
    parents=True,
    exist_ok=True,
)

documents_for_metadata = list(
    RESULT["documents"]
)

if HCX_DOCUMENT_LIMIT is not None:
    documents_for_metadata = (
        documents_for_metadata[:HCX_DOCUMENT_LIMIT]
    )

metadata_records = []
metadata_review_rows = []
enriched_documents = []

if HCX_RUN_METADATA:
    metadata_by_doc_id = {}

    for document in tqdm(
        documents_for_metadata,
        desc="HCX 문서 메타데이터",
    ):
        doc_id = document["doc_id"]

        source_text = (
            document.get("content_text")
            or document.get("body_markdown")
            or document.get("content_markdown")
            or ""
        )

        input_hash = (
            document.get("content_hash")
            or document.get("parsed_content_sha256")
            or sha256_text(source_text)
        )

        cache_key = sha256_text(
            "|".join([
                doc_id,
                input_hash,
                HCX_CHAT_MODEL,
                HCX_METADATA_PROMPT_VERSION,
            ])
        )

        cache_path = (
            hcx_metadata_cache_dir
            / f"{cache_key}.json"
        )

        started_at = time.time()
        status = "success"
        error_message = ""
        cache_hit = False

        try:
            if cache_path.exists():
                metadata = json.loads(
                    cache_path.read_text(
                        encoding="utf-8"
                    )
                )
                cache_hit = True

            elif (
                document.get("record_type")
                == "link_only"
                or not source_text.strip()
            ):
                metadata = (
                    deterministic_link_only_metadata(
                        document
                    )
                )
                status = "deterministic_non_content"

            else:
                raw_text = hcx_chat_text(
                    system_prompt=(
                        "당신은 금융 공공문서 메타데이터 "
                        "생성기입니다. 공식 원문에 없는 "
                        "정보를 추측하지 말고 JSON만 "
                        "출력하세요."
                    ),
                    user_prompt=build_metadata_prompt(
                        document
                    ),
                    max_tokens=1200,
                    temperature=0.0,
                )

                raw_metadata = extract_json_object(
                    raw_text
                )

                metadata = validate_hcx_metadata(
                    raw_metadata,
                    document,
                )

                cache_path.write_text(
                    json.dumps(
                        metadata,
                        ensure_ascii=False,
                        indent=2,
                    ),
                    encoding="utf-8",
                )

            record = {
                "doc_id": doc_id,
                "parent_doc_id": document.get(
                    "parent_doc_id"
                ),
                "business_function": document.get(
                    "business_function"
                ),
                "source_url": document.get(
                    "source_url"
                ),
                "input_content_sha256": input_hash,
                "model": HCX_CHAT_MODEL,
                "prompt_version": (
                    HCX_METADATA_PROMPT_VERSION
                ),
                "generated_at": utc_now_iso(),
                "generation_status": status,
                "cache_hit": cache_hit,
                **metadata,
            }

            metadata_records.append(record)
            metadata_by_doc_id[doc_id] = record

        except Exception as error:
            status = "failed"
            error_message = (
                f"{type(error).__name__}: {error}"
            )

        metadata_review_rows.append({
            "doc_id": doc_id,
            "status": status,
            "cache_hit": cache_hit,
            "elapsed_seconds": round(
                time.time() - started_at,
                3,
            ),
            "error": error_message,
        })

    for document in RESULT["documents"]:
        enriched = copy.deepcopy(document)

        metadata = metadata_by_doc_id.get(
            document["doc_id"]
        )

        if metadata:
            enriched["llm_metadata"] = metadata
            enriched["summary"] = metadata["summary"]
            enriched["sub_category"] = (
                metadata["sub_category"]
            )
            enriched["user_roles"] = (
                metadata["user_roles"]
            )
            enriched["keywords"] = (
                metadata["keywords"]
            )
            enriched["question_types"] = (
                metadata["question_types"]
            )
        else:
            enriched["llm_metadata"] = {
                "generation_status": "not_processed"
            }

        enriched_documents.append(enriched)

    metadata_jsonl_path = (
        RESULT["paths"].processed
        / "hcx_metadata.jsonl"
    )
    enriched_documents_path = (
        RESULT["paths"].processed
        / "documents_hcx_enriched.jsonl"
    )
    metadata_review_path = (
        RESULT["paths"].quality
        / "hcx_metadata_review.csv"
    )

    with metadata_jsonl_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for record in metadata_records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    with enriched_documents_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for document in enriched_documents:
            file.write(
                json.dumps(
                    document,
                    ensure_ascii=False,
                )
                + "\n"
            )

    metadata_review_df = pd.DataFrame(
        metadata_review_rows
    )
    metadata_review_df.to_csv(
        metadata_review_path,
        index=False,
        encoding="utf-8-sig",
    )

    print("HCX 메타데이터 생성 완료")
    print("- 성공:", len(metadata_records))
    print(
        "- 실패:",
        sum(
            row["status"] == "failed"
            for row in metadata_review_rows
        ),
    )
    print("- 출력:", metadata_jsonl_path)
    print("- 출력:", enriched_documents_path)

    display(metadata_review_df)
else:
    print("HCX_RUN_METADATA=False: 메타데이터 생성을 건너뜁니다.")

HCX 문서 메타데이터:   0%|          | 0/44 [00:00<?, ?it/s]

HCX 메타데이터 생성 완료
- 성공: 44
- 실패: 0
- 출력: /content/kdic_crawl_output_20260716_172301/processed/hcx_metadata.jsonl
- 출력: /content/kdic_crawl_output_20260716_172301/processed/documents_hcx_enriched.jsonl


,doc_id,status,cache_hit,elapsed_seconds,error
0,DP-001,success,False,4.673,
1,DP-002,success,False,4.211,
2,DP-003,success,False,4.193,
3,DP-004,success,False,5.478,
4,DP-005,success,False,7.212,
5,DP-006,success,False,5.080,
6,DP-007,success,False,3.855,
7,BI-001,success,False,4.833,
8,BI-002,success,False,8.336,
9,BI-003,success,False,4.245,


## 13. HCX `bge-m3` 청크 임베딩 생성

이미 만들어진 `chunks.jsonl`의 청크를 벡터로 변환합니다.

HCX가 청크를 새로 나누지 않습니다.

```text
기존 구조 기반 청크
→ bge-m3 임베딩
→ 1,024차원 벡터
```

### 저장 파일

```text
processed/chunk_embeddings_hcx.jsonl
quality/hcx_embedding_report.csv
processed/hcx/cache/embeddings/
```

각 청크는 입력 텍스트 해시를 기준으로 캐시되며, 재실행 시 성공한 벡터를 재사용합니다.

In [ ]:
import math
import numpy as np


def hcx_embed_batch(
    texts: list[str],
) -> list[list[float]]:
    if not texts:
        return []

    vectors: list[list[float]] = []

    for index, text in enumerate(texts):
        cleaned_text = (
            str(text)
            .replace("\x00", "")
            .strip()
        )

        if not cleaned_text:
            raise ValueError(
                f"{index}번 임베딩 입력 텍스트가 비어 있습니다."
            )

        response = hcx_client.embeddings.create(
            model=HCX_EMBEDDING_MODEL,
            input=cleaned_text,
            encoding_format="float",
        )

        if not response.data:
            raise RuntimeError(
                f"{index}번 임베딩 응답에 data가 없습니다."
            )

        vector = list(
            response.data[0].embedding
        )

        if not vector:
            raise RuntimeError(
                f"{index}번 임베딩 벡터가 비어 있습니다."
            )

        vectors.append(vector)

    return vectors


hcx_embedding_cache_dir = (
    hcx_processed_dir / "cache" / "embeddings"
)
hcx_embedding_cache_dir.mkdir(
    parents=True,
    exist_ok=True,
)

chunks_for_embedding = list(
    RESULT["chunks"]
)

if HCX_CHUNK_LIMIT is not None:
    chunks_for_embedding = (
        chunks_for_embedding[:HCX_CHUNK_LIMIT]
    )

embedding_records = []
embedding_report_rows = []

if HCX_RUN_EMBEDDING:
    pending = []

    for chunk in chunks_for_embedding:
        text = str(
            chunk.get("content", "")
        ).strip()

        if not text:
            embedding_report_rows.append({
                "chunk_id": chunk["chunk_id"],
                "status": "skipped_empty",
                "dimensions": 0,
                "error": "",
            })
            continue

        text_hash = sha256_text(text)
        cache_key = sha256_text(
            "|".join([
                chunk["chunk_id"],
                text_hash,
                HCX_EMBEDDING_MODEL,
            ])
        )
        cache_path = (
            hcx_embedding_cache_dir
            / f"{cache_key}.json"
        )

        if cache_path.exists():
            cached = json.loads(
                cache_path.read_text(
                    encoding="utf-8"
                )
            )
            embedding_records.append(cached)
            embedding_report_rows.append({
                "chunk_id": chunk["chunk_id"],
                "status": "cache_hit",
                "dimensions": len(
                    cached["embedding"]
                ),
                "error": "",
            })
        else:
            pending.append({
                "chunk": chunk,
                "text": text,
                "text_hash": text_hash,
                "cache_path": cache_path,
            })

    for start in tqdm(
        range(
            0,
            len(pending),
            HCX_EMBEDDING_BATCH_SIZE,
        ),
        desc="HCX 청크 임베딩",
    ):
        batch = pending[
            start:start + HCX_EMBEDDING_BATCH_SIZE
        ]

        batch_texts = [
            item["text"]
            for item in batch
        ]

        try:
            vectors = hcx_embed_batch(
                batch_texts
            )

            for item, vector in zip(
                batch,
                vectors,
            ):
                chunk = item["chunk"]

                record = {
                    "chunk_id": chunk["chunk_id"],
                    "document_id": chunk[
                        "document_id"
                    ],
                    "business_function": chunk.get(
                        "business_function"
                    ),
                    "source_url": chunk.get(
                        "source_url"
                    ),
                    "chunk_type": chunk.get(
                        "chunk_type"
                    ),
                    "input_content_sha256": item[
                        "text_hash"
                    ],
                    "model": HCX_EMBEDDING_MODEL,
                    "dimensions": len(vector),
                    "embedding": vector,
                }

                embedding_records.append(record)

                item["cache_path"].write_text(
                    json.dumps(
                        record,
                        ensure_ascii=False,
                    ),
                    encoding="utf-8",
                )

                embedding_report_rows.append({
                    "chunk_id": chunk["chunk_id"],
                    "status": "success",
                    "dimensions": len(vector),
                    "error": "",
                })

        except Exception as error:
            error_message = (
                f"{type(error).__name__}: {error}"
            )

            for item in batch:
                embedding_report_rows.append({
                    "chunk_id": item["chunk"][
                        "chunk_id"
                    ],
                    "status": "failed",
                    "dimensions": 0,
                    "error": error_message,
                })

    embedding_by_chunk_id = {
        record["chunk_id"]: record
        for record in embedding_records
    }

    embedding_records = [
        embedding_by_chunk_id[chunk["chunk_id"]]
        for chunk in chunks_for_embedding
        if chunk["chunk_id"]
        in embedding_by_chunk_id
    ]

    embedding_jsonl_path = (
        RESULT["paths"].processed
        / "chunk_embeddings_hcx.jsonl"
    )
    embedding_report_path = (
        RESULT["paths"].quality
        / "hcx_embedding_report.csv"
    )

    with embedding_jsonl_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for record in embedding_records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    embedding_report_df = pd.DataFrame(
        embedding_report_rows
    )
    embedding_report_df.to_csv(
        embedding_report_path,
        index=False,
        encoding="utf-8-sig",
    )

    successful_dimensions = {
        record["dimensions"]
        for record in embedding_records
    }

    if embedding_records and (
        len(successful_dimensions) != 1
    ):
        raise RuntimeError(
            "임베딩 벡터 차원이 서로 다릅니다."
        )

    print("HCX 임베딩 완료")
    print("- 전체 청크:", len(chunks_for_embedding))
    print("- 성공 벡터:", len(embedding_records))
    print("- 벡터 차원:", successful_dimensions)
    print("- 출력:", embedding_jsonl_path)

    display(embedding_report_df)
else:
    print("HCX_RUN_EMBEDDING=False: 임베딩 생성을 건너뜁니다.")

HCX 청크 임베딩:   0%|          | 0/19 [00:00<?, ?it/s]

HCX 임베딩 완료
- 전체 청크: 302
- 성공 벡터: 302
- 벡터 차원: {1024}
- 출력: /content/kdic_crawl_output_20260716_172301/processed/chunk_embeddings_hcx.jsonl


,chunk_id,status,dimensions,error
0,DP-001_chunk_000,success,1024,
1,DP-002_chunk_000,success,1024,
2,DP-002_chunk_001,success,1024,
3,DP-002_chunk_002,success,1024,
4,DP-002_chunk_003,success,1024,
...,...,...,...,...
297,MT-010_operation_layer01_chunk_010,success,1024,
298,MT-010_operation_layer01_chunk_011,success,1024,
299,MT-010_operation_layer01_chunk_012,success,1024,
300,MT-010_operation_layer01_chunk_013,success,1024,


## 14. 의미 검색과 HCX 근거 기반 답변 예시

생성한 임베딩으로 질문과 가까운 청크를 찾고, HCX에는 검색된 공식 근거만 전달합니다.

```text
질문
→ 질문 임베딩
→ 코사인 유사도 Top-K 검색
→ 검색된 공식 청크
→ HCX 답변
→ 문서 ID와 공식 출처
```

이 셀의 질문은 `HCX_RAG_QUESTION`에서 변경할 수 있습니다.

In [ ]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray,
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b)
        / denominator
    )


def semantic_search_hcx(
    question: str,
    *,
    top_k: int = 5,
    business_function: str | None = None,
) -> list[dict]:
    if not embedding_records:
        raise RuntimeError(
            "임베딩 결과가 없습니다. "
            "먼저 임베딩 생성 셀을 실행하세요."
        )

    query_vector = np.asarray(
        hcx_embed_batch([question])[0],
        dtype=np.float32,
    )

    chunks_by_id = {
        chunk["chunk_id"]: chunk
        for chunk in RESULT["chunks"]
    }

    scored = []

    for record in embedding_records:
        if (
            business_function
            and record.get("business_function")
            != business_function
        ):
            continue

        vector = np.asarray(
            record["embedding"],
            dtype=np.float32,
        )

        score = cosine_similarity(
            query_vector,
            vector,
        )

        chunk = chunks_by_id.get(
            record["chunk_id"]
        )

        if not chunk:
            continue

        scored.append({
            "score": score,
            "chunk": chunk,
        })

    scored.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    return scored[:top_k]


def generate_grounded_hcx_answer(
    question: str,
    search_results: list[dict],
) -> str:
    evidence_blocks = []

    for rank, result in enumerate(
        search_results,
        start=1,
    ):
        chunk = result["chunk"]

        evidence_blocks.append(
            "\n".join([
                f"[근거 {rank}]",
                f"문서 ID: {chunk.get('document_id')}",
                f"청크 ID: {chunk.get('chunk_id')}",
                f"업무: {chunk.get('business_function')}",
                f"출처: {chunk.get('source_url')}",
                "내용:",
                chunk.get("content", ""),
            ])
        )

    evidence_text = "\n\n".join(
        evidence_blocks
    )

    return hcx_chat_text(
        system_prompt=(
            "당신은 예금보험공사 공식 문서 기반 "
            "RAG 답변기입니다. 제공된 근거에 있는 "
            "정보만 사용하세요. 근거에 없는 금액, "
            "기한, 조건을 추측하지 마세요. "
            "근거가 부족하면 공식 자료에서 확인할 "
            "수 없다고 답하세요. 답변 마지막에는 "
            "사용한 문서 ID를 표시하세요."
        ),
        user_prompt=f"""
[사용자 질문]
{question}

[검색된 공식 근거]
{evidence_text}

한국어로 이해하기 쉽게 답변하세요.
신청·조회 링크가 근거에 있으면 공식 URL만 안내하세요.
""".strip(),
        max_tokens=900,
        temperature=0.0,
    )


if HCX_RUN_RAG_DEMO:
    if not HCX_RUN_EMBEDDING:
        raise RuntimeError(
            "RAG 예시를 실행하려면 "
            "HCX_RUN_EMBEDDING=True가 필요합니다."
        )

    rag_results = semantic_search_hcx(
        HCX_RAG_QUESTION,
        top_k=HCX_RAG_TOP_K,
    )

    print("질문:", HCX_RAG_QUESTION)
    print("\n검색 결과")

    rag_result_rows = []

    for rank, result in enumerate(
        rag_results,
        start=1,
    ):
        chunk = result["chunk"]

        rag_result_rows.append({
            "rank": rank,
            "score": round(
                result["score"],
                6,
            ),
            "document_id": chunk.get(
                "document_id"
            ),
            "chunk_id": chunk.get(
                "chunk_id"
            ),
            "business_function": chunk.get(
                "business_function"
            ),
            "chunk_type": chunk.get(
                "chunk_type"
            ),
            "source_url": chunk.get(
                "source_url"
            ),
            "preview": chunk.get(
                "content",
                "",
            )[:180],
        })

    rag_result_df = pd.DataFrame(
        rag_result_rows
    )
    display(rag_result_df)

    rag_answer = generate_grounded_hcx_answer(
        HCX_RAG_QUESTION,
        rag_results,
    )

    print("\nHCX 답변\n")
    print(rag_answer)

    hcx_rag_demo_dir = (
        RESULT["paths"].root / "evaluation"
    )
    hcx_rag_demo_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    hcx_rag_demo_path = (
        hcx_rag_demo_dir
        / "hcx_rag_demo.json"
    )

    hcx_rag_demo_path.write_text(
        json.dumps(
            {
                "question": HCX_RAG_QUESTION,
                "chat_model": HCX_CHAT_MODEL,
                "embedding_model": (
                    HCX_EMBEDDING_MODEL
                ),
                "top_k": HCX_RAG_TOP_K,
                "retrieved": rag_result_rows,
                "answer": rag_answer,
                "generated_at": utc_now_iso(),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print("\n저장:", hcx_rag_demo_path)
else:
    print("HCX_RUN_RAG_DEMO=False: RAG 예시를 건너뜁니다.")

질문: 카카오뱅크로 돈을 잘못보냈는데 어떻게 해야해?

검색 결과


,rank,score,document_id,chunk_id,business_function,chunk_type,source_url,preview
0,1,0.630929,MT-011,MT-011_chunk_000,착오송금 반환 신청,section,https://fins.kdic.or.kr/ir/aplygudn/MtrsStutChc/selectScrn.do,착오송금인 돈을 잘못 보냈어요.\n\n- 신청자격 확인\n- 바로 신청\n\n착오송금수취인 모르는 돈을 받았어요.\n\n- 이의제기\n- 채무잔액 확인\n- 이체수수료 환급신청\n- 반환 확인\n\n※ 착오송금반환지원을 처음 하시는 분은 신청자격 확인을 먼저 진행해주십시오.
1,2,0.591078,MT-001,MT-001_chunk_000,착오송금 반환 신청,section,https://www.kdic.or.kr/sp/kmrs/kmrsItrd/selectScrn.do,### 제도 소개 안내영상\n\n### 잘못 보낸 돈 되찾기 서비스란 착오송금인이 실수로 잘못 보낸 돈을 최소한의 비용으로 빠르게 되찾을 수 있도록 도와드리는 제도입니다.
2,3,0.574641,MT-015,MT-015_chunk_000,착오송금 반환 신청,section,https://www.kdic.or.kr/sp/kmrs/kmrsItrdAplyTrgt/selectScrn.do,#### 신청 가능한 착오송금 금액 한도가 있나요?\n\n신청 가능 한도는 착오송금 건당 5만원 이상 ~ 1억원 이하 입니다.\n\n#### 언제까지 신청하면 되나요?\n\n잘못 이체한 날로부터 1년 이내까지 신청 가능합니다
3,4,0.567115,MT-004,MT-004_chunk_007,착오송금 반환 신청,faq,https://fins.kdic.or.kr/cm/bbs/selectFaqMsdrGvbkAply.do,"### Q. Toss나 카카오페이와 같이 간편송금을 통해 발생한 착오송금도 지원대상이 되나요?\nToss 등 선불전자지급수단을 활용하는 간편송금업자도 착오송금 반환지원 적용이 가능하나, 현행 법상 수취인의 실지명의 확보가 불가능한 경우에 해당하는 간편송금은 반환지원 대상에서 제외됩니다. ① 송금인이 수취인의 ‘계좌번호’를"
4,5,0.566736,MT-005,MT-005_chunk_008,착오송금 반환 신청,faq,https://fins.kdic.or.kr/cm/bbs/selectFaqTop10.do,"### Q. 9. Toss나 카카오페이와 같이 간편송금을 통해 발생한 착오송금도 지원대상이 되나요?\nToss 등 선불전자지급수단을 활용하는 간편송금업자도 착오송금 반환지원 적용이 가능하나, 현행 법상 수취인의 실지명의 확보가 불가능한 경우에 해당하는 간편송금은 반환지원 대상에서 제외됩니다. ① 송금인이 수취인의 ‘계좌번호"



HCX 답변

카카오뱅크로 돈을 잘못 보내셨다면 다음과 같은 방법으로 착오송금 반환 신청을 할 수 있습니다.

먼저, 착오송금 반환 신청 자격을 확인해야 합니다. 신청 자격은 착오송금 건당 5만 원 이상~1억 원 이하의 금액이며, 잘못 이체한 날로부터 1년 이내에 신청 가능합니다. 또한, 송금 시 계좌번호를 입력하여 카카오뱅크로 송금한 경우에는 착오송금 반환 신청 대상이 됩니다. 그러나 연락처 송금과 같이 별도의 절차 없이 송금한 경우는 반환 지원 대상에서 제외될 수 있으니 참고하시기 바랍니다.

위의 내용에 해당한다면 '바로 신청' 메뉴를 클릭하여 착오송금 반환 신청을 진행하시면 됩니다.

원활한 처리를 위해 아래의 링크를 통해 착오송금 반환 신청을 하시기 바랍니다.
 - [착오송금 반환 신청](https://fins.kdic.or.kr/ir/aplygudn/MtrsStutChc/selectScrn.do)

위 내용을 참고하시어 착오송금 문제를 해결하실 수 있기를 바랍니다. (문서 ID: MT-011, MT-001, MT-015, MT-004, MT-005)

저장: /content/kdic_crawl_output_20260716_172301/evaluation/hcx_rag_demo.json


## 15. HCX 결과 품질 검사

다음을 자동으로 확인합니다.

```text
API 키가 원문 파일에 저장되지 않았는가
원본 documents.jsonl이 변경되지 않았는가
HCX 메타데이터에 문서 ID·모델·프롬프트 버전이 있는가
성공한 임베딩의 차원이 동일한가
임베딩 chunk_id가 기존 chunks.jsonl에 존재하는가
실패 항목이 보고서에 남았는가
```

In [ ]:
from pathlib import Path


# 노트북 출력 파일에 API 키가 쓰이지 않았는지 검사
for candidate in RESULT["paths"].root.rglob("*"):
    if not candidate.is_file():
        continue

    if candidate.suffix.lower() not in {
        ".json",
        ".jsonl",
        ".csv",
        ".md",
        ".txt",
        ".py",
    }:
        continue

    try:
        text = candidate.read_text(
            encoding="utf-8"
        )
    except UnicodeDecodeError:
        continue

    assert HCX_API_KEY not in text, (
        f"API 키가 파일에 포함됨: {candidate}"
    )

# 메타데이터 검사
if HCX_RUN_METADATA:
    assert metadata_records, (
        "HCX 메타데이터 성공 결과가 없습니다."
    )

    for record in metadata_records:
        assert record["doc_id"]
        assert record["model"] == HCX_CHAT_MODEL
        assert record["prompt_version"] == (
            HCX_METADATA_PROMPT_VERSION
        )
        assert record["summary"]
        assert record["sub_category"]

# 임베딩 검사
if HCX_RUN_EMBEDDING:
    assert embedding_records, (
        "HCX 임베딩 성공 결과가 없습니다."
    )

    original_chunk_ids = {
        chunk["chunk_id"]
        for chunk in RESULT["chunks"]
    }

    dimensions = {
        record["dimensions"]
        for record in embedding_records
    }

    assert len(dimensions) == 1
    assert all(
        record["chunk_id"] in original_chunk_ids
        for record in embedding_records
    )

    assert all(
        len(record["embedding"])
        == record["dimensions"]
        for record in embedding_records
    )

print("HCX API 키 비저장 검사: 통과")

if HCX_RUN_METADATA:
    print(
        "HCX 메타데이터 검사: 통과",
        len(metadata_records),
    )

if HCX_RUN_EMBEDDING:
    print(
        "HCX 임베딩 검사: 통과",
        len(embedding_records),
        "개 /",
        dimensions,
        "차원",
    )

HCX API 키 비저장 검사: 통과
HCX 메타데이터 검사: 통과 44
HCX 임베딩 검사: 통과 302 개 / {1024} 차원


## 16. HCX 결과를 포함한 최종 ZIP 재생성

원래 크롤링 ZIP은 HCX 실행 전에 생성되므로, HCX 결과 파일을 포함하여 ZIP을 다시 만듭니다.

In [ ]:
import shutil

hcx_archive_base = (
    RESULT["paths"].root.parent
    / f"{RESULT['paths'].root.name}_HCX"
)

hcx_archive_path = Path(
    shutil.make_archive(
        base_name=str(hcx_archive_base),
        format="zip",
        root_dir=RESULT["paths"].root.parent,
        base_dir=RESULT["paths"].root.name,
    )
)

RESULT["hcx_archive_path"] = hcx_archive_path

print("HCX 통합 ZIP 생성 완료")
print(hcx_archive_path)
print(
    "파일 크기:",
    round(
        hcx_archive_path.stat().st_size
        / 1024
        / 1024,
        2,
    ),
    "MB",
)

HCX 통합 ZIP 생성 완료
/content/kdic_crawl_output_20260716_172301_HCX.zip
파일 크기: 6.11 MB


## 17. 주요 결과 파일

```text
manifest/
├── review_policy_42.csv
├── manifest_full_42.csv
└── manifest_run.csv

raw/
├── html/
├── response_meta/
└── assets/
    └── 업무명/
        └── 문서ID/
            └── images/

parsed/
├── markdown/
├── json/
└── attachments/

processed/
├── documents.jsonl
├── chunks.jsonl
├── attachment_manifest.csv
├── attachment_manifest.jsonl
├── supplementary_links.csv
└── supplementary_links.jsonl

quality/
├── quality_report.csv
├── quality_summary.json
├── regression_tests.csv
└── attachment_review.csv
```

### Markdown 필드

```text
body_markdown
→ RAG 본문용 Markdown

export_markdown
→ 큰 제목·공식 바로가기·관련 링크까지 포함한 검수용 Markdown
```

기존 호환성을 위해 `content_markdown`, `markdown_export` 필드도 함께 유지됩니다.

### 모달 규정 전문 출력

```text
parsed/
├── markdown/
│   └── 착오송금 반환 신청/
│       └── embedded/
│           ├── MT-010_rule_layer01.md
│           └── MT-010_operation_layer01.md
└── json/
    └── 착오송금 반환 신청/
        └── embedded/
            ├── MT-010_rule_layer01.json
            └── MT-010_operation_layer01.json
```

두 내장 문서는 `processed/documents.jsonl`과 `processed/chunks.jsonl`에도 포함됩니다.


### HCX 추가 결과

```text
processed/
├── hcx_metadata.jsonl
├── documents_hcx_enriched.jsonl
├── chunk_embeddings_hcx.jsonl
└── hcx/
    └── cache/
        ├── metadata/
        └── embeddings/

quality/
├── hcx_metadata_review.csv
└── hcx_embedding_report.csv

evaluation/
└── hcx_rag_demo.json
```

`documents.jsonl`과 `chunks.jsonl`은 결정론적 원본으로 그대로 유지됩니다.


## 18. 결과 ZIP 다운로드

전체 실행이 끝난 후 아래 셀을 실행하면
Markdown, JSON, 청크, 이미지와 품질 보고서를 포함한 결과 ZIP이 내려받아집니다.

In [ ]:
archive_path = RESULT.get(
    "hcx_archive_path",
    RESULT["archive_path"],
)

print("다운로드할 결과 ZIP:", archive_path)

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print(
        "Colab이 아닌 환경입니다. "
        "위 경로에서 ZIP을 확인하세요."
    )

다운로드할 결과 ZIP: /content/kdic_crawl_output_20260716_172301_HCX.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>